# 🏁 Formula 1 Podium Prediction Companion — Milestone 1 (complete)
## Part 1 Data Cleaning & Auditing · Part 2 Data-Engineering Questions · Part 3 Feature Selection & Engineering · Part 4 Predictive Modeling · Part 5 Experiments · Part 6 Explainability · Part 7 Inference

**Dataset:** Formula 1 World Championship 1950–2024 (Kaggle / Ergast), 14 relational CSV tables.
**Unit of analysis:** one driver entering one race · **Target:** `podium = positionOrder <= 3` · **Prediction cutoff:** after qualifying, before the race starts.

This notebook is the **single, unified version** of the team's four EDA/cleaning write-ups:

| Code | Source document | Scope |
|---|---|---|
| **A** | *F1 Race Calendar: EDA & Cleaning Summary (three-table contribution)* | `races`, `circuits`, `seasons` |
| **B** | *F1 Podium Prediction — EDA Insights & Proposed Solutions* | all tables; insight / problem / decision IDs (`I-xx`, `P-xx`, `D-xx`) |
| **C** | *F1 Podium Prediction: EDA & Cleaning Summary (Milestone 1)* | all 14 tables |
| **D** | *F1_EDA_Cleaning_Summary.docx* | `drivers`, `results`, `qualifying`, `sprint_results`, `driver_standings` |

Every problem is now solved **in exactly one way**, and every number in the text below the code cells is **computed by the notebook itself** (the documents are no longer the source of truth — this notebook is).

### What Part 1 of the milestone requires, and where it is answered

| Milestone 1 requirement (Part 1: Data Cleaning & Auditing) | Section |
|---|---|
| Initial EDA **before** cleaning, with visualizations, not manual inspection | §3 |
| Clean sentinels (`\N`) and apply semantic typing (dates, durations → seconds) | §4.1–4.2 |
| Exactly one row per (raceId, driverId): find duplicates, choose a keep-rule, justify it | §4.4 |
| Primary keys unique & non-null; foreign keys reconcile | §3.3–3.4 (raw) and §4.8 (clean) |
| Raw CSV files immutable; cleaning on copies | §2 (fingerprint) and §8 (proof) |
| EDA **after** cleaning, showing the impact of cleaning | §5 |
| A unique driver–race modeling table with documented duplicate handling | §6 |

| Part 2 & Part 3 requirement | Section |
|---|---|
| Q1 — front-row → podium conversion by circuit, pre-hybrid vs hybrid, grid vs qualifying | §10 |
| Q2 — home advantage after controlling for starting position | §11 |
| Q3 — mechanical-retirement rate by constructor, 2014–2021 vs 2022–2024 | §12 |
| Leakage-safe form / standings / grid / qualifying features (past races only), with leakage tests | §13 |
| Train / validation / test split, justified | §14 |
| Feature choices justified with plots, statistics and a selection rule | §15 |
| Effect of preprocessing on the data + preprocessing-order experiments | §16 |
| Baseline + ≥ 2 statistical models + shallow neural network, how they work, limitations, tuning attempts | §17–18 |
| All metrics on train / validation / test, primary/secondary metric justified, plots | §19 |
| Feature-group ablation; preprocessing experiments repeated with the selected model | §20–21 |
| Global XAI: permutation importance (feature + group) and SHAP; local XAI: SHAP waterfall + LIME; non-causal reading | §22–25 |
| Inference function on a complete and a partially-missing record | §26 |
| Requirement checklist and final summary | §27 |

## 0.1 How the four documents were unified (conflict register)

The documents mostly agree. Where they disagreed, the team settled the conflict as follows (IDs follow document B so the register can be traced back):

| ID | Topic | A | B | C | D | **Unified decision (this notebook)** |
|---|---|---|---|---|---|---|
| D-01 / P-03 | Keep-rule for duplicate (raceId, driverId) rows | — | best `positionOrder` | best `positionOrder` → more laps → lower `resultId` | **points → laps** | **Best `positionOrder`, ties → more laps → lower `resultId`.** All four rules are compared side by side in §4.4 |
| D-03 / P-04 | Definition of a non-starter | — | status list **or** (grid = 0 and laps = 0) | status list only | — | **Status list or (grid = 0 and laps = 0)**; genuine pit-lane starters kept and flagged (§6.2) |
| D-05 / P-05 | Status grouping (needed for Question 3) | — | 10 hand-checked groups, tyres & team errors separate | 10 different groups, tyres/health/operational in "other" | — | **Document B's map** + an `ambiguous` flag for a later sensitivity check (§4.5) |
| D-02 / P-09 | Where Part 1 ends / modeling period | 3 tables only | 2003–2024 base table | recommends 2003 | full history | **14 cleaned tables (all years) + a 2003–2024 driver–race base table** (§6) |
| P-01 | Which strings count as missing | `\N` + blanks + null-like | `\N` + empty | `\N` + blanks | `\N` + blanks | `\N` and empty/whitespace-only cells. Other null-like strings (`NULL`, `NaN`, `NA`, …) are **counted** in §3.2 to prove they don't occur |
| P-02 | `results.time` (mixes total time and gaps like `+5.478`) | — | keep as text | keep as text | split | **Kept as text** (it is post-race and never a feature); `milliseconds` already holds the numeric value |
| P-02 | Times of day (`races.time`, session times, `pit_stops.time`) | string | timedelta | — | — | **timedelta** |
| P-06 / P-07 / D-04 | Very long pit stops / laps | — | keep + flag | keep, use medians | — | **Keep + flag** (`is_red_flag_stop`, `is_outlier_lap`) |
| P-12 | 1995 Japanese GP Q1 = 1,002 s | — | report only | — | — | **Keep + flag** (`q_time_suspect`); outside the modeling period |
| P-15 | `constructor_results.status` (99.9 % empty) | — | — | drop | — | **Dropped** from the clean copy (§4.3) |
| P-14 | > 25-point rows, fastest-lap/rank mismatches | — | — | explained (double points, Indy 500) | flagged for review | **Explained by a check cell** (§4.7); values not changed |
| P-16 | Races with ≠ 3 podium rows after dedup | — | — | open issue | — | **Diagnosed** in §5.6 |
| P-17 | Proof that the raw data were not modified | deep copies + assertions | copies | copies | file-hash comparison + reload | **SHA-256 file hashes + DataFrame fingerprints**, checked at the end (§8) |

Small numeric differences between the documents (e.g. podium rate from pole 63 % vs 64 %, qualifying coverage 5–60 % vs 6–60 %) come from rounding or a different base; they are simply **recomputed** here.

**Decisions intentionally left for later parts** (not Part 1): train/validation/test split (D-06), primary metric (D-07), grid-0 encoding (D-08), lagged standings (P-08), points normalisation (P-11). They are listed in the hand-off (§8.3).

---
# 1. Setup
Imports, plotting style, paths and the table metadata (primary keys, grain keys and foreign keys) that drive every automated check in the notebook.

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import os, glob, hashlib, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.max_rows', 200)

print('pandas', pd.__version__, '| numpy', np.__version__)

In [ ]:
# ── Plot style: one consistent, colour-blind-checked palette for every figure ──
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN = PALETTE
GREY = '#8a8a85'
plt.rcParams.update({
    'figure.figsize': (11, 4.5), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e6e3', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'axes.titleweight': 'bold', 'axes.titlesize': 12,
    'axes.labelcolor': '#333333', 'xtick.color': '#555555', 'ytick.color': '#555555',
    'axes.prop_cycle': plt.cycler(color=PALETTE), 'lines.linewidth': 2,
})

def pct_axis(ax, axis='y', decimals=0):
    # Format an axis that holds shares (0–1) as percentages.
    fmt = mtick.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == 'y' else ax.xaxis).set_major_formatter(fmt)

def say(text):
    # Render a computed, data-driven sentence as Markdown under a cell.
    display(Markdown(text))

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
# Works on Kaggle (dataset attached as input) and locally (set F1_DATA_DIR or put the CSVs in ./data).
def find_data_dir():
    env = os.environ.get('F1_DATA_DIR')
    if env and (Path(env) / 'races.csv').exists():
        return Path(env)
    for cand in [Path('/kaggle/input/formula-1-world-championship-1950-2020'), Path('data'), Path('.')]:
        if (cand / 'races.csv').exists():
            return cand
    hits = glob.glob('/kaggle/input/**/races.csv', recursive=True)
    if hits:
        return Path(hits[0]).parent
    raise FileNotFoundError('races.csv not found: attach the Kaggle dataset or set F1_DATA_DIR')

DATA_DIR = find_data_dir()
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
CLEAN_DIR = OUT_DIR / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print('Reading raw CSVs from :', DATA_DIR)
print('Writing outputs to    :', OUT_DIR)

In [ ]:
# ── Table metadata: drives every automated key / grain / link check ───────────
TABLES = ['races', 'circuits', 'seasons', 'drivers', 'constructors', 'status',
          'results', 'qualifying', 'sprint_results', 'driver_standings',
          'constructor_results', 'constructor_standings', 'pit_stops', 'lap_times']

# Primary key = the column(s) that identify a row (must be unique and non-null)
PK = {
    'races': ['raceId'], 'circuits': ['circuitId'], 'seasons': ['year'], 'drivers': ['driverId'],
    'constructors': ['constructorId'], 'status': ['statusId'], 'results': ['resultId'],
    'qualifying': ['qualifyId'], 'sprint_results': ['resultId'],
    'driver_standings': ['driverStandingsId'], 'constructor_results': ['constructorResultsId'],
    'constructor_standings': ['constructorStandingsId'],
    'pit_stops': ['raceId', 'driverId', 'stop'], 'lap_times': ['raceId', 'driverId', 'lap'],
}

# Grain ("meaning") key = what one row is supposed to represent
GRAIN = {
    'races': ['year', 'round'], 'results': ['raceId', 'driverId'], 'qualifying': ['raceId', 'driverId'],
    'sprint_results': ['raceId', 'driverId'], 'driver_standings': ['raceId', 'driverId'],
    'constructor_results': ['raceId', 'constructorId'], 'constructor_standings': ['raceId', 'constructorId'],
}

# Foreign keys: (child table, child column, parent table, parent column) — 23 links
FK = [
    ('races', 'circuitId', 'circuits', 'circuitId'), ('races', 'year', 'seasons', 'year'),
    ('results', 'raceId', 'races', 'raceId'), ('results', 'driverId', 'drivers', 'driverId'),
    ('results', 'constructorId', 'constructors', 'constructorId'), ('results', 'statusId', 'status', 'statusId'),
    ('qualifying', 'raceId', 'races', 'raceId'), ('qualifying', 'driverId', 'drivers', 'driverId'),
    ('qualifying', 'constructorId', 'constructors', 'constructorId'),
    ('sprint_results', 'raceId', 'races', 'raceId'), ('sprint_results', 'driverId', 'drivers', 'driverId'),
    ('sprint_results', 'constructorId', 'constructors', 'constructorId'), ('sprint_results', 'statusId', 'status', 'statusId'),
    ('driver_standings', 'raceId', 'races', 'raceId'), ('driver_standings', 'driverId', 'drivers', 'driverId'),
    ('constructor_standings', 'raceId', 'races', 'raceId'), ('constructor_standings', 'constructorId', 'constructors', 'constructorId'),
    ('constructor_results', 'raceId', 'races', 'raceId'), ('constructor_results', 'constructorId', 'constructors', 'constructorId'),
    ('pit_stops', 'raceId', 'races', 'raceId'), ('pit_stops', 'driverId', 'drivers', 'driverId'),
    ('lap_times', 'raceId', 'races', 'raceId'), ('lap_times', 'driverId', 'drivers', 'driverId'),
]
assert len(TABLES) == 14 and len(FK) == 23

# Every decision taken in the notebook is appended here and exported as decision_log.csv (§8)
DECISIONS = []
def log_decision(id_, step, finding, decision, why, alternatives, impact, sources):
    DECISIONS.append(dict(id=id_, step=step, finding=finding, decision=decision, why=why,
                          alternatives=alternatives, impact=impact, sources=sources))

---
# 2. Load the raw data (read-only) and fingerprint it

**Design choice — read everything as text.** Each CSV is loaded with `dtype=str, keep_default_na=False`, so pandas does **not** silently turn anything into `NaN` or guess a type. This lets the raw EDA measure the data *exactly as delivered* (including every `\N`).

**Immutability (P-17).** Before touching anything we store:
1. a **SHA-256 hash of every CSV file** (document D's method), and
2. a **content fingerprint of every raw DataFrame** (document A's method).

Both are re-computed at the very end (§8.1) to prove that neither the files nor the in-memory raw tables changed. All cleaning happens on copies (`typed[...]`, `clean[...]`).

In [ ]:
def file_sha256(path, chunk=1 << 20):
    # Hash a file in 1 MB chunks (memory-safe for lap_times.csv).
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def df_fingerprint(df):
    # Order-sensitive content hash of a DataFrame (values + index).
    return int(pd.util.hash_pandas_object(df, index=True).sum())

RAW_FILE_HASH = {t: file_sha256(DATA_DIR / f'{t}.csv') for t in TABLES}

# Read every table as plain text: no type guessing, no automatic NaN conversion
raw = {t: pd.read_csv(DATA_DIR / f'{t}.csv', dtype=str, keep_default_na=False, na_filter=False)
       for t in TABLES}
RAW_DF_FINGERPRINT = {t: df_fingerprint(raw[t]) for t in TABLES}

pd.DataFrame({'rows': {t: len(raw[t]) for t in TABLES},
              'columns': {t: raw[t].shape[1] for t in TABLES},
              'sha256 (first 16 chars)': {t: RAW_FILE_HASH[t][:16] for t in TABLES}})

---
# 3. Exploratory Data Analysis on the **raw** data

We ask every table the same five questions with **one reusable function** (`eda_table`), so nothing relies on opening CSVs by hand:

1. **What's in it?** `\N` sentinels, blank cells, other null-like strings, unique values per column.
2. **Is the key clean?** Primary key unique and never missing; grain key (e.g. one row per driver per race).
3. **Do the links work?** Every foreign key exists in its parent table.
4. **Which seasons are covered?** Join to `races` for each row's year.
5. **What do the numbers look like?** Statistics and plots on a *temporary* numeric copy (the raw tables stay text).

In [ ]:
SENTINEL = '\\N'                                       # the literal two-character string \N
N_COL = 'n_\\N'                                         # column name used in the EDA summaries
OTHER_NULL_LIKE = ['NULL', 'null', 'NaN', 'nan', 'None', 'NA', 'N/A', '-']

def sentinel_mask(df):
    # True where a raw cell is a missing-value placeholder: '\N' or empty / whitespace-only.
    return df.apply(lambda s: s.eq(SENTINEL) | s.str.strip().eq(''))

def tmp_num(s):
    # Temporary numeric view of a raw text column (placeholders -> NaN). Used for EDA only.
    return pd.to_numeric(s.where(~(s.eq(SENTINEL) | s.str.strip().eq(''))), errors='coerce')

# Year of every race, used to place any raceId-keyed row in time
RACE_YEAR = pd.Series(raw['races']['year'].astype(int).values, index=raw['races']['raceId'])

def eda_table(name, df):
    # Profile one raw table. Returns (one-row summary dict, per-column profile DataFrame).
    is_N = df.eq(SENTINEL)
    is_blank = df.apply(lambda s: s.str.strip().eq(''))
    missing = is_N | is_blank
    prof = pd.DataFrame({
        'n_\\N': is_N.sum(), 'n_blank': is_blank.sum(),
        'n_other_null_like': df.isin(OTHER_NULL_LIKE).sum(),
        'pct_missing': (missing.mean() * 100).round(2),
        'n_unique': df.nunique(),
        'example': [df.loc[~missing[c], c].iloc[0] if (~missing[c]).any() else '' for c in df.columns],
    })
    pk = PK[name]
    grain = GRAIN.get(name)
    years = df['raceId'].map(RACE_YEAR) if 'raceId' in df.columns else None
    summary = {
        'table': name, 'rows': len(df), 'cols': df.shape[1],
        'n_\\N': int(is_N.values.sum()), 'n_blank': int(is_blank.values.sum()),
        'pct_cells_missing': round(missing.values.mean() * 100, 2),
        'cols_with_missing': int((missing.sum() > 0).sum()),
        'pk': '+'.join(pk), 'pk_duplicates': int(df.duplicated(pk).sum()),
        'pk_missing': int(missing[pk].any(axis=1).sum()),
        'grain': '+'.join(grain) if grain else '', 'grain_duplicates': int(df.duplicated(grain).sum()) if grain else np.nan,
        'full_row_duplicates': int(df.duplicated().sum()),
        'cells_with_outer_spaces': int(df.apply(lambda s: s.ne(s.str.strip())).values.sum()),
        'first_year': int(years.min()) if years is not None else np.nan,
        'last_year': int(years.max()) if years is not None else np.nan,
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }
    return summary, prof

eda_raw, profiles_raw = [], {}
for t in TABLES:
    s, p = eda_table(t, raw[t])
    eda_raw.append(s)
    profiles_raw[t] = p
eda_summary_raw = pd.DataFrame(eda_raw).set_index('table')
eda_summary_raw

## 3.1 Table overview
The 14 tables differ in size by four orders of magnitude, from a 75-row lookup (`seasons`) to more than half a million lap records. `races` is the hub: almost every fact table reaches the calendar through `raceId`.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
order = eda_summary_raw['rows'].sort_values()
ax.barh(order.index, order.values, color=BLUE, height=0.6)
ax.set_xscale('log')
ax.set_xlabel('rows (log scale)')
ax.set_title('Rows per raw table')
for y, v in enumerate(order.values):
    ax.text(v * 1.08, y, f'{v:,}', va='center', fontsize=9, color='#333333')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

say(f"**{len(TABLES)} tables, {eda_summary_raw['rows'].sum():,} rows in total.** "
    f"`results` (the table that holds the target) has **{len(raw['results']):,}** driver-race rows "
    f"over **{raw['results']['raceId'].nunique():,}** races.")

## 3.2 Placeholder ("sentinel") values per column (P-01)
`\N` is a *string*, so pandas cannot see it as missing: numeric operations on these columns would fail or silently give wrong results. The profile below counts `\N`, blank cells and other null-like strings column by column.

In [ ]:
# Long table: one row per column that contains any placeholder
sent = (pd.concat({t: p for t, p in profiles_raw.items()}, names=['table', 'column'])
          .pipe(lambda d: d[(d[N_COL] > 0) | (d['n_blank'] > 0) | (d['n_other_null_like'] > 0)])
          .sort_values('pct_missing', ascending=False))
display(sent[[N_COL, 'n_blank', 'n_other_null_like', 'pct_missing']])

fig, ax = plt.subplots(figsize=(11, max(4, 0.26 * len(sent))))
lbl = [f'{t}.{c}' for t, c in sent.index]
ax.barh(lbl[::-1], sent['pct_missing'].values[::-1] / 100, color=BLUE, height=0.65)
pct_axis(ax, 'x')
ax.set_title('Share of cells that are placeholders (\\N or blank), by column')
ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

n_N, n_blank = int(eda_summary_raw[N_COL].sum()), int(eda_summary_raw['n_blank'].sum())
n_other = int(sent['n_other_null_like'].sum())
say(f"**{n_N:,} `\\N` cells and {n_blank:,} blank cells** in **{len(sent)} columns** across "
    f"**{(eda_summary_raw[N_COL] + eda_summary_raw['n_blank'] > 0).sum()} tables**. "
    f"Other null-like strings (`NULL`, `NaN`, `NA`, …): **{n_other}**, so only `\\N` and blanks need converting.")

**Reading the plot.** Missingness is concentrated in a few columns and most of it is *structural* rather than an error: race-weekend session dates exist only for recent seasons, `q2`/`q3` exist only for drivers who advanced, finishing `position`/`time` exist only for classified finishers, and `number`/`code` exist only for modern drivers. This matters for cleaning: these gaps must become real `NaN`, **not** be filled with guesses.

## 3.3 Primary keys and grain (raw)
A primary key must be unique and never missing. The grain key checks what one row is *supposed* to mean — the Milestone requires one row per (raceId, driverId).

In [ ]:
pk_raw = eda_summary_raw[['pk', 'pk_duplicates', 'pk_missing', 'grain', 'grain_duplicates', 'full_row_duplicates']]
display(pk_raw)

n_pk_bad = int((pk_raw['pk_duplicates'] + pk_raw['pk_missing']).gt(0).sum())
grain_bad = pk_raw.loc[pk_raw['grain_duplicates'].fillna(0) > 0, 'grain_duplicates']
say(f"**Primary keys:** {14 - n_pk_bad}/14 are unique and never missing. "
    f"**Grain violations:** " + (', '.join(f'`{t}` has {int(v)} extra rows' for t, v in grain_bad.items()) or 'none') +
    ". The `results` duplicates are handled in §4.4.")

## 3.4 Foreign keys (raw)
Every child value must exist in the parent table (otherwise joins silently drop or orphan rows). Placeholders are excluded from the check because a missing link is not an orphan.

In [ ]:
def fk_check(tables, child, col, parent, pcol, is_raw):
    c = tables[child][col]
    if is_raw:
        c = c[~(c.eq(SENTINEL) | c.str.strip().eq(''))]
    else:
        c = c.dropna()
    parent_vals = set(tables[parent][pcol].dropna().tolist())
    orphans = ~c.isin(parent_vals)
    return {'link': f'{child}.{col} → {parent}.{pcol}', 'values_checked': len(c),
            'orphans': int(orphans.sum()), 'example_orphan': c[orphans].iloc[0] if orphans.any() else ''}

fk_raw = pd.DataFrame([fk_check(raw, *f, is_raw=True) for f in FK])
display(fk_raw)
say(f"**{(fk_raw['orphans'] == 0).sum()}/23 foreign-key links have zero orphans** "
    f"({fk_raw['values_checked'].sum():,} values checked). Joins between the tables are safe (I-01).")

## 3.5 Coverage over time: which seasons does each table describe?
A table can be complete for the seasons it covers and still be **absent** for others. This is the most important finding for the modeling period (P-09, D-02).

In [ ]:
# Share of each season's races that have at least one row in each raceId-keyed table
races_per_year = RACE_YEAR.groupby(RACE_YEAR).size()
fact_tables = [t for t in TABLES if 'raceId' in raw[t].columns and t != 'races']
cov = pd.DataFrame({t: raw[t]['raceId'].drop_duplicates().map(RACE_YEAR).value_counts()
                       .reindex(races_per_year.index, fill_value=0) / races_per_year
                    for t in fact_tables}).T

fig, ax = plt.subplots(figsize=(13, 4.2))
im = ax.imshow(cov.values, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
ax.set_yticks(range(len(cov.index)), cov.index)
yrs = cov.columns.tolist()
ax.set_xticks(range(0, len(yrs), 5), yrs[::5], rotation=0)
ax.set_title('Coverage by season: share of races with at least one row')
ax.grid(False)
cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout(); plt.show()

first_year = {t: int(cov.columns[cov.loc[t] > 0][0]) for t in cov.index}
display(pd.Series(first_year, name='first season with data').to_frame().T)

In [ ]:
# Qualifying: coverage by season, and how often q2/q3 exist (knock-out format from 2006)
q = raw['qualifying'].assign(year=raw['qualifying']['raceId'].map(RACE_YEAR))
q_cov = cov.loc['qualifying']
q_has = q.groupby('year').agg(q2=('q2', lambda s: (~(s.eq(SENTINEL) | s.str.strip().eq(''))).mean()),
                              q3=('q3', lambda s: (~(s.eq(SENTINEL) | s.str.strip().eq(''))).mean()))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(q_cov.index, q_cov.values, color=BLUE, marker='o', ms=3)
ax[0].axvline(2003, color=ORANGE, ls='--', lw=1.5); ax[0].text(2003.5, 0.08, 'complete from 2003', color=ORANGE)
ax[0].set_xlim(1990, cov.columns.max() + 1); pct_axis(ax[0])
ax[0].set_title('Share of races with qualifying data')
ax[1].plot(q_has.index, q_has['q2'], label='q2 present', color=AQUA)
ax[1].plot(q_has.index, q_has['q3'], label='q3 present', color=MAGENTA)
ax[1].axvline(2006, color=GREY, ls=':', lw=1.2); ax[1].text(2006.3, 0.03, 'knock-out format from 2006', color=GREY)
pct_axis(ax[1]); ax[1].legend(frameon=False); ax[1].set_title('Share of qualifying rows with a q2 / q3 time')
plt.tight_layout(); plt.show()

gap = q_cov.loc[1996:2002]
# first season from which every later season has qualifying for >= 95 % of its races
complete_from = [y for y in q_cov.index if y >= 1994 and (q_cov.loc[y:] >= 0.95).all()][0]
say(f"Qualifying starts in **{first_year['qualifying']}**, but between 1996 and 2002 only "
    f"**{gap.min():.0%}–{gap.max():.0%}** of races have it; coverage stays ≥ 95 % **from {complete_from}** on. "
    "(Correction of an earlier statement that coverage was ≥ 90 % from 1994.)")

In [ ]:
# Missingness by season for the columns that matter most, as a heatmap (% of rows missing)
def miss_by_year(table, cols):
    df = raw[table]
    yr = df['raceId'].map(RACE_YEAR) if table != 'races' else df['year'].astype(int)
    m = sentinel_mask(df[cols])
    return m.groupby(yr.values).mean().T.rename(index=lambda c: f'{table}.{c}')

heat = pd.concat([
    miss_by_year('results', ['position', 'time', 'fastestLapTime', 'rank']),
    miss_by_year('qualifying', ['q1', 'q2', 'q3']),
    miss_by_year('races', ['time', 'fp1_date', 'quali_date', 'sprint_date']),
])
fig, ax = plt.subplots(figsize=(13, 4.4))
im = ax.imshow(heat.values, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
ax.set_yticks(range(len(heat.index)), heat.index)
ax.set_xticks(range(0, heat.shape[1], 5), heat.columns[::5])
ax.set_title('Share of rows missing by season (darker = more missing; blank columns = table has no rows that year)')
ax.grid(False)
cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout(); plt.show()

**Interpretation (I-11).** Data coverage depends on the era: `results` and `driver_standings` go back to 1950, constructor tables start in 1958 (when the constructors' championship began), `lap_times` in 1996, `pit_stops` in 2011, `sprint_results` in 2021. Fastest-lap columns are recorded only from 2004, race start time (`races.time`) from 2005, and weekend-session dates only for the most recent seasons. Features built from these tables can only apply to part of the history — this is why the base table is restricted to the era in which qualifying, the strongest pre-race signal, is complete (§6.1).

## 3.6 Race calendar, circuits and seasons (document A)
Checks specific to the hub table: dates parse, the stored `year` agrees with the date, rounds are numbered 1…n inside each season, no race name repeats within a season, every circuit and season is used, and coordinates are in valid ranges.

In [ ]:
r = raw['races']
r_date = pd.to_datetime(r['date'], format='%Y-%m-%d', errors='coerce')
r_year = r['year'].astype(int)
rounds_ok = r.assign(rnd=r['round'].astype(int), yr=r_year).groupby('yr')['rnd'].apply(
    lambda s: sorted(s.tolist()) == list(range(1, len(s) + 1)))
c = raw['circuits']
lat, lng, alt = tmp_num(c['lat']), tmp_num(c['lng']), tmp_num(c['alt'])

calendar_checks = pd.Series({
    'race dates that fail to parse': int(r_date.isna().sum()),
    'year ≠ calendar year of date': int((r_date.dt.year != r_year).sum()),
    'seasons whose rounds are not exactly 1..n': int((~rounds_ok).sum()),
    'race name repeated within a season': int(r.duplicated(['year', 'name']).sum()),
    'circuits never used by a race': int((~c['circuitId'].isin(r['circuitId'])).sum()),
    'seasons with no race': int((~raw['seasons']['year'].isin(r['year'])).sum()),
    'latitude outside [-90, 90]': int((~lat.between(-90, 90)).sum()),
    'longitude outside [-180, 180]': int((~lng.between(-180, 180)).sum()),
}, name='count').to_frame()
display(calendar_checks)
say(f"Races span **{r_year.min()}–{r_year.max()}** ({r_year.nunique()} seasons, max round {r['round'].astype(int).max()}). "
    f"Circuit altitude ranges **{alt.min():.0f} m to {alt.max():,.0f} m**; latitude {lat.min():.2f}° to {lat.max():.2f}°, "
    f"longitude {lng.min():.2f}° to {lng.max():.2f}°.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.4, 1, 1.2]})
rpy = r_year.value_counts().sort_index()
ax[0].bar(rpy.index, rpy.values, color=BLUE, width=0.8)
ax[0].set_title('Races per season'); ax[0].set_xlabel('season')

top_c = r['circuitId'].map(c.set_index('circuitId')['country']).value_counts().head(15)[::-1]
ax[1].barh(top_c.index, top_c.values, color=BLUE, height=0.65)
ax[1].set_title('Top 15 host countries (races)'); ax[1].grid(axis='y', visible=False)

n_races = r['circuitId'].value_counts().reindex(c['circuitId']).fillna(0).values
ax[2].scatter(lng, lat, s=10 + n_races * 1.2, color=BLUE, alpha=0.55, edgecolor='white', linewidth=0.8)
ax[2].set_xlim(-180, 180); ax[2].set_ylim(-60, 75)
ax[2].set_title('Circuit locations (size = races hosted)'); ax[2].set_xlabel('longitude'); ax[2].set_ylabel('latitude')
plt.tight_layout(); plt.show()
say(f"The calendar grew from **{rpy.iloc[0]} races in {rpy.index[0]}** to **{rpy.iloc[-1]} in {rpy.index[-1]}** "
    f"(min since 2000: {rpy.loc[2000:].min()} in {rpy.loc[2000:].idxmin()}).")

## 3.7 The target: how rare is a podium? (I-02, I-03, I-04)
A temporary numeric view of `results` is enough to look at the target before cleaning. `positionOrder` is always filled (it ranks every entrant, including non-finishers), so it is the correct column to define the label.

In [ ]:
res_tmp = pd.DataFrame({
    'raceId': raw['results']['raceId'], 'driverId': raw['results']['driverId'],
    'year': raw['results']['raceId'].map(RACE_YEAR),
    'grid': tmp_num(raw['results']['grid']), 'positionOrder': tmp_num(raw['results']['positionOrder']),
})
res_tmp['podium'] = (res_tmp['positionOrder'] <= 3).astype(int)
res_tmp['decade'] = (res_tmp['year'] // 10) * 10
entrants = res_tmp.groupby('raceId').size()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
by_dec = res_tmp.groupby('decade')['podium'].mean()
ax[0].bar(by_dec.index.astype(str) + 's', by_dec.values, color=BLUE, width=0.6)
pct_axis(ax[0]); ax[0].set_title('Podium rate by decade')

per_race = pd.DataFrame({'entrants': entrants, 'year': res_tmp.groupby('raceId')['year'].first()})
yr = per_race.groupby('year')['entrants'].median()
ax[1].plot(yr.index, yr.values, color=BLUE)
ax[1].set_title('Median entrants per race'); ax[1].set_xlabel('season')

g = res_tmp[res_tmp['grid'].between(1, 20)].groupby('grid')['podium'].mean()
ax[2].bar(g.index, g.values, color=BLUE, width=0.7)
pct_axis(ax[2]); ax[2].set_xticks(range(1, 21, 2))
ax[2].set_title('Podium rate by grid position (all seasons)'); ax[2].set_xlabel('grid')
plt.tight_layout(); plt.show()

g0 = res_tmp.loc[res_tmp['grid'] == 0, 'podium']
say(f"**Podiums are rare: {res_tmp['podium'].mean():.1%} of driver-race rows** "
    f"(always predicting 'no podium' would already be {1 - res_tmp['podium'].mean():.1%} accurate → accuracy is misleading; PR-AUC/F1 later). "
    f"The rate ranges from **{by_dec.min():.1%} ({by_dec.idxmin()}s)** to **{by_dec.max():.1%} ({by_dec.idxmax()}s)** because field size changed "
    f"(median entrants {yr.max():.0f} at the peak vs {yr.iloc[-1]:.0f} in {yr.index[-1]}). "
    f"Grid position is strongly linked to podiums: P1 → **{g.loc[1]:.0%}**, P3 → {g.loc[3]:.0%}, P5 → {g.loc[5]:.0%}, P10 → {g.loc[10]:.0%}. "
    f"**{len(g0):,} rows have grid = 0** (pit-lane start or no grid slot) with a podium rate of {g0.mean():.1%}. "
    "These are associations, not causes.")

## 3.8 The same driver more than once in a race (P-03)
The Milestone requires exactly one row per (raceId, driverId). We measure how often this is violated, when, and how many rows are involved.

In [ ]:
dup_mask = raw['results'].duplicated(['raceId', 'driverId'], keep=False)
dups = res_tmp[dup_mask].copy()
pair_size = dups.groupby(['raceId', 'driverId']).size()

fig, ax = plt.subplots(figsize=(11, 3.6))
py = dups.drop_duplicates(['raceId', 'driverId'])['year'].value_counts().sort_index()
ax.bar(py.index, py.values, color=ORANGE, width=0.8)
ax.set_title('Repeated (raceId, driverId) pairs per season'); ax.set_xlabel('season'); ax.set_ylabel('pairs')
plt.tight_layout(); plt.show()

say(f"**{len(pair_size)} pairs ({len(dups)} rows, {len(dups) - len(pair_size)} extra rows) in {dups['raceId'].nunique()} races**, "
    f"seasons {int(dups['year'].min())}–{int(dups['year'].max())}. "
    f"{(pair_size == 2).sum()} pairs have 2 rows, {(pair_size == 3).sum()} have 3. "
    "These are shared drives (drivers swapping cars mid-race in the early championship), plus a few drivers entered in two cars.")

In [ ]:
# A few concrete examples, so the duplicate pattern is visible (not just counted)
ex = (raw['results'][dup_mask]
      .merge(raw['races'][['raceId', 'year', 'name']], on='raceId')
      .merge(raw['drivers'][['driverId', 'forename', 'surname']], on='driverId')
      .merge(raw['status'], on='statusId'))
ex[['year', 'name', 'forename', 'surname', 'constructorId', 'grid', 'positionOrder', 'points', 'laps', 'status']] \
  .sort_values(['year', 'name', 'surname']).head(12)

## 3.9 Status labels and `positionText` codes (P-05)
`status` holds the free-text outcome of every entry. It is needed to identify non-starters (§6.2) and mechanical retirements (Question 3), so its labels must be grouped reliably (§4.5).

In [ ]:
st_counts = (raw['results']['statusId'].value_counts()
             .rename_axis('statusId').reset_index(name='rows')
             .merge(raw['status'], on='statusId'))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top = st_counts.head(20)[::-1]
ax[0].barh(top['status'], top['rows'], color=BLUE, height=0.65)
ax[0].set_title('20 most frequent status labels in results'); ax[0].grid(axis='y', visible=False)
pt = raw['results']['positionText']
codes = pt[~pt.str.isnumeric()].value_counts()
ax[1].bar(codes.index, codes.values, color=ORANGE, width=0.6)
ax[1].set_title('Non-numeric positionText codes')
for i, v in enumerate(codes.values):
    ax[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

say(f"**{raw['status'].shape[0]} distinct status labels**, {raw['status']['status'].duplicated().sum()} duplicated labels, "
    f"{(raw['status']['status'] != raw['status']['status'].str.strip()).sum()} labels with stray spaces. "
    "`positionText` codes: R = retired, F = failed to qualify, W = withdrew, N = not classified, D = disqualified, E = excluded.")

## 3.10 Numeric distributions and extreme values (temporary numeric copy)
Extreme values are investigated before deciding whether they are errors. Durations are converted with a *temporary* parser here; the real conversion is in §4.2.

In [ ]:
# Durations: the 'milliseconds' columns give a quick numeric view (the text is parsed and cross-checked in §4.2)
pts = tmp_num(raw['results']['points']); laps = tmp_num(raw['results']['laps'])
pit_s = tmp_num(raw['pit_stops']['milliseconds']) / 1000
lap_s = tmp_num(raw['lap_times']['milliseconds']) / 1000

fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
ax[0].hist(pts, bins=50, color=BLUE); ax[0].set_yscale('log'); ax[0].set_title('results.points'); ax[0].set_xlabel('points')
ax[1].hist(laps, bins=50, color=BLUE); ax[1].set_yscale('log'); ax[1].set_title('results.laps'); ax[1].set_xlabel('laps')
ax[2].hist(np.log10(pit_s), bins=60, color=BLUE); ax[2].set_title('pit_stops duration'); ax[2].set_xlabel('log10(seconds)')
ax[2].axvline(np.log10(60), color=ORANGE, ls='--', lw=1.5)
ax[3].hist(np.log10(lap_s), bins=80, color=BLUE); ax[3].set_title('lap_times lap time'); ax[3].set_xlabel('log10(seconds)')
ax[3].axvline(np.log10(300), color=ORANGE, ls='--', lw=1.5)
plt.tight_layout(); plt.show()

pit_text_colon = raw['pit_stops']['duration'].str.contains(':')
say(f"- **Points** reach **{pts.max():.0f}**; **{(pts > 25).sum()} rows exceed 25** (explained in §4.7).\n"
    f"- **Laps** reach **{laps.max():.0f}** — far more than a Grand Prix (examined in §4.7).\n"
    f"- **Pit stops:** median **{pit_s.median():.1f} s** (pit-lane drive + stop); **{(pit_s > 60).sum()} stops last over 60 s** "
    f"(max {pit_s.max() / 60:.0f} min). Exactly **{pit_text_colon.sum()}** durations are written as `m:ss.sss` — "
    f"the same stops: {bool((pit_text_colon == (pit_s > 60)).all())}. These are red-flag stoppages or repairs (P-06).\n"
    f"- **Lap times:** median **{lap_s.median():.1f} s**, **{(lap_s > 300).sum()} laps over 5 minutes** (max {lap_s.max() / 3600:.1f} h): "
    "safety-car and red-flag laps (P-07). Circuits differ in length, so raw lap times are not comparable across races.")

In [ ]:
# Drivers & a note for Question 2: circuit 'country' vs driver 'nationality' use different vocabularies
dob = pd.to_datetime(raw['drivers']['dob'].where(raw['drivers']['dob'].ne(SENTINEL)), format='%Y-%m-%d', errors='coerce')
nat = raw['drivers']['nationality'].value_counts().head(6).index.tolist()
ctry = raw['circuits']['country'].value_counts().head(6).index.tolist()
say(f"Drivers born **{dob.min():%Y}–{dob.max():%Y}**; {dob.isna().sum()} birth dates fail to parse. "
    f"Constructors with the same name: {raw['constructors']['name'].duplicated().sum()}. "
    f"Circuit `country` uses place names ({', '.join(ctry)}) while driver `nationality` uses demonyms "
    f"({', '.join(nat)}), so Question 2 (home advantage) will need an explicit country ↔ nationality mapping.")

## 3.11 Raw EDA — what we learned

| ID | Insight / problem | Consequence |
|---|---|---|
| I-01 | All 14 primary keys unique and non-null; all 23 foreign keys reconcile | Joins are safe; no key repair needed |
| P-01 | Missing values are stored as the text `\N` (and blanks in `qualifying.q2/q3`) | Convert to real `NaN` on copies (§4.1) |
| P-02 | Every column is text: dates, numbers and durations cannot be compared or averaged | Semantic typing via one schema (§4.2) |
| P-03 | Repeated (raceId, driverId) pairs in `results`, all in early seasons | Keep-rule needed (§4.4) |
| P-05 | 139 free-text status labels | Hand-checked grouping (§4.5) |
| P-06 / P-07 | Very long pit stops (> 60 s) and laps (> 5 min) are real red-flag events | Keep and flag (§4.6) |
| P-14 | Rows with > 25 points and > 100 laps | Explain before deciding (§4.7) |
| I-02 / I-03 | Podiums are ~13 % of rows; the base rate depends on field size and era | Imbalanced target, era-aware modeling |
| I-04 | Grid position is the strongest single signal seen so far | Core pre-race feature |
| I-11 | Coverage depends on era; qualifying complete only from 2003 | Modeling period 2003–2024 (§6.1) |

---
# 4. Cleaning (on copies only)

Order of operations (each step is checked with an assertion before the next one starts):

| Step | What | Output |
|---|---|---|
| C1 | Placeholders → `NaN`, strip outer spaces | `stage1[...]` |
| C2 | Semantic typing via one `SCHEMA` (ints, floats, dates, times of day, durations → seconds) | `typed[...]` |
| C3 | Drop the near-empty `constructor_results.status` | `typed[...]` |
| C4 | One row per (raceId, driverId) in `results` (keep-rule comparison) | `clean[...]` |
| C5 | Status labels → 10 groups + ambiguous flag | `clean['status']` |
| C6 | Flags for red-flag pit stops, outlier laps, impossible qualifying times | new boolean columns |
| C7 | Review of remaining anomalies (points, laps, fastest lap) | explanation only |
| C8 | Re-verify all 14 primary keys and 23 foreign keys on the cleaned tables | `key_verification` |

Nothing is imputed: a value that is missing in the source stays missing, because its absence is meaningful (era, format, or a driver not reaching Q3).

## 4.1 C1 — Placeholders → `NaN` (P-01)
`\N` and empty/whitespace-only cells become real missing values. The assertion proves that **the number of `NaN` created equals the number of placeholders counted in the raw EDA**, table by table — nothing more, nothing less.

In [ ]:
STRIPPED = []                                  # (table, column, raw value) of every cell that had outer spaces

def replace_sentinels(df, name):
    # Return a deep copy where placeholders are NaN and outer spaces are stripped.
    out = df.copy(deep=True)
    mask = sentinel_mask(out)
    out = out.mask(mask)                       # \N / blank -> NaN
    n_stripped = 0
    for col in out.columns:
        stripped = out[col].str.strip()
        changed = (out[col].notna() & stripped.ne(out[col])).fillna(False).astype(bool)
        n_stripped += int(changed.sum())
        STRIPPED.extend((name, col, repr(v)) for v in out.loc[changed, col].unique())
        out[col] = stripped
    return out, int(mask.values.sum()), n_stripped

stage1, c1_report = {}, []
for t in TABLES:
    stage1[t], n_sent, n_strip = replace_sentinels(raw[t], t)
    n_nan = int(stage1[t].isna().values.sum())
    assert n_nan == n_sent == eda_summary_raw.loc[t, N_COL] + eda_summary_raw.loc[t, 'n_blank'], t
    assert not stage1[t].isin([SENTINEL]).values.any(), t
    c1_report.append({'table': t, 'placeholders_before': n_sent, 'NaN_after': n_nan, 'cells_stripped': n_strip})
c1_report = pd.DataFrame(c1_report).set_index('table')
display(c1_report.T)

tot = int(c1_report['NaN_after'].sum())
log_decision('P-01', 'C1', f"{int(eda_summary_raw[N_COL].sum()):,} '\\N' + {int(eda_summary_raw['n_blank'].sum())} blank cells",
             'Replace \\N and empty/whitespace-only cells with NaN on copies',
             'Numeric operations on text placeholders fail or give wrong results',
             'Fill with 0 / mean (rejected: invents data)', f'{tot:,} NaN created = placeholders counted; 0 \\N left', 'A, B, C, D')
ws_note = ('; '.join(f'`{t}.{c}` = {v}' for t, c, v in STRIPPED[:5]) or 'none')
say(f"✅ **{tot:,} placeholders → {tot:,} `NaN`**; 0 `\\N` strings left. "
    f"Cells with stray outer spaces (stripped, so e.g. a nationality matches its other spellings): "
    f"**{int(c1_report['cells_stripped'].sum())}** → {ws_note}.")

## 4.2 C2 — Semantic typing (P-02)

One `SCHEMA` dictionary assigns a **meaning** to every column, and one function converts it while counting failures (a *failure* = a non-missing value that could not be converted; it must be 0):

| Kind | Rule | Examples |
|---|---|---|
| `int` | nullable integer `Int64` (keeps `NaN`) | ids, `grid`, `positionOrder`, `laps` |
| `float` | `float64` | `points`, `lat`, `lng`, `fastestLapSpeed` |
| `date` | `datetime64` with an **explicit** `%Y-%m-%d` format (no guessing) | `races.date`, `drivers.dob`, session dates |
| `tod` | time of day → `timedelta` | `races.time`, session times, `pit_stops.time` |
| `dur` | duration text → **seconds** in a new `*_s` column; the text column is dropped | `"1:26.572"` → 86.572 (`q1_s`, `lap_time_s`, `duration_s`, `fastestLapTime_s`) |
| `str` | pandas `string` dtype | names, refs, `positionText`, `results.time` |

**Kept as text on purpose:** `results.time` / `sprint_results.time` mix the winner's total time with gaps (`+5.478`, `+1 Lap`), so they are not one quantity; they are post-race anyway, and `milliseconds` already holds the numeric race time.

In [ ]:
def duration_to_seconds(s):
    # 'ss.sss' | 'm:ss.sss' | 'h:mm:ss.sss'  ->  float seconds  (NaN stays NaN).
    s = s.astype('string').str.strip()
    out = pd.Series(np.nan, index=s.index, dtype='float64')
    n_colon = s.str.count(':')
    for k in sorted(n_colon.dropna().unique()):
        rows = n_colon.eq(k).fillna(False).astype(bool)
        parts = s[rows].str.split(':', expand=True).apply(pd.to_numeric, errors='coerce')
        val = parts[0].astype('float64')
        for i in range(1, int(k) + 1):
            val = val * 60 + parts[i].astype('float64')
        out[rows] = val.values
    return out

# Unit tests on known examples (including the 1995 'typo' format and a red-flag pit stop)
_t = duration_to_seconds(pd.Series(['1:26.572', '23.456', '16:44.718', '1:02:03.500', None, '0:59.999']))
_expected = [86.572, 23.456, 1004.718, 3723.5, np.nan, 59.999]
assert np.allclose(_t.values, _expected, equal_nan=True), _t.tolist()
print('duration_to_seconds unit tests passed:', _t.round(3).tolist())

In [ ]:
I, F, D, T, U, S = 'int', 'float', 'date', 'tod', 'dur', 'str'
SCHEMA = {
    'circuits': dict(circuitId=I, circuitRef=S, name=S, location=S, country=S, lat=F, lng=F, alt=I, url=S),
    'constructors': dict(constructorId=I, constructorRef=S, name=S, nationality=S, url=S),
    'seasons': dict(year=I, url=S),
    'status': dict(statusId=I, status=S),
    'drivers': dict(driverId=I, driverRef=S, number=I, code=S, forename=S, surname=S, dob=D, nationality=S, url=S),
    'races': dict(raceId=I, year=I, round=I, circuitId=I, name=S, date=D, time=T, url=S,
                  fp1_date=D, fp1_time=T, fp2_date=D, fp2_time=T, fp3_date=D, fp3_time=T,
                  quali_date=D, quali_time=T, sprint_date=D, sprint_time=T),
    'results': dict(resultId=I, raceId=I, driverId=I, constructorId=I, number=I, grid=I, position=I,
                    positionText=S, positionOrder=I, points=F, laps=I, time=S, milliseconds=I,
                    fastestLap=I, rank=I, fastestLapTime=U, fastestLapSpeed=F, statusId=I),
    'sprint_results': dict(resultId=I, raceId=I, driverId=I, constructorId=I, number=I, grid=I, position=I,
                           positionText=S, positionOrder=I, points=F, laps=I, time=S, milliseconds=I,
                           fastestLap=I, rank=I, fastestLapTime=U, fastestLapSpeed=F, statusId=I),
    'qualifying': dict(qualifyId=I, raceId=I, driverId=I, constructorId=I, number=I, position=I, q1=U, q2=U, q3=U),
    'driver_standings': dict(driverStandingsId=I, raceId=I, driverId=I, points=F, position=I, positionText=S, wins=I),
    'constructor_standings': dict(constructorStandingsId=I, raceId=I, constructorId=I, points=F, position=I, positionText=S, wins=I),
    'constructor_results': dict(constructorResultsId=I, raceId=I, constructorId=I, points=F, status=S),
    'pit_stops': dict(raceId=I, driverId=I, stop=I, lap=I, time=T, duration=U, milliseconds=I),
    'lap_times': dict(raceId=I, driverId=I, lap=I, position=I, time=U, milliseconds=I),
}
# Name of the new seconds column for every duration column
DUR_NAME = {('results', 'fastestLapTime'): 'fastestLapTime_s', ('sprint_results', 'fastestLapTime'): 'fastestLapTime_s',
            ('qualifying', 'q1'): 'q1_s', ('qualifying', 'q2'): 'q2_s', ('qualifying', 'q3'): 'q3_s',
            ('pit_stops', 'duration'): 'duration_s', ('lap_times', 'time'): 'lap_time_s'}

def convert(s, kind):
    # Convert one column; return (converted Series, number of failed conversions).
    present = s.notna()
    if kind == I:
        num = pd.to_numeric(s, errors='coerce')
        bad = present & (num.isna() | (num % 1 != 0))
        out = num.where(~bad).astype('Int64')
    elif kind == F:
        out = pd.to_numeric(s, errors='coerce').astype('float64'); bad = present & out.isna()
    elif kind == D:
        out = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce'); bad = present & out.isna()
    elif kind == T:
        out = pd.to_timedelta(s, errors='coerce'); bad = present & out.isna()
    elif kind == U:
        out = duration_to_seconds(s); bad = present & out.isna()
    else:
        out = s.astype('string'); bad = pd.Series(False, index=s.index)
    return out, int(bad.sum())

typed, conv_log = {}, []
for t in TABLES:
    df = stage1[t]
    unknown = [c for c in df.columns if c not in SCHEMA[t]]
    if unknown:
        print(f'⚠️ {t}: columns not in SCHEMA kept as string → {unknown}')
    new = {}
    for col in df.columns:
        kind = SCHEMA[t].get(col, S)
        out, n_fail = convert(df[col], kind)
        name = DUR_NAME.get((t, col), col) if kind == U else col
        new[name] = out
        conv_log.append({'table': t, 'column': col, 'kind': kind, 'new_column': name, 'failures': n_fail,
                         'dtype': str(out.dtype)})
    typed[t] = pd.DataFrame(new, index=df.index)
conv_log = pd.DataFrame(conv_log)
assert conv_log['failures'].sum() == 0, conv_log.query('failures > 0')
n_typed = int((conv_log['kind'] != S).sum())
display(conv_log[conv_log['kind'] != S].groupby('kind').agg(columns=('column', 'count'),
        failures=('failures', 'sum'), examples=('new_column', lambda s: ', '.join(s.head(6)))))
say(f"✅ **{n_typed} columns** converted from text to numbers, dates, times of day or seconds with **0 conversion failures**.")

**Cross-check of the duration parser.** Lap and pit-stop durations are stored twice in the source: as text and as `milliseconds`. If the parser is right, the two agree everywhere.

In [ ]:
xc = pd.DataFrame({
    'lap_times': [len(typed['lap_times']), (typed['lap_times']['lap_time_s'] - typed['lap_times']['milliseconds'] / 1000).abs().max()],
    'pit_stops': [len(typed['pit_stops']), (typed['pit_stops']['duration_s'] - typed['pit_stops']['milliseconds'] / 1000).abs().max()],
}, index=['rows compared', 'max |parsed − milliseconds/1000| (s)']).T
display(xc)
assert (xc['max |parsed − milliseconds/1000| (s)'] < 0.0005).all()

mem_raw = sum(raw[t].memory_usage(deep=True).sum() for t in TABLES) / 1e6
mem_typed = sum(typed[t].memory_usage(deep=True).sum() for t in TABLES) / 1e6
log_decision('P-02', 'C2', f'{n_typed} columns hold numbers, dates, times or durations as text',
             'One SCHEMA: Int64 / float / explicit-format dates / timedelta / durations -> *_s seconds',
             'Typed values can be compared, sorted, averaged; explicit formats avoid silent mis-parsing',
             'Keep results.time as text (mixed totals and gaps); milliseconds holds the numeric value',
             f'0 failures; parser matches milliseconds to <0.5 ms; memory {mem_raw:.0f} MB -> {mem_typed:.0f} MB', 'A, B, C, D')
say(f"✅ Parsed durations match `milliseconds` on **{int(xc['rows compared'].sum()):,} rows** (max difference "
    f"{xc.iloc[:, 1].max():.4f} s). Memory: **{mem_raw:.0f} MB → {mem_typed:.0f} MB**.")

## 4.3 C3 — Drop `constructor_results.status` (P-15)
The column is almost empty and the few values it has describe the *outcome* of a race (a disqualification), i.e. post-race information.

In [ ]:
cr_status = typed['constructor_results']['status']
display(cr_status.value_counts(dropna=False).rename('rows').to_frame())
pct_empty = cr_status.isna().mean()
typed['constructor_results'] = typed['constructor_results'].drop(columns='status')
log_decision('P-15', 'C3', f'constructor_results.status is {pct_empty:.1%} empty ({cr_status.notna().sum()} values)',
             'Drop the column from the clean copy', 'Near-empty and post-race information', 'Keep as a flag',
             'Column removed; raw file unchanged', 'C')
say(f"`constructor_results.status` is **{pct_empty:.1%} empty**; its {cr_status.notna().sum()} values are all "
    f"`{cr_status.dropna().unique().tolist()}` — consistent with one team excluded for a whole season. Column dropped.")

## 4.4 C4 — One row per driver per race (P-03 / D-01)

Four keep-rules were proposed across the documents. They are applied to the **same duplicate groups** and compared on what matters for the target: how many podium labels they keep and how many they change.

| Rule | Sort order inside a (raceId, driverId) group | Proposed by |
|---|---|---|
| **best_finish (chosen)** | lowest `positionOrder` → more `laps` → lower `resultId` | B, C |
| most_laps | more `laps` → lowest `positionOrder` → lower `resultId` | alternative in B |
| points_then_laps | more `points` → more `laps` → lower `resultId` | D |
| first_row | lowest `resultId` | naive baseline |

In [ ]:
RULES = {
    'best_finish (chosen)':     (['positionOrder', 'laps', 'resultId'], [True, False, True]),
    'most_laps':                (['laps', 'positionOrder', 'resultId'], [False, True, True]),
    'points_then_laps (doc D)': (['points', 'laps', 'resultId'], [False, False, True]),
    'first_row':                (['resultId'], [True]),
}
KEY = ['raceId', 'driverId']
res_typed = typed['results']
dup_rows = res_typed[res_typed.duplicated(KEY, keep=False)].copy()
dup_rows['podium'] = (dup_rows['positionOrder'] <= 3).astype(int)

def apply_rule(df, rule):
    cols, asc = RULES[rule]
    return df.sort_values(cols, ascending=asc, na_position='last').drop_duplicates(KEY, keep='first')

kept = {rule: apply_rule(dup_rows, rule).set_index(KEY) for rule in RULES}
ref = kept['best_finish (chosen)']
cmp_rows = []
for rule, k in kept.items():
    k = k.loc[ref.index]
    full = apply_rule(res_typed, rule)
    cmp_rows.append({'rule': rule, 'pairs': len(k), 'podiums kept in duplicate pairs': int(k['podium'].sum()),
                     'pairs keeping a different row than chosen': int((k['resultId'] != ref['resultId']).sum()),
                     'podium labels changed vs chosen': int((k['podium'] != ref['podium']).sum()),
                     'rows after dedup': len(full),
                     'total podium rows after dedup': int((full['positionOrder'] <= 3).sum())})
dedup_cmp = pd.DataFrame(cmp_rows).set_index('rule')
display(dedup_cmp)

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.barh(dedup_cmp.index[::-1], dedup_cmp['podiums kept in duplicate pairs'][::-1],
        color=[BLUE if 'chosen' in i else GREY for i in dedup_cmp.index[::-1]], height=0.55)
ax.set_title('Podium rows kept inside the duplicate pairs, by keep-rule'); ax.grid(axis='y', visible=False)
for y, v in enumerate(dedup_cmp['podiums kept in duplicate pairs'][::-1]):
    ax.text(v + 0.2, y, str(v), va='center')
plt.tight_layout(); plt.show()

In [ ]:
# Apply the chosen rule to the full results table
clean = {t: typed[t].copy(deep=True) for t in TABLES}
clean['results'] = apply_rule(res_typed, 'best_finish (chosen)').sort_values('resultId').reset_index(drop=True)
assert not clean['results'].duplicated(KEY).any()

# The other driver-level tables must already be unique at the (raceId, driverId) grain
for t in ['qualifying', 'sprint_results', 'driver_standings']:
    assert not clean[t].duplicated(KEY).any(), f'{t} has duplicate (raceId, driverId)'

removed = len(res_typed) - len(clean['results'])
row_c = dedup_cmp.loc['best_finish (chosen)']
row_d = dedup_cmp.loc['points_then_laps (doc D)']
row_f = dedup_cmp.loc['first_row']
log_decision('P-03 / D-01', 'C4', f"{len(ref)} repeated (raceId, driverId) pairs, {len(dup_rows)} rows",
             'Keep the row with the best positionOrder (ties: more laps, then lower resultId)',
             'The label is defined by positionOrder; this keeps the podium the driver is credited with',
             f"most_laps: {int(dedup_cmp.loc['most_laps', 'podium labels changed vs chosen'])} labels changed; "
             f"points_then_laps: {int(row_d['podium labels changed vs chosen'])}; first_row: {int(row_f['podium labels changed vs chosen'])}",
             f'{removed} rows removed; (raceId, driverId) unique', 'B, C (D proposed points_then_laps)')
say(f"✅ **{removed} rows removed**, (raceId, driverId) is now unique ({len(clean['results']):,} rows). "
    f"**Why best finish:** the target is `positionOrder <= 3`, so the rule should keep the row that decides the label. "
    f"It keeps **{int(row_c['podiums kept in duplicate pairs'])}** podiums inside the duplicate pairs; "
    f"`most_laps` changes {int(dedup_cmp.loc['most_laps', 'podium labels changed vs chosen'])} labels, "
    f"document D's `points_then_laps` changes **{int(row_d['podium labels changed vs chosen'])}**, "
    f"and `first_row` changes **{int(row_f['podium labels changed vs chosen'])}** — so the rule matters, and best finish preserves credited podiums. "
    f"(None of these pairs is in the 2003–2024 modeling period, so the choice does not affect the model; it is documented for completeness.)")

## 4.5 C5 — Status labels → 10 groups (P-05 / D-05, document B's map)

A keyword search makes visible mistakes (e.g. "Driver Seat" counted as a driver incident, "Front wing" left unclassified), so **every label is mapped by hand**. Labels whose group is debatable get `status_ambiguous = True`, so Question 3 can show its ranking with and without them (sensitivity check).

Grouping choices (document B): wings, seats and generic "Technical" → **Mechanical**; tyres, punctures and wheels → **Tyre/wheel**; wheel nuts, refuelling, fuel-rig and out-of-fuel → **Team/operational**; "Did not qualify / prequalify", "Withdrew" and "107% Rule" → **Did not start**.

In [ ]:
STATUS_GROUPS = {
    'Finished': ['Finished'],
    'Mechanical': ['Engine', 'Gearbox', 'Transmission', 'Clutch', 'Hydraulics', 'Electrical', 'Radiator', 'Suspension',
                   'Brakes', 'Differential', 'Overheating', 'Mechanical', 'Driveshaft', 'Fuel pressure', 'Water pressure',
                   'Throttle', 'Steering', 'Technical', 'Electronics', 'Heat shield fire', 'Exhaust', 'Oil leak',
                   'Water leak', 'Fuel pump', 'Track rod', 'Oil pressure', 'Engine fire', 'Engine misfire', 'Pneumatics',
                   'Handling', 'Fire', 'Wheel bearing', 'Fuel system', 'Oil line', 'Launch control', 'Fuel', 'Power loss',
                   'Vibrations', 'Drivetrain', 'Ignition', 'Chassis', 'Battery', 'Halfshaft', 'Crankshaft', 'Alternator',
                   'Oil pump', 'Fuel leak', 'Injection', 'Distributor', 'Turbo', 'CV joint', 'Water pump', 'Spark plugs',
                   'Fuel pipe', 'Oil pipe', 'Axle', 'Water pipe', 'Magneto', 'Supercharger', 'Power Unit', 'ERS',
                   'Brake duct', 'Undertray', 'Cooling system', 'Front wing', 'Broken wing', 'Rear wing',
                   'Driver Seat', 'Seat', 'Stalled'],
    'Tyre/wheel': ['Tyre', 'Puncture', 'Tyre puncture', 'Wheel', 'Wheel rim'],
    'Team/operational': ['Wheel nut', 'Refuelling', 'Fuel rig', 'Out of fuel', 'Safety belt'],
    'Accident/incident': ['Accident', 'Collision', 'Spun off', 'Collision damage', 'Damage', 'Debris', 'Fatal accident'],
    'Driver health': ['Physical', 'Injured', 'Injury', 'Eye injury', 'Driver unwell', 'Illness'],
    'Regulatory': ['Disqualified', 'Excluded', 'Underweight'],
    'Did not start': ['Did not qualify', 'Did not prequalify', 'Withdrew', '107% Rule', 'Did not start'],
    'Other/unknown': ['Retired', 'Not classified', 'Not restarted', 'Safety', 'Safety concerns'],
}
LAPPED = re.compile(r'^\+\d+ Laps?$')      # '+1 Lap', '+12 Laps' -> finished but lapped
AMBIGUOUS = {'Technical', 'Handling', 'Stalled', 'Fire', 'Front wing', 'Broken wing', 'Rear wing', 'Driver Seat', 'Seat',
             'Undertray', 'Vibrations', 'Tyre', 'Puncture', 'Tyre puncture', 'Wheel', 'Wheel rim', 'Wheel nut',
             'Out of fuel', 'Safety belt', 'Debris', 'Damage', 'Retired', 'Not classified'}
GROUP_ORDER = ['Finished', 'Lapped', 'Mechanical', 'Tyre/wheel', 'Team/operational', 'Accident/incident',
               'Driver health', 'Regulatory', 'Did not start', 'Other/unknown']

label_to_group = {lbl: g for g, lbls in STATUS_GROUPS.items() for lbl in lbls}
assert len(label_to_group) == sum(len(v) for v in STATUS_GROUPS.values()), 'a label is listed in two groups'

st = clean['status']
st['status_group'] = st['status'].map(lambda s: 'Lapped' if LAPPED.match(s) else label_to_group.get(s)).astype('string')
st['status_ambiguous'] = st['status'].isin(AMBIGUOUS)
unmapped = st.loc[st['status_group'].isna(), 'status'].tolist()
assert not unmapped, f'Unmapped status labels: {unmapped}'
assert len(GROUP_ORDER) == 10 and set(st['status_group']) <= set(GROUP_ORDER)

usage = (clean['results'].merge(st, on='statusId')
         .groupby('status_group').agg(rows=('resultId', 'size'), labels=('statusId', 'nunique'))
         .reindex(GROUP_ORDER))
usage['share'] = usage['rows'] / usage['rows'].sum()
display(usage.style.format({'share': '{:.1%}'}))

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.barh(usage.index[::-1], usage['share'][::-1], color=BLUE, height=0.6)
pct_axis(ax, 'x'); ax.grid(axis='y', visible=False)
ax.set_title('Outcome of every driver-race row, by status group')
for y, v in enumerate(usage['share'][::-1]):
    ax.text(v + 0.003, y, f'{v:.1%}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

amb_rows = int(clean['results']['statusId'].isin(st.loc[st['status_ambiguous'], 'statusId']).sum())
log_decision('P-05 / D-05', 'C5', f'{len(st)} free-text status labels', 'Hand-checked map into 10 groups + ambiguous flag (document B)',
             'Needed for non-starters (§6.2) and Question 3; keyword grouping produced visible errors',
             "Tyres/wheel nuts as mechanical; wings as accidents; document C's 'other' bucket",
             f'0 labels unmapped; {int(st["status_ambiguous"].sum())} ambiguous labels cover {amb_rows} rows', 'B (C)')
say(f"✅ All **{len(st)}** labels mapped to exactly one of 10 groups. **{int(st['status_ambiguous'].sum())} ambiguous labels cover only "
    f"{amb_rows} rows ({amb_rows / len(clean['results']):.1%})**, so they cannot change Question 3's conclusions much — "
    "but the flag allows a with/without check.")

## 4.6 C6 — Flags instead of deletions (P-06, P-07, P-12 / D-04)

Very long pit stops and laps are **real events** (red flags, safety cars, long repairs), not data errors, so they are **kept** and flagged; a later pit-stop or pace feature can then exclude them or use medians. Thresholds are chosen from the distributions in §3.10:

* `is_red_flag_stop` — pit-stop duration **> 60 s** (a normal stop including pit-lane drive is ~20–30 s; > 60 s coincides exactly with the `m:ss` format in the raw file).
* `is_outlier_lap` — lap time **> 300 s** (no racing lap on the calendar takes 5 minutes).
* `q_time_suspect` — any qualifying time **> 300 s** (impossible for a flying lap; e.g. the 1995 Japanese GP Q1 of ~1,002 s, probably a typo for 1:42.640). The value is **not edited** — we do not know the true value, and it lies outside the modeling period.

In [ ]:
ps, lt, qd = clean['pit_stops'], clean['lap_times'], clean['qualifying']
ps['is_red_flag_stop'] = ps['duration_s'] > 60
lt['is_outlier_lap'] = lt['lap_time_s'] > 300
qd['q_time_suspect'] = (qd[['q1_s', 'q2_s', 'q3_s']] > 300).any(axis=1)

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].hist(ps['duration_s'].clip(upper=120), bins=80, color=BLUE)
ax[0].axvline(60, color=ORANGE, ls='--'); ax[0].set_title('Pit-stop duration (clipped at 120 s)'); ax[0].set_xlabel('seconds')
ax[0].text(62, ax[0].get_ylim()[1] * 0.8, f"{ps['is_red_flag_stop'].sum()} flagged", color=ORANGE)
ax[1].hist(lt['lap_time_s'].clip(upper=360), bins=120, color=BLUE)
ax[1].axvline(300, color=ORANGE, ls='--'); ax[1].set_yscale('log'); ax[1].set_title('Lap time (clipped at 360 s, log count)')
ax[1].text(240, ax[1].get_ylim()[1] * 0.3, f"{lt['is_outlier_lap'].sum()} flagged", color=ORANGE)
ax[1].set_xlabel('seconds')
plt.tight_layout(); plt.show()

sus = qd[qd['q_time_suspect']].merge(clean['races'][['raceId', 'year', 'name']], on='raceId') \
        .merge(clean['drivers'][['driverId', 'surname']], on='driverId')
display(sus[['year', 'name', 'surname', 'position', 'q1_s', 'q2_s', 'q3_s']])
for id_, f, n in [('P-06', 'pit stops > 60 s', ps['is_red_flag_stop'].sum()), ('P-07', 'laps > 300 s', lt['is_outlier_lap'].sum()),
                  ('P-12', 'qualifying times > 300 s', qd['q_time_suspect'].sum())]:
    log_decision(id_, 'C6', f'{n} {f}', 'Keep the value, add a boolean flag', 'Real events (or unknown true value); flag keeps every option open',
                 'Set to NaN; drop rows', f'{n} rows flagged, 0 values changed', 'B (C)')
say(f"✅ Flagged **{ps['is_red_flag_stop'].sum()} pit stops**, **{lt['is_outlier_lap'].sum()} laps** and "
    f"**{qd['q_time_suspect'].sum()} qualifying row(s)**; no value was changed or removed.")

## 4.7 C7 — Review of remaining anomalies (P-14, document D's open items)
Before calling something an error we check whether the F1 rules explain it.

In [ ]:
rr = clean['results'].merge(clean['races'][['raceId', 'year', 'name']], on='raceId')

# (a) more than 25 points in one race
hi = rr[rr['points'] > 25]
display(hi.groupby(['year', 'name'])['points'].agg(rows='size', max_points='max').reset_index()
          .sort_values('rows', ascending=False).head(10))
fl = hi[(hi['points'] == 26)]
say(f"**(a) {len(hi)} rows have > 25 points.** {len(fl)} of them are exactly 26 = 25 for a win + 1 bonus point for the fastest lap "
    f"(awarded 2019–2024); the others come from the **2014 Abu Dhabi double-points finale** (up to 50) — real rules, not errors. "
    "Raw points are therefore not comparable across seasons (P-11): later features should use positions or shares, not raw points.")

In [ ]:
# (b) more than 100 laps: the Indianapolis 500 counted for the championship 1950–1960
long_races = rr[rr['laps'] > 100].groupby(['year', 'name'])['laps'].max().reset_index()
display(long_races)
indy = long_races[long_races['name'].str.contains('Indianapolis')]
say(f"**(b)** {len(long_races)} races have more than 100 laps ({long_races['year'].min()}–{long_races['year'].max()}): "
    f"{len(indy)} are the **Indianapolis 500** (up to {int(indy['laps'].max()) if len(indy) else '-'} laps; it counted for the championship 1950–1960), "
    f"the rest are early Grands Prix on short circuits ({', '.join(sorted(set(long_races['name']) - set(indy['name'])))}). "
    "Real races, all outside the modeling period → kept.")

# (c) fastest-lap time present without its rank, or vice versa
inc = rr[rr['fastestLapTime_s'].notna() != rr['rank'].notna()]
display(inc.groupby('year').size().rename('rows').to_frame().T)
say(f"**(c) {len(inc)} rows** have a fastest-lap time without a rank or the reverse, all in "
    f"{inc['year'].min()}–{inc['year'].max()} — a recording inconsistency in post-race columns that are never used as features. Kept as-is.")
log_decision('P-14', 'C7', f'{len(hi)} rows > 25 points; {len(long_races)} races > 100 laps; {len(inc)} fastest-lap/rank mismatches',
             'Explain, do not change', 'Fastest-lap bonus (26), 2014 double points, Indy 500; post-race columns',
             'Cap or set to NaN (rejected: they are real)', '0 values changed', 'C, D')

## 4.8 C8 — Re-verify every key on the cleaned tables (P-17 / I-01)
Cleaning changed types and removed rows, so all **14 primary keys** and **23 foreign keys** are checked again. A typed integer key that silently became `NaN` would fail here.

In [ ]:
key_rows = []
for t in TABLES:
    pk = PK[t]
    key_rows.append({'check': f'PK {t} ({"+".join(pk)})', 'type': 'primary key',
                     'violations': int(clean[t].duplicated(pk).sum() + clean[t][pk].isna().any(axis=1).sum())})
for f in FK:
    r_ = fk_check(clean, *f, is_raw=False)
    key_rows.append({'check': r_['link'], 'type': 'foreign key', 'violations': r_['orphans']})
key_verification = pd.DataFrame(key_rows)
key_verification['passed'] = key_verification['violations'] == 0
display(key_verification.groupby('type')['passed'].agg(['sum', 'count']).rename(columns={'sum': 'passed', 'count': 'checks'}))
assert key_verification['passed'].all(), key_verification[~key_verification['passed']]

# Race calendar checks again, now on typed columns
rc = clean['races']
assert (rc['date'].dt.year == rc['year']).all() and rc['date'].notna().all()
say(f"✅ **{key_verification['passed'].sum()}/{len(key_verification)} key checks pass** on the cleaned tables "
    "(14 primary keys + 23 foreign keys); every race date is valid and agrees with its season.")

---
# 5. EDA **after** cleaning: what changed, and what it made possible

## 5.1 Before vs after, table by table

In [ ]:
def is_text(s):
    return pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)

ba = []
for t in TABLES:
    r_, c_ = raw[t], clean[t]
    text_cols = [c for c in c_.columns if is_text(c_[c])]
    ba.append({'table': t, 'rows_raw': len(r_), 'rows_clean': len(c_), 'rows_removed': len(r_) - len(c_),
               'placeholders_raw': int(eda_summary_raw.loc[t, N_COL] + eda_summary_raw.loc[t, 'n_blank']),
               '\\N_left': int(sum(c_[c].eq(SENTINEL).sum() for c in text_cols)),
               'NaN_clean': int(c_.drop(columns=[c for c in c_.columns if c_[c].dtype == bool]).isna().values.sum()),
               'typed_cols_raw': 0, 'typed_cols_clean': int(sum(not is_text(c_[c]) for c in c_.columns)),
               'memory_raw_mb': round(r_.memory_usage(deep=True).sum() / 1e6, 1),
               'memory_clean_mb': round(c_.memory_usage(deep=True).sum() / 1e6, 1)})
cleaning_before_after = pd.DataFrame(ba).set_index('table')
display(cleaning_before_after)
assert cleaning_before_after['\\N_left'].sum() == 0

# NaN accounting: clean NaN = placeholders − NaN that left with the removed duplicate rows − NaN in the dropped column
nan_in_removed_rows = int(typed['results'].isna().values.sum() - clean['results'].isna().values.sum())
nan_in_dropped_col = int(cr_status.isna().sum())
assert cleaning_before_after['NaN_clean'].sum() == (cleaning_before_after['placeholders_raw'].sum()
                                                    - nan_in_removed_rows - nan_in_dropped_col)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(TABLES)); w = 0.38
ax[0].bar(x - w / 2, cleaning_before_after['placeholders_raw'], w, label='placeholders (raw text)', color=GREY)
ax[0].bar(x + w / 2, cleaning_before_after['NaN_clean'], w, label='NaN (clean)', color=BLUE)
ax[0].set_yscale('symlog'); ax[0].set_xticks(x, TABLES, rotation=60, ha='right'); ax[0].legend(frameon=False)
ax[0].set_title('Missing values: hidden text placeholders → real NaN')
ax[1].bar(x - w / 2, cleaning_before_after['memory_raw_mb'], w, label='raw (all text)', color=GREY)
ax[1].bar(x + w / 2, cleaning_before_after['memory_clean_mb'], w, label='clean (typed)', color=BLUE)
ax[1].set_yscale('symlog'); ax[1].set_xticks(x, TABLES, rotation=60, ha='right'); ax[1].legend(frameon=False)
ax[1].set_title('Memory (MB)')
plt.tight_layout(); plt.show()

say(f"Rows removed: **{cleaning_before_after['rows_removed'].sum()}** (all duplicate `results` rows). "
    f"Typed columns: **0 → {cleaning_before_after['typed_cols_clean'].sum()}**. "
    f"Memory: **{cleaning_before_after['memory_raw_mb'].sum():.0f} MB → {cleaning_before_after['memory_clean_mb'].sum():.0f} MB**. "
    f"NaN accounting (asserted): {cleaning_before_after['placeholders_raw'].sum():,} placeholders − {nan_in_removed_rows} in removed duplicate rows "
    f"− {nan_in_dropped_col:,} in the dropped `constructor_results.status` = **{cleaning_before_after['NaN_clean'].sum():,} NaN**.")

## 5.2 Missingness was preserved exactly (document A's check)
For every raw column, the share of *semantic* missing values (placeholders) must equal the share of `NaN` after typing — proof that typing neither invented nor lost a missing value. (Compared before deduplication, so the row sets are identical.)

In [ ]:
mm = []
for t in TABLES:
    rm = sentinel_mask(raw[t]).sum()
    for col in raw[t].columns:
        new = DUR_NAME.get((t, col), col)
        if new not in typed[t].columns:            # dropped column (constructor_results.status)
            continue
        mm.append({'table': t, 'column': col, 'raw_missing': int(rm[col]), 'typed_NaN': int(typed[t][new].isna().sum())})
mm = pd.DataFrame(mm)
mismatch = mm[mm['raw_missing'] != mm['typed_NaN']]
assert mismatch.empty, mismatch
say(f"✅ **{len(mm)} columns compared, 0 differences** between raw placeholder counts and typed `NaN` counts.")

## 5.3 Missing values that remain, and why they are fine

In [ ]:
REASON = {
    ('results', 'position'): ('structural', 'non-finishers have no finishing position; the target uses positionOrder (always filled)'),
    ('results', 'time'): ('structural', 'only lead-lap finishers get a race time; post-race'),
    ('results', 'milliseconds'): ('structural', 'same as results.time; post-race'),
    ('results', 'fastestLap'): ('recording gap', 'fastest-lap data recorded from 2004 only'),
    ('results', 'rank'): ('recording gap', 'fastest-lap data recorded from 2004 only'),
    ('results', 'fastestLapTime_s'): ('recording gap', 'fastest-lap data recorded from 2004 only'),
    ('results', 'fastestLapSpeed'): ('recording gap', 'fastest-lap data recorded from 2004 only'),
    ('results', 'number'): ('recording gap', 'car number missing for a handful of early entries'),
    ('qualifying', 'q1_s'): ('structural', 'no time set (crash, technical issue)'),
    ('qualifying', 'q2_s'): ('structural + format', 'only faster drivers advance; no Q2 before 2006'),
    ('qualifying', 'q3_s'): ('structural + format', 'only the top 10 advance; no Q3 before 2006'),
    ('drivers', 'number'): ('structural', 'permanent numbers only exist from 2014'),
    ('drivers', 'code'): ('structural', 'three-letter codes are a modern convention'),
    ('races', 'time'): ('recording gap', 'race start time recorded from 2005'),
}
rem = []
for t in TABLES:
    na = clean[t].isna().mean()
    for col, p in na[na > 0].items():
        if t == 'races' and re.match(r'(fp\d|quali|sprint)_(date|time)', col):
            reason = ('recording gap / format', 'weekend-session data only for recent seasons; sprint and FP3 follow the weekend format')
        elif t == 'sprint_results':
            reason = ('structural', 'as in results (non-finishers / fastest lap)')
        else:
            reason = REASON.get((t, col), ('see EDA', ''))
        rem.append({'column': f'{t}.{col}', 'pct_missing': p, 'type': reason[0], 'why it stays missing': reason[1]})
remaining = pd.DataFrame(rem).sort_values('pct_missing', ascending=False).reset_index(drop=True)
display(remaining.style.format({'pct_missing': '{:.1%}'}))

fig, ax = plt.subplots(figsize=(11, max(4, 0.25 * len(remaining))))
ax.barh(remaining['column'][::-1], remaining['pct_missing'][::-1], color=BLUE, height=0.65)
pct_axis(ax, 'x'); ax.grid(axis='y', visible=False); ax.set_title('Share missing per column after cleaning (all kept, none imputed)')
plt.tight_layout(); plt.show()

## 5.4 Analyses that typing made possible
These were impossible on raw text: ages need dates, podium curves need integers, qualifying-time trends need seconds.

In [ ]:
# (a) Driver age at each race — also validates the date conversion of races.date and drivers.dob
age = (clean['results'][['raceId', 'driverId']]
       .merge(clean['races'][['raceId', 'date']], on='raceId')
       .merge(clean['drivers'][['driverId', 'dob']], on='driverId'))
age['age'] = (age['date'] - age['dob']).dt.days / 365.25

# (b) podium rate by grid and by qualifying position (2003–2024, where qualifying is complete)
rg = (clean['results'].merge(clean['races'][['raceId', 'year']], on='raceId')
      .merge(clean['qualifying'][['raceId', 'driverId', 'position']].rename(columns={'position': 'quali_pos'}),
             on=['raceId', 'driverId'], how='left'))
rg['podium'] = (rg['positionOrder'] <= 3).astype(int)
modern = rg[rg['year'] >= 2003]
by_grid = modern[modern['grid'].between(1, 20)].groupby('grid')['podium'].mean()
by_q = modern[modern['quali_pos'].between(1, 20)].groupby('quali_pos')['podium'].mean()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(age['age'].dropna(), bins=40, color=BLUE)
ax[0].set_title('Driver age at race'); ax[0].set_xlabel('years')
ax[1].plot(by_grid.index, by_grid.values, marker='o', ms=4, label='by grid position', color=BLUE)
ax[1].plot(by_q.index, by_q.values, marker='s', ms=4, label='by qualifying position', color=ORANGE)
pct_axis(ax[1]); ax[1].set_xticks(range(1, 21)); ax[1].legend(frameon=False)
ax[1].set_title('Podium rate by starting position, 2003–2024'); ax[1].set_xlabel('position')
plt.tight_layout(); plt.show()

diff = modern.dropna(subset=['quali_pos'])
diff = diff[diff['grid'] > 0]
say(f"- **Driver age at race:** {age['age'].min():.1f}–{age['age'].max():.1f} years (median {age['age'].median():.1f}), a plausible range "
    "that confirms both date columns were parsed correctly.\n"
    f"- **Podium rate from pole (2003–2024): {by_grid.loc[1]:.0%} by grid, {by_q.loc[1]:.0%} by qualifying**; "
    f"P3 → {by_grid.loc[3]:.0%} / {by_q.loc[3]:.0%}; P10 → {by_grid.loc[10]:.0%}. Both curves fall steeply (I-04, I-05).\n"
    f"- **Grid ≠ qualifying position in {(diff['grid'] != diff['quali_pos']).mean():.1%}** of 2003–2024 rows (mainly grid penalties), "
    "so both are kept: qualifying measures pace, grid is where the car actually starts (I-06; needed for Question 1).")

In [ ]:
# (c) pole time per season and (d) reliability over time
pole = (clean['qualifying'].assign(best_s=clean['qualifying'][['q1_s', 'q2_s', 'q3_s']].min(axis=1))
        .query('position == 1 and not q_time_suspect').merge(clean['races'][['raceId', 'year']], on='raceId'))
pole_med = pole.groupby('year')['best_s'].median()

rs = (clean['results'].merge(clean['status'], on='statusId').merge(clean['races'][['raceId', 'year']], on='raceId'))
starters = rs[rs['status_group'] != 'Did not start']
rel = starters.groupby('year').agg(finished=('status_group', lambda s: s.isin(['Finished', 'Lapped']).mean()),
                                   mechanical=('status_group', lambda s: s.eq('Mechanical').mean()))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(pole_med.index, pole_med.values, marker='o', ms=3, color=BLUE)
ax[0].set_title('Median pole time per season'); ax[0].set_ylabel('seconds')
ax[1].plot(rel.index, rel['finished'], label='finished (incl. lapped)', color=BLUE)
ax[1].plot(rel.index, rel['mechanical'], label='mechanical retirement', color=ORANGE)
pct_axis(ax[1]); ax[1].legend(frameon=False); ax[1].set_title('Outcome of starters per season')
plt.tight_layout(); plt.show()

say(f"- **Pole times** swing between {pole_med.min():.0f} s and {pole_med.max():.0f} s per season, driven by rule eras and the circuit mix → "
    "qualifying must be expressed **relative to the same race** (e.g. gap to pole), never in raw seconds.\n"
    f"- **Reliability changed drastically (I-09):** finishing share of starters was {rel.loc[:1989, 'finished'].min():.0%}–"
    f"{rel.loc[:1989, 'finished'].max():.0%} before 1990 and {rel.loc[2015:, 'finished'].min():.0%}–{rel.loc[2015:, 'finished'].max():.0%} since 2015; "
    "mechanical retirements fell accordingly. Old podiums depended on survival — another reason to model recent seasons.")

## 5.5 Leakage evidence: standings already include the current race (P-08)
`driver_standings` is recorded **after** each race. If the jump in a driver's points from the previous round equals the points they scored in *this* race (+ sprint), the table contains the race result and must only be used **lagged by one round** in feature engineering.

In [ ]:
ds = (clean['driver_standings'].merge(clean['races'][['raceId', 'year', 'round']], on='raceId')
      .sort_values(['driverId', 'year', 'round']))
ds['prev_points'] = ds.groupby(['driverId', 'year'])['points'].shift(1).fillna(0)
ds['jump'] = ds['points'] - ds['prev_points']
# Use the un-deduplicated results so shared-drive points are fully counted
gp = typed['results'].groupby(['raceId', 'driverId'])['points'].sum()
sp = typed['sprint_results'].groupby(['raceId', 'driverId'])['points'].sum()
race_pts = gp.add(sp, fill_value=0).rename('race_points')
ds = ds.join(race_pts, on=['raceId', 'driverId'])
ds['match'] = np.isclose(ds['jump'], ds['race_points'].fillna(0))
share = ds['match'].mean()
by_era = ds.groupby(pd.cut(ds['year'].astype(int), [1949, 1990, 2002, 2024], labels=['1950–1990', '1991–2002', '2003–2024']), observed=True)['match'].mean()
display(by_era.rename('standings jump = same-race points').to_frame().T.style.format('{:.1%}'))
log_decision('P-08', '5.5', f'Standings jump equals same-race points in {share:.1%} of rows',
             'Use standings from the previous round only (feature engineering, Part 3)',
             'Current-round standings contain the race result (leakage)', '—', 'Documented; implemented in Part 3', 'B, C')
say(f"In **{share:.1%}** of standings rows the jump equals the same-race points (the rest are mostly early seasons where only the best results counted). "
    "→ **Standings are post-race: only the previous round may be used as a feature.**")

## 5.6 Diagnostic: races without exactly three podium rows (P-16, document C's open issue)
Deduplication removes repeated *drivers*, so it cannot fix a race where two **different** drivers share a credited position (shared cars in the 1950s).

In [ ]:
pod_per_race = (clean['results'].assign(podium=clean['results']['positionOrder'] <= 3)
                .groupby('raceId')['podium'].sum())
odd = pod_per_race[pod_per_race != 3]
dup_pos = clean['results'].duplicated(['raceId', 'positionOrder'], keep=False)
odd_tbl = (clean['races'].set_index('raceId').loc[odd.index, ['year', 'name']]
           .assign(podium_rows=odd, rows_sharing_a_positionOrder=clean['results'][dup_pos].groupby('raceId').size()
                   .reindex(odd.index).fillna(0).astype(int)))
display(odd_tbl.sort_values('year'))
explained = (odd_tbl['rows_sharing_a_positionOrder'] > 0).mean() if len(odd_tbl) else 1.0
say(f"**{len(odd)} races** do not have exactly 3 podium rows (seasons {odd_tbl['year'].min() if len(odd) else '-'}–"
    f"{odd_tbl['year'].max() if len(odd) else '-'}). In **{explained:.0%}** of them two drivers share a `positionOrder` (shared car) "
    "or the race has missing positions. Each driver's label is still correct by our definition (`positionOrder <= 3`), "
    "and **none of these races is in the 2003–2024 modeling period** (asserted in §6.4).")

---
# 6. Base driver–race modeling table (2003–2024)

The milestone's unit of analysis is *one driver entering one race*. This section turns the cleaned `results` into that table: **one row per (raceId, driverId), keys + pre-race columns + the target only**. Features (form, lagged standings, relative qualifying gaps…) are added in Part 3.

## 6.1 Choosing the modeling period (P-09 / D-02)
Qualifying position is the strongest pre-race signal after grid (§5.4), so the period should be one in which qualifying exists for (almost) every row. The table compares the candidate periods on the same rows (non-starters already excluded, see §6.2).

In [ ]:
base_all = (clean['results']
            .merge(clean['status'][['statusId', 'status', 'status_group']], on='statusId', how='left')
            .merge(clean['races'][['raceId', 'year', 'round', 'date', 'circuitId']], on='raceId', how='left')
            .merge(clean['qualifying'][['raceId', 'driverId', 'position', 'q1_s', 'q2_s', 'q3_s']]
                   .rename(columns={'position': 'quali_position'}), on=['raceId', 'driverId'], how='left'))
base_all['podium'] = (base_all['positionOrder'] <= 3).astype('int8')

# Non-starter rule (D-03): status in the 'Did not start' group OR (grid = 0 and laps = 0)
in_list = base_all['status_group'].eq('Did not start').fillna(False).astype(bool)
no_start = (base_all['grid'].eq(0) & base_all['laps'].eq(0)).fillna(False).astype(bool)
base_all['is_non_starter'] = in_list | no_start
starters_all = base_all[~base_all['is_non_starter']]

periods = []
for start in [1950, 1994, 2003, 2014]:
    sub = starters_all[starters_all['year'].between(start, 2024)]
    periods.append({'period': f'{start}–2024', 'rows': len(sub), 'races': sub['raceId'].nunique(),
                    'rows with qualifying position': sub['quali_position'].notna().mean(),
                    'podium rate': sub['podium'].mean()})
periods = pd.DataFrame(periods).set_index('period')
display(periods.style.format({'rows with qualifying position': '{:.1%}', 'podium rate': '{:.1%}', 'rows': '{:,}'}))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
cols_ = [BLUE if p == '2003–2024' else GREY for p in periods.index]
ax[0].bar(periods.index, periods['rows'], color=cols_, width=0.55); ax[0].set_title('Rows available')
ax[1].bar(periods.index, periods['rows with qualifying position'], color=cols_, width=0.55)
pct_axis(ax[1]); ax[1].set_title('Share of rows with a qualifying position')
plt.tight_layout(); plt.show()

p03 = periods.loc['2003–2024']
log_decision('P-09 / D-02', '6.1', f"qualifying missing before 1994 and patchy until 2002",
             'Model on seasons 2003–2024', 'Qualifying available for almost every row; keeps far more rows than 2014+',
             'All seasons + missing flags; 1994+; 2014+ (hybrid era) — can be tested as an experiment in Part 5',
             f"{int(p03['rows']):,} rows, {int(p03['races'])} races, {p03['rows with qualifying position']:.1%} with qualifying", 'B, C')
say(f"**Decision: 2003–2024.** It gives **{int(p03['rows']):,} rows over {int(p03['races'])} races** with a qualifying position for "
    f"**{p03['rows with qualifying position']:.1%}** of rows, versus {periods.loc['1994–2024', 'rows with qualifying position']:.1%} for 1994+ "
    f"and only {int(periods.loc['2014–2024', 'rows']):,} rows for the hybrid era alone.")

## 6.2 Removing entries that never started (P-04 / D-03)
"Will this driver finish on the podium?" is not a real question for a car that never took the start, and at the prediction cutoff (after qualifying) it is already known. Two rules were proposed; the cross-tab shows why they are combined:

* **status list only** would miss cars that never started but carry another label (e.g. "Accident" in the warm-up lap, with grid = 0 and laps = 0);
* **grid = 0 only** would also drop genuine **pit-lane starters**, who did race (grid = 0 but laps > 0).

In [ ]:
xt = pd.crosstab(in_list.map({True: 'status = did not start', False: 'other status'}),
                 [base_all['grid'].eq(0).fillna(False).map({True: 'grid = 0', False: 'grid > 0'}),
                  base_all['laps'].eq(0).fillna(False).map({True: 'laps = 0', False: 'laps > 0'})])
display(xt)

base_all['pit_lane_start'] = base_all['grid'].eq(0).fillna(False).astype(bool) & ~base_all['is_non_starter']
removed_ns = base_all[base_all['is_non_starter']]
assert removed_ns['podium'].sum() == 0, 'a removed non-starter has a podium!'
only_rule2 = int((no_start & ~in_list).sum())
# What do the cars caught only by the grid/laps rule have as status? (crashes / injuries in practice, exclusions …)
display(base_all.loc[no_start & ~in_list, 'status'].value_counts().head(10).rename('rows caught only by grid = 0 & laps = 0').to_frame().T)
has_dns_label = 'Did not start' in set(clean['status']['status'])
grid0_raced = int(base_all['pit_lane_start'].sum())
log_decision('P-04 / D-03', '6.2', f'{len(removed_ns):,} entries never started (grid 0 podium rate 0%)',
             "Remove if status group = 'Did not start' OR (grid = 0 and laps = 0); flag pit-lane starters",
             'Not a real prediction; known at the cutoff',
             f'grid = 0 only (drops {grid0_raced} real pit-lane starters); status list only (misses {only_rule2})',
             f'{len(removed_ns):,} rows removed (0 podiums); {grid0_raced} pit-lane starters kept + flagged', 'B (C)')
say(f"**{len(removed_ns):,} non-starter rows removed over the full history, with 0 podiums among them** "
    f"({int(in_list.sum()):,} by status, {only_rule2} more by grid = 0 & laps = 0). "
    f"**{grid0_raced} genuine pit-lane starters are kept** and flagged `pit_lane_start` (their encoding is decision D-08 in Part 3). "
    + ("" if has_dns_label else "Note: the status table has **no 'Did not start' label**, which is exactly why the grid/laps rule is needed "
       "(and why document C's status-only count was lower than document B's)."))

## 6.3 Build the table and audit it for leakage (P-10)
Only **keys, columns known before the race starts, and the target** are kept. Everything that describes the race outcome is listed and excluded explicitly, and an assertion guards it.

In [ ]:
BASE_COLS = ['raceId', 'year', 'round', 'date', 'circuitId', 'driverId', 'constructorId',     # keys / calendar
             'grid', 'pit_lane_start', 'quali_position', 'q1_s', 'q2_s', 'q3_s',             # pre-race (after qualifying)
             'podium']                                                                       # target

# Every results column, classified against the prediction cutoff (after qualifying, before the start)
LEAKAGE_AUDIT = pd.DataFrame([
    ('resultId', 'id', 'row id, no information'), ('raceId', 'key', 'known'), ('driverId', 'key', 'known'),
    ('constructorId', 'key', 'known (entry list)'), ('number', 'pre-race', 'known but not informative; not kept'),
    ('grid', 'pre-race', 'starting slot incl. penalties — kept'),
    ('position', 'POST-RACE', 'finishing position'), ('positionText', 'POST-RACE', 'finishing code (R, D, …)'),
    ('positionOrder', 'TARGET SOURCE', 'defines podium; never a feature'), ('points', 'POST-RACE', 'scored in this race'),
    ('laps', 'POST-RACE', 'laps completed'), ('time', 'POST-RACE', 'race time / gap'), ('milliseconds', 'POST-RACE', 'race time'),
    ('fastestLap', 'POST-RACE', 'race data'), ('rank', 'POST-RACE', 'race data'), ('fastestLapTime_s', 'POST-RACE', 'race data'),
    ('fastestLapSpeed', 'POST-RACE', 'race data'), ('statusId', 'POST-RACE', 'outcome of the race'),
], columns=['results column', 'class', 'note'])
LEAKAGE_AUDIT['in base table'] = LEAKAGE_AUDIT['results column'].isin(BASE_COLS)
display(LEAKAGE_AUDIT)

post_race = set(LEAKAGE_AUDIT.loc[LEAKAGE_AUDIT['class'].isin(['POST-RACE', 'TARGET SOURCE']), 'results column'])
assert not (post_race & set(BASE_COLS)), 'post-race column in base table!'

base = (base_all[~base_all['is_non_starter'] & base_all['year'].between(2003, 2024)][BASE_COLS]
        .sort_values(['date', 'raceId', 'grid']).reset_index(drop=True))
log_decision('P-10', '6.3', 'positionOrder, points, laps, status, time, fastest-lap columns are post-race',
             'Base table keeps only keys, pre-race columns and the target', 'Prediction is made before the start',
             '—', f'{base.shape[1]} columns; 0 post-race columns (asserted)', 'B, C')
base.head()

**Other tables, same rule (hand-off to feature engineering):** `driver_standings` / `constructor_standings` / `constructor_results` → previous round only (P-08); `lap_times`, `pit_stops`, `sprint_results` of the current weekend → only past races (sprint happens before the race, but its use must be justified against the cutoff); qualifying times → relative to the same race (gap to pole).

Also note that `raceId` is **not in date order** (I-12, `raceId` 1 is the 2009 Australian GP), so the table is sorted by `date`, and every time-based feature or split must use `date` (or `year`, `round`), never `raceId`.

## 6.4 Validate the base table

In [ ]:
checks = {
    'one row per (raceId, driverId)': not base.duplicated(['raceId', 'driverId']).any(),
    'every season 2003–2024 present': set(base['year']) == set(range(2003, 2025)),
    'every race has exactly 3 podium rows': bool((base.groupby('raceId')['podium'].sum() == 3).all()),
    'no non-starters left': not base.set_index(['raceId', 'driverId']).index.isin(
        removed_ns.set_index(['raceId', 'driverId']).index).any(),
    'target is 0/1 only': set(base['podium'].unique()) <= {0, 1},
    'no post-race columns': not (post_race & set(base.columns)),
    'raceId not in date order (I-12) handled: sorted by date': bool(base['date'].is_monotonic_increasing),
}
display(pd.Series(checks, name='passed').to_frame())
assert all(checks.values())

per_season = base.groupby('year').agg(rows=('podium', 'size'), podium_rate=('podium', 'mean'))
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].bar(per_season.index, per_season['rows'], color=BLUE, width=0.75); ax[0].set_title('Base table rows per season')
ax[1].plot(per_season.index, per_season['podium_rate'], marker='o', ms=4, color=BLUE)
ax[1].set_ylim(0, 0.2); ax[1].yaxis.set_major_locator(mtick.MultipleLocator(0.05)); pct_axis(ax[1])
ax[1].set_title('Podium rate per season (= 3 / field size)')
plt.tight_layout(); plt.show()

missing_q = base['quali_position'].isna().sum()
say(f"✅ **Base table: {len(base):,} rows × {base.shape[1]} columns, {base['raceId'].nunique()} races, "
    f"{base['driverId'].nunique()} drivers, podium rate {base['podium'].mean():.1%}.** "
    f"Only **{missing_q}** rows lack a qualifying position (kept as `NaN`; handled by the imputation step in Part 3). "
    f"Pit-lane starters: {int(base['pit_lane_start'].sum())}.")

---
# 7. Save outputs
* `clean/` — the 14 cleaned, typed tables (Parquet keeps the types; CSV fallback if `pyarrow` is missing)
* `base_driver_race_2003_2024.parquet` (+ `.csv`) — the modeling base table
* `eda_summary_raw.csv`, `cleaning_before_after.csv`, `key_verification.csv`, `dedup_rule_comparison.csv`, `status_map.csv`, `leakage_audit.csv`, `decision_log.csv`

In [ ]:
def save_table(df, path_no_ext):
    # Parquet preserves dtypes (Int64, datetime, timedelta); fall back to CSV if no engine is installed.
    try:
        df.to_parquet(f'{path_no_ext}.parquet', index=False)
        return f'{path_no_ext}.parquet'
    except (ImportError, ValueError) as e:
        df.to_csv(f'{path_no_ext}.csv', index=False)
        return f'{path_no_ext}.csv'

saved = [save_table(clean[t], CLEAN_DIR / t) for t in TABLES]
saved.append(save_table(base, OUT_DIR / 'base_driver_race_2003_2024'))
base.to_csv(OUT_DIR / 'base_driver_race_2003_2024.csv', index=False)

eda_summary_raw.to_csv(OUT_DIR / 'eda_summary_raw.csv')
cleaning_before_after.to_csv(OUT_DIR / 'cleaning_before_after.csv')
key_verification.to_csv(OUT_DIR / 'key_verification.csv', index=False)
dedup_cmp.to_csv(OUT_DIR / 'dedup_rule_comparison.csv')
clean['status'].to_csv(OUT_DIR / 'status_map.csv', index=False)
LEAKAGE_AUDIT.to_csv(OUT_DIR / 'leakage_audit.csv', index=False)
print('\n'.join(str(s) for s in saved[-3:]), '\n… and', len(saved) - 3, 'more tables + 7 audit CSVs in', OUT_DIR)

---
# 8. Final checks and summary
## 8.1 Raw data were never modified (P-17)

In [ ]:
after_hash = {t: file_sha256(DATA_DIR / f'{t}.csv') for t in TABLES}
after_fp = {t: df_fingerprint(raw[t]) for t in TABLES}
immut = pd.DataFrame({'file hash unchanged': {t: after_hash[t] == RAW_FILE_HASH[t] for t in TABLES},
                      'raw DataFrame unchanged': {t: after_fp[t] == RAW_DF_FINGERPRINT[t] for t in TABLES}})
assert immut.values.all()
say(f"✅ **All 14 CSV files have the same SHA-256 hash as at the start, and all 14 raw DataFrames are unchanged.** "
    "Every modification lives in the copies (`stage1`, `typed`, `clean`, `base`).")

## 8.2 Decision log
Every decision taken in the notebook, with the evidence it was based on and the source documents that proposed it.

In [ ]:
decision_log = pd.DataFrame(DECISIONS)
decision_log.to_csv(OUT_DIR / 'decision_log.csv', index=False)
decision_log

In [ ]:
say(f'''
### Part 1 in numbers
| | Before | After |
|---|---|---|
| `\\N` / blank placeholders | {int(eda_summary_raw[N_COL].sum() + eda_summary_raw['n_blank'].sum()):,} | **0** (converted to `NaN`) |
| Typed columns | 0 (all text) | **{cleaning_before_after['typed_cols_clean'].sum()}** |
| Memory | {cleaning_before_after['memory_raw_mb'].sum():.0f} MB | **{cleaning_before_after['memory_clean_mb'].sum():.0f} MB** |
| `results` rows | {len(raw['results']):,} | **{len(clean['results']):,}** |
| Duplicate (raceId, driverId) pairs | {len(ref)} | **0** |
| Key checks passing | — | **{key_verification['passed'].sum()} / {len(key_verification)}** |
| Raw CSV files changed | — | **0 / 14** |
| Base table (2003–2024) | — | **{len(base):,} rows, {base['raceId'].nunique()} races, podium rate {base['podium'].mean():.1%}** |
''')

## 8.3 Hand-off to the next parts, and known limitations

**Decided in Part 1 (status: Agreed)** — D-01 keep-rule, D-02 period 2003–2024, D-03 non-starter rule, D-04 keep + flag, D-05 status map (document B).

**Handled further down in this notebook:** Q1–Q3 (Part 2, §10–12); P-08, P-11, D-06, D-08 and the imputation of the remaining missing values (Part 3, §13–16).

**Original list of open items**

| ID | Item | Where | Note |
|---|---|---|---|
| P-08 | Standings are post-race → use the **previous round** only | Part 3 (features) | evidence in §5.5 |
| P-11 | Raw points not comparable across seasons → use positions or share of leader's points | Part 3 | evidence in §4.7 |
| D-06 | Train / validation / test split **by season** (time-based) | Part 3 | podium prior changes over time (I-03) and `raceId` is not chronological (I-12) |
| D-07 | Primary metric **PR-AUC** (ROC-AUC, F1 secondary) | Part 4 | positives ≈ 14 % (I-02) |
| D-08 | Encoding of `grid = 0` for pit-lane starters (keep 0 + flag, or recode to last + 1) | Part 3 | candidate for the preprocessing-order experiment |
| Q1 | Front-row conversion by circuit; qualifying vs grid | Part 2 | both columns kept (I-06) |
| Q2 | Home advantage needs a `country` ↔ `nationality` mapping | Part 2 | vocabularies differ (§3.10) |
| Q3 | Mechanical retirements 2014–2021 vs 2022–2024 | Part 2 | use `status_group == 'Mechanical'`; report with/without `status_ambiguous` |

**Limitations to state in the report**
* Qualifying coverage drifts: none before 1994, patchy until 2002 → the model covers 2003–2024 only.
* Era sensitivity: field size, reliability, points systems and qualifying format all changed; results may not transfer between eras.
* Lap-time, pit-stop and sprint data cover only part of the history (sprints: 18 weekends).
* Some status labels are ambiguous (e.g. a broken wing may come from contact), so mechanical-retirement rates depend on the grouping.
* All relationships shown here are **associations, not causes**.

---
---
# PART 2 — Data-Engineering Questions

Each answer uses **multi-hop joins** across the cleaned tables, and comes with a visualization and a short note on **grain, missingness and the pre-race cutoff**. All three questions use **race starters only** (the non-starters removed in §6.2), because "converting a start", "podium at home" and "retiring" only make sense for a car that took the start.

## 9. Shared preparation
One row per (raceId, driverId) starter, enriched with circuit, driver and constructor attributes. Small samples are common when slicing by circuit or team, so every rate gets a **95 % Wilson confidence interval** (it stays valid for rates near 0 % or 100 % and for small n).

In [ ]:
# Starters over the full history, joined to circuits (country), drivers (nationality, dob) and constructors (name)
starts = (base_all[~base_all['is_non_starter']]
          .merge(clean['circuits'][['circuitId', 'name', 'country']].rename(columns={'name': 'circuit'}), on='circuitId', how='left')
          .merge(clean['drivers'][['driverId', 'forename', 'surname', 'nationality', 'dob']], on='driverId', how='left')
          .merge(clean['constructors'][['constructorId', 'name']].rename(columns={'name': 'constructor'}), on='constructorId', how='left')
          .merge(clean['status'][['statusId', 'status_ambiguous']], on='statusId', how='left'))
starts['driver'] = starts['forename'] + ' ' + starts['surname']
assert not starts.duplicated(['raceId', 'driverId']).any()            # grain: one row per driver start

def wilson(k, n, z=1.96):
    # 95 % Wilson score interval for a proportion k/n (vectorised).
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.divide(k, n, out=np.full_like(k, np.nan), where=n > 0)
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return centre - half, centre + half

say(f"`starts`: **{len(starts):,} driver starts** in {starts['raceId'].nunique():,} races ({int(starts['year'].min())}–{int(starts['year'].max())}), "
    f"joined from `results` → `races` → `circuits`, `drivers`, `constructors`, `status`.")

## 10. Question 1 — Which circuits convert a front-row start into a podium most often, and does the ranking change in the hybrid era (2014+)?

**Design**
* **Joins:** `results` → `races` (year, circuit) → `circuits` (name) and `qualifying` (qualifying position).
* **Grain:** one row per driver start. A *front-row start* is **grid ∈ {1, 2}**; separately, a *front-row qualifier* is **qualifying position ∈ {1, 2}**. They differ when grid penalties move a driver (§5.4), so both are reported.
* **Period:** 2003–2024, the seasons in which qualifying is complete (§3.5), split into **pre-hybrid 2003–2013** and **hybrid 2014–2024**, so the grid and qualifying versions are compared on identical seasons.
* **Minimum sample:** a circuit is ranked in an era only if it hosted **≥ 5 races** in that era (≈ 10 front-row starts); smaller samples are shown with their wide confidence intervals, not ranked.
* **Cutoff:** grid and qualifying position are both known before the start; the podium is the outcome. No leakage issue.

In [ ]:
Q1_MIN_RACES = 5
ERA = {'pre': '2003–2013 (pre-hybrid)', 'hybrid': '2014–2024 (hybrid)'}
q1 = starts[starts['year'].between(2003, 2024)].copy()
q1['era'] = np.where(q1['year'] >= 2014, 'hybrid', 'pre')
q1['front_grid'] = q1['grid'].isin([1, 2]).astype(bool)
q1['front_quali'] = q1['quali_position'].isin([1, 2]).astype(bool)

def conversion(df, flag):
    # Podium conversion of front-row starts, per circuit and era.
    g = (df[df[flag]].groupby(['circuit', 'era'])
         .agg(races=('raceId', 'nunique'), front_row_starts=('podium', 'size'), podiums=('podium', 'sum'))
         .reset_index())
    g['rate'] = g['podiums'] / g['front_row_starts']
    g['ci_low'], g['ci_high'] = wilson(g['podiums'], g['front_row_starts'])
    g['eligible'] = g['races'] >= Q1_MIN_RACES
    return g

conv_grid, conv_quali = conversion(q1, 'front_grid'), conversion(q1, 'front_quali')

def ranking(conv, era, k=10):
    t = (conv[(conv['era'] == era) & conv['eligible']]
         .sort_values(['rate', 'front_row_starts'], ascending=[False, False]).head(k))
    t = t.assign(rank=range(1, len(t) + 1)).set_index('rank')
    return t[['circuit', 'races', 'front_row_starts', 'podiums', 'rate', 'ci_low', 'ci_high']]

top_tables = pd.concat({ERA[e]: ranking(conv_grid, e) for e in ['pre', 'hybrid']}, axis=1)
display(top_tables.style.format({c: '{:.0%}' for c in top_tables.columns if c[1] in ('rate', 'ci_low', 'ci_high')})
        .set_caption('Top 10 circuits by front-row (grid 1–2) → podium conversion'))

overall = pd.DataFrame({
    'front row by grid → podium': q1[q1['front_grid']].groupby('era')['podium'].mean(),
    'front row by qualifying → podium': q1[q1['front_quali']].groupby('era')['podium'].mean()}).loc[['pre', 'hybrid']]
display(overall.rename(index=ERA).style.format('{:.1%}'))

In [ ]:
# Rank stability between eras: circuits eligible in both eras
wide = (conv_grid[conv_grid['eligible']].pivot(index='circuit', columns='era', values='rate').dropna())
rho = wide['pre'].corr(wide['hybrid'], method='spearman')
top5_pre, top5_hyb = set(wide['pre'].nlargest(5).index), set(wide['hybrid'].nlargest(5).index)

fig, ax = plt.subplots(1, 2, figsize=(15, max(5, 0.32 * len(wide))), gridspec_kw={'width_ratios': [1.35, 1]})
w = wide.sort_values('hybrid')
y = np.arange(len(w))
ax[0].hlines(y, w['pre'], w['hybrid'], color='#cfcfca', lw=2)
ax[0].scatter(w['pre'], y, color=GREY, s=36, label=ERA['pre'], zorder=3)
ax[0].scatter(w['hybrid'], y, color=BLUE, s=36, label=ERA['hybrid'], zorder=3)
ax[0].set_yticks(y, w.index); pct_axis(ax[0], 'x'); ax[0].grid(axis='y', visible=False)
ax[0].legend(frameon=False, loc='lower right')
ax[0].set_title(f'Front-row (grid) → podium conversion per circuit\n(circuits with ≥ {Q1_MIN_RACES} races in both eras)')

# Grid vs qualifying definition, per circuit (pooled 2003–2024, eligible circuits)
def pooled(c):
    g = c.groupby('circuit')[['podiums', 'front_row_starts', 'races']].sum()
    return g[g['races'] >= Q1_MIN_RACES * 2]
pg, pq = pooled(conv_grid), pooled(conv_quali)
both = (pg['podiums'] / pg['front_row_starts']).to_frame('grid').join((pq['podiums'] / pq['front_row_starts']).rename('quali'), how='inner')
ax[1].scatter(both['quali'], both['grid'], color=BLUE, s=30, alpha=0.8)
ax[1].plot([0.4, 1], [0.4, 1], color=GREY, ls='--', lw=1)
for c in (both['grid'] - both['quali']).abs().nlargest(4).index:
    ax[1].annotate(c.replace(' Circuit', '').replace('Autodromo ', ''), (both.loc[c, 'quali'], both.loc[c, 'grid']),
                   fontsize=8, xytext=(4, -10), textcoords='offset points', color='#333333')
pct_axis(ax[1]); pct_axis(ax[1], 'x')
ax[1].set_xlabel('conversion when front row = qualifying 1–2'); ax[1].set_ylabel('conversion when front row = grid 1–2')
ax[1].set_title('Grid vs qualifying definition, 2003–2024\n(dashed = identical)')
plt.tight_layout(); plt.show()

# How often does a front-row qualifier NOT start from the front row (penalties)?
moved = q1[q1['front_quali'] & ~q1['front_grid']]
say(f"**Rank stability:** across the **{len(wide)} circuits** ranked in both eras, the Spearman rank correlation between eras is "
    f"**ρ = {rho:.2f}**, and **{len(top5_pre & top5_hyb)} of the top 5** circuits are the same "
    f"(pre-hybrid top 5: {', '.join(wide['pre'].nlargest(5).index)}; hybrid top 5: {', '.join(wide['hybrid'].nlargest(5).index)}). "
    f"**Grid vs qualifying:** {len(moved)} front-row qualifiers did not start from the front row (penalties / pit-lane starts); "
    f"they still reached the podium {moved['podium'].mean():.0%} of the time, versus "
    f"{q1.loc[q1['front_grid'], 'podium'].mean():.0%} for front-row starters overall.")

In [ ]:
pre_top, hyb_top = ranking(conv_grid, 'pre', 3), ranking(conv_grid, 'hybrid', 3)
fmt = lambda t: '; '.join(f"{r.circuit} ({r.rate:.0%}, n = {r.front_row_starts})" for r in t.itertuples())
say(f'''
### ✅ Answer to Question 1
* **Most reliable conversion, pre-hybrid (2003–2013):** {fmt(pre_top)}.
* **Most reliable conversion, hybrid era (2014–2024):** {fmt(hyb_top)}.
* **Does the ranking change?** {"Yes — substantially" if rho < 0.4 else ("Partly" if rho < 0.7 else "Only a little")}: rank correlation ρ = {rho:.2f} and {len(top5_pre & top5_hyb)}/5 top circuits shared. Overall conversion rose from
  {overall.loc['pre', 'front row by grid → podium']:.0%} to {overall.loc['hybrid', 'front row by grid → podium']:.0%}, consistent with the dominance of one or two teams in the hybrid era.
* **Grid vs qualifying:** conversion from a front-row *qualifying* position is {overall.loc['hybrid', 'front row by qualifying → podium']:.0%} in the hybrid era vs {overall.loc['hybrid', 'front row by grid → podium']:.0%} from a front-row *grid* slot — penalties move some fast cars back but they often recover, so qualifying measures pace and grid measures track position; both are kept as features (Part 3).

**Grain:** one row per driver start (raceId, driverId), front-row rows only. **Missingness:** qualifying position is missing for {q1['quali_position'].isna().sum()} starts ({int((q1['front_grid'] & q1['quali_position'].isna()).sum())} of them on the front row by grid), so the grid and qualifying versions use almost the same rows. **Cutoff:** both inputs are known before the start.
**Caveat:** ~{int(conv_grid.loc[conv_grid['eligible'], 'front_row_starts'].median())} front-row starts per circuit and era → confidence intervals are wide (see table); differences of a few points between circuits are not meaningful. These are associations, not causal effects of the circuit.
''')

## 11. Question 2 — Do drivers podium more often at home, after controlling for starting position?

**Design**
* **Joins:** `results` → `races` → `circuits` (`country`) and `results` → `drivers` (`nationality`).
* **Home race:** the circuit's country is the driver's country. `country` uses place names and `nationality` uses demonyms (§3.10), so an explicit, hand-written **demonym → country map** is used (dual nationalities such as "American-Italian" count both countries; nationalities whose country never hosted a championship race have no home race).
* **Controlling for starting position:** comparisons are made **within grid bands** (1–3, 4–6, 7–10, 11–15, 16+), then pooled with the **Cochran–Mantel–Haenszel** odds ratio (a weighted average of the within-band odds ratios). A logistic regression `podium ~ home + grid band + decade` is the robustness check.
* **Population:** starters from a grid slot (grid ≥ 1), **restricted to drivers who have at least one home race**, so the home and away groups contain the same drivers (otherwise "away" would be dominated by nationalities that never race at home).
* **Cutoff:** nationality, circuit and grid are all known before the start.

In [ ]:
NAT_TO_COUNTRY = {   # driver demonym  ->  circuit 'country' value(s)
    'American': ['USA', 'United States'], 'American-Italian': ['USA', 'United States', 'Italy'],
    'Argentine': ['Argentina'], 'Argentinian': ['Argentina'], 'Argentine-Italian': ['Argentina', 'Italy'],
    'Australian': ['Australia'], 'Austrian': ['Austria'], 'Belgian': ['Belgium'], 'Brazilian': ['Brazil'],
    'British': ['UK'], 'Canadian': ['Canada'], 'Chinese': ['China'], 'Dutch': ['Netherlands'], 'French': ['France'],
    'German': ['Germany'], 'East German': ['Germany'], 'Hungarian': ['Hungary'], 'Indian': ['India'], 'Italian': ['Italy'],
    'Japanese': ['Japan'], 'Malaysian': ['Malaysia'], 'Mexican': ['Mexico'], 'Monegasque': ['Monaco'],
    'Portuguese': ['Portugal'], 'Russian': ['Russia'], 'South African': ['South Africa'], 'Spanish': ['Spain'],
    'Swedish': ['Sweden'], 'Swiss': ['Switzerland'],
    # countries that never hosted a championship race in this dataset -> no home race
    'Chilean': [], 'Colombian': [], 'Czech': [], 'Danish': [], 'Finnish': [], 'Indonesian': [], 'Irish': [],
    'Liechtensteiner': [], 'New Zealander': [], 'Polish': [], 'Rhodesian': [], 'Thai': [], 'Uruguayan': [], 'Venezuelan': [],
}
unknown_nat = sorted(set(starts['nationality'].dropna()) - set(NAT_TO_COUNTRY))
if unknown_nat:
    print('⚠️ nationalities without a mapping (treated as having no home race):', unknown_nat)
missing_country = sorted({c for v in NAT_TO_COUNTRY.values() for c in v} - set(clean['circuits']['country']))
print('Mapped countries not present in circuits (harmless spelling variants):', missing_country)

starts['home'] = [c in NAT_TO_COUNTRY.get(n, []) for n, c in zip(starts['nationality'], starts['country'])]
GRID_BANDS = ([0, 3, 6, 10, 15, 99], ['1–3', '4–6', '7–10', '11–15', '16+'])
q2_all = starts[starts['grid'] >= 1].copy()
q2_all['grid_band'] = pd.cut(q2_all['grid'].astype(float), GRID_BANDS[0], labels=GRID_BANDS[1])
q2_all['decade'] = (q2_all['year'] // 10 * 10).astype(int)
q2_all['home'] = q2_all['home'].astype(bool)
home_drivers = set(q2_all.loc[q2_all['home'], 'driverId'])
q2 = q2_all[q2_all['driverId'].isin(home_drivers)].copy()

band = (q2.groupby(['grid_band', 'home'], observed=True)['podium'].agg(['sum', 'size']).unstack('home'))
band.columns = [f'{a}_{"home" if b else "away"}' for a, b in band.columns]
band['home_rate'] = band['sum_home'] / band['size_home']; band['away_rate'] = band['sum_away'] / band['size_away']
band['difference (pp)'] = (band['home_rate'] - band['away_rate']) * 100
display(band[['size_home', 'home_rate', 'size_away', 'away_rate', 'difference (pp)']]
        .rename(columns={'size_home': 'home starts', 'size_away': 'away starts'})
        .style.format({'home_rate': '{:.1%}', 'away_rate': '{:.1%}', 'difference (pp)': '{:+.1f}'}))
say(f"{len(home_drivers)} drivers have at least one home race; they account for **{len(q2):,} starts**, "
    f"**{int(q2['home'].sum()):,} at home** ({q2['home'].mean():.1%}). Raw podium rate: home **{q2.loc[q2['home'], 'podium'].mean():.1%}** "
    f"vs away **{q2.loc[~q2['home'], 'podium'].mean():.1%}** — but home drivers may simply start further forward, hence the grid bands.")

In [ ]:
from statsmodels.stats.contingency_tables import StratifiedTable
import statsmodels.formula.api as smf

def cmh(df):
    # Cochran–Mantel–Haenszel pooled odds ratio (home vs away) across grid bands.
    tabs = []
    for _, d in df.groupby('grid_band', observed=True):
        h, p = d['home'].values, d['podium'].values == 1
        t = np.array([[(h & p).sum(), (h & ~p).sum()], [(~h & p).sum(), (~h & ~p).sum()]], dtype=float)
        if (t.sum(axis=1) > 0).all():
            tabs.append(t)
    st_ = StratifiedTable(tabs)
    lo, hi = st_.oddsratio_pooled_confint()
    return {'pooled OR': st_.oddsratio_pooled, 'CI low': lo, 'CI high': hi,
            'p (OR = 1)': st_.test_null_odds().pvalue, 'p (same OR in every band)': st_.test_equal_odds().pvalue}

def logit_home(df, extra=''):
    m = smf.logit(f'podium ~ home + C(grid_band){extra}', data=df.assign(home=df['home'].astype(int),
                  podium=df['podium'].astype(int), grid_band=df['grid_band'].astype(str))).fit(disp=0)
    lo, hi = np.exp(m.conf_int().loc['home'])
    return {'pooled OR': np.exp(m.params['home']), 'CI low': lo, 'CI high': hi, 'p (OR = 1)': m.pvalues['home'],
            'p (same OR in every band)': np.nan}

q2_modern = q2[q2['year'].between(2003, 2024)]
q2_results = pd.DataFrame({
    'CMH, 1950–2024 (primary)': cmh(q2),
    'Logit + decade, 1950–2024': logit_home(q2, ' + C(decade)'),
    'CMH, 2003–2024': cmh(q2_modern),
    'CMH, all drivers incl. no home race': cmh(q2_all),
}).T
display(q2_results.style.format({'pooled OR': '{:.2f}', 'CI low': '{:.2f}', 'CI high': '{:.2f}',
                                 'p (OR = 1)': '{:.3f}', 'p (same OR in every band)': '{:.3f}'}))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.4))
x = np.arange(len(band)); w = 0.38
for i, (side, col) in enumerate([('home', ORANGE), ('away', BLUE)]):
    rate = band[f'{side}_rate']; lo, hi = wilson(band[f'sum_{side}'], band[f'size_{side}'])
    ax[0].bar(x + (i - 0.5) * w, rate, w, color=col, label=side)
    ax[0].errorbar(x + (i - 0.5) * w, rate, yerr=[rate - lo, hi - rate], fmt='none', ecolor='#444444', lw=1, capsize=3)
ax[0].set_xticks(x, band.index); ax[0].set_xlabel('grid band'); pct_axis(ax[0]); ax[0].legend(frameon=False)
ax[0].set_title('Podium rate at home vs away, within grid bands (95 % CI)')

# Per-band odds ratios (Woolf CI with 0.5 continuity correction) and the pooled CMH estimate
a, b = band['sum_home'] + .5, band['size_home'] - band['sum_home'] + .5
c, d = band['sum_away'] + .5, band['size_away'] - band['sum_away'] + .5
or_b = (a * d) / (b * c); se = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
ax[1].errorbar(or_b, x, xerr=[or_b - np.exp(np.log(or_b) - 1.96 * se), np.exp(np.log(or_b) + 1.96 * se) - or_b],
               fmt='o', color=BLUE, ecolor=BLUE, capsize=3)
prim = q2_results.loc['CMH, 1950–2024 (primary)']
ax[1].axvspan(prim['CI low'], prim['CI high'], color=ORANGE, alpha=0.15, label='pooled CMH 95 % CI')
ax[1].axvline(prim['pooled OR'], color=ORANGE, lw=2, label=f"pooled OR = {prim['pooled OR']:.2f}")
ax[1].axvline(1, color=GREY, ls='--', lw=1)
ax[1].set_xscale('log'); ax[1].set_yticks(x, band.index); ax[1].invert_yaxis(); ax[1].set_ylabel('grid band')
ax[1].xaxis.set_minor_locator(mtick.NullLocator()); ax[1].set_xticks([0.5, 0.75, 1, 1.5, 2])
ax[1].xaxis.set_major_formatter(mtick.FormatStrFormatter('%.2g'))
ax[1].set_title('Home vs away odds ratio by grid band (1 = no effect)'); ax[1].legend(frameon=False, loc='upper right')
plt.tight_layout(); plt.show()

sig = prim['CI low'] > 1 or prim['CI high'] < 1
mod = q2_results.loc['CMH, 2003–2024']
say(f'''
### ✅ Answer to Question 2
* **Controlling for starting position, the pooled home-vs-away odds ratio is {prim['pooled OR']:.2f} (95 % CI {prim['CI low']:.2f}–{prim['CI high']:.2f}, p = {prim['p (OR = 1)']:.3f}).**
  {("This is a statistically significant " + ("home advantage" if prim['pooled OR'] > 1 else "home *disadvantage*") + ".") if sig else "The interval includes 1, so there is **no statistically significant home advantage** once starting position is accounted for."}
  Raw rates were {q2.loc[q2['home'], 'podium'].mean():.1%} at home vs {q2.loc[~q2['home'], 'podium'].mean():.1%} away; within grid bands the differences shrink to {band['difference (pp)'].min():+.1f} to {band['difference (pp)'].max():+.1f} percentage points, so the raw gap was mostly about *where* drivers started.
* **Robustness:** logistic regression with decade controls gives OR {q2_results.loc['Logit + decade, 1950–2024', 'pooled OR']:.2f} ({q2_results.loc['Logit + decade, 1950–2024', 'CI low']:.2f}–{q2_results.loc['Logit + decade, 1950–2024', 'CI high']:.2f}); the 2003–2024 modeling era gives {mod['pooled OR']:.2f} ({mod['CI low']:.2f}–{mod['CI high']:.2f}). Homogeneity test p = {prim['p (same OR in every band)']:.2f} → {"no evidence that the effect differs between grid bands" if prim['p (same OR in every band)'] > 0.05 else "the effect differs between grid bands"}.
* **Consequence for modeling:** `home_race` is kept as a candidate feature, but the evidence predicts a small contribution; the feature-selection evidence (§15) and the ablation (Part 5) decide.

**Grain:** one row per driver start from a grid slot ({len(q2):,} starts, {len(home_drivers)} drivers). **Missingness:** none in nationality or circuit country; pit-lane starters (grid 0) are excluded because they have no grid band. **Cutoff:** all inputs known before the start.
**Caveats:** "home" uses the circuit's country, so e.g. the San Marino GP (Imola) counts as home for Italians; nationality is the one recorded in the dataset. Association, not a causal effect of racing at home.
''')

## 12. Question 3 — Which constructors have the highest mechanical-retirement rate in 2014–2021 vs 2022–2024?

**Design**
* **Joins:** `results` → `status` (status group from §4.5) → `races` (year) → `constructors` (name).
* **Definition (milestone):** a mechanical retirement is a **race start that ended because a car component failed**, excluding crashes, driver causes and regulation decisions. With document B's status map this is `status_group == 'Mechanical'`; accidents, driver health, regulatory and team/operational errors (e.g. a loose wheel nut) are excluded.
* **Rate:** mechanical retirements ÷ **race starts** (non-starters removed), per constructor and period. Constructors need **≥ 40 starts** in a period (≈ one full season with two cars) to be ranked.
* **Sensitivity:** *strict* = mechanical labels not flagged ambiguous (drops e.g. "Technical", wings, "Stalled"); *broad* = mechanical + tyre/wheel failures. If the ranking holds under both, the answer does not depend on the ambiguous labels.
* **Team identity:** the data treats re-branded teams as different constructors (e.g. Toro Rosso → AlphaTauri → RB). The ranking uses the constructors as recorded; a **lineage view** links re-brands so a team can be followed across the two periods.

In [ ]:
Q3_MIN_STARTS = 40
q3 = starts[starts['year'].between(2014, 2024)].copy()
q3['period'] = np.where(q3['year'] >= 2022, '2022–2024', '2014–2021')
q3['mech'] = q3['status_group'].eq('Mechanical')
q3['mech_strict'] = q3['mech'] & ~q3['status_ambiguous'].astype(bool)
q3['mech_broad'] = q3['mech'] | q3['status_group'].eq('Tyre/wheel')
q3['unknown_cause'] = q3['status_group'].eq('Other/unknown')

def mech_rates(df, key):
    g = df.groupby([key, 'period']).agg(starts=('mech', 'size'), mechanical=('mech', 'sum'),
                                       strict=('mech_strict', 'sum'), broad=('mech_broad', 'sum')).reset_index()
    for c in ['mechanical', 'strict', 'broad']:
        g[f'{c}_rate'] = g[c] / g['starts']
    g['ci_low'], g['ci_high'] = wilson(g['mechanical'], g['starts'])
    return g[g['starts'] >= Q3_MIN_STARTS]

mr = mech_rates(q3, 'constructor')
for p in ['2014–2021', '2022–2024']:
    t = mr[mr['period'] == p].sort_values('mechanical_rate', ascending=False)
    display(t.set_index('constructor')[['starts', 'mechanical', 'mechanical_rate', 'ci_low', 'ci_high', 'strict_rate', 'broad_rate']]
            .style.format({c: '{:.1%}' for c in ['mechanical_rate', 'ci_low', 'ci_high', 'strict_rate', 'broad_rate']})
            .set_caption(f'Mechanical-retirement rate per start, {p}'))

per = q3.groupby('period').agg(mech=('mech', 'mean'), unknown=('unknown_cause', 'mean'))
display(per.rename(columns={'mech': 'mechanical share of starts', 'unknown': "generic 'Retired' / unknown cause"}).style.format('{:.1%}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharex=True)
for ax, p in zip(axes, ['2014–2021', '2022–2024']):
    t = mr[mr['period'] == p].sort_values('mechanical_rate')
    y = np.arange(len(t))
    ax.barh(y, t['mechanical_rate'], color=BLUE, height=0.6)
    ax.errorbar(t['mechanical_rate'], y, xerr=[t['mechanical_rate'] - t['ci_low'], t['ci_high'] - t['mechanical_rate']],
                fmt='none', ecolor='#444444', lw=1, capsize=2)
    ax.set_yticks(y, t['constructor']); ax.grid(axis='y', visible=False); pct_axis(ax, 'x')
    ax.set_title(f'Mechanical retirements per start, {p}')
plt.tight_layout(); plt.show()

# Lineage view: follow re-branded teams across the two periods
LINEAGE = {'Toro Rosso': 'Toro Rosso / AlphaTauri / RB', 'AlphaTauri': 'Toro Rosso / AlphaTauri / RB', 'RB F1 Team': 'Toro Rosso / AlphaTauri / RB',
           'Force India': 'Force India / Racing Point / Aston Martin', 'Racing Point': 'Force India / Racing Point / Aston Martin',
           'Aston Martin': 'Force India / Racing Point / Aston Martin', 'Sauber': 'Sauber / Alfa Romeo', 'Alfa Romeo': 'Sauber / Alfa Romeo',
           'Lotus F1': 'Lotus F1 / Renault / Alpine', 'Renault': 'Lotus F1 / Renault / Alpine', 'Alpine F1 Team': 'Lotus F1 / Renault / Alpine',
           'Marussia': 'Marussia / Manor', 'Manor Marussia': 'Marussia / Manor'}
q3['lineage'] = q3['constructor'].map(lambda n: LINEAGE.get(n, n))
lin = mech_rates(q3, 'lineage').pivot(index='lineage', columns='period', values='mechanical_rate').dropna()
rho3 = lin['2014–2021'].corr(lin['2022–2024'], method='spearman')

fig, ax = plt.subplots(figsize=(11, 0.45 * len(lin) + 1.5))
l = lin.sort_values('2022–2024'); y = np.arange(len(l))
ax.hlines(y, l['2014–2021'], l['2022–2024'], color='#cfcfca', lw=2)
ax.scatter(l['2014–2021'], y, color=GREY, s=40, label='2014–2021', zorder=3)
ax.scatter(l['2022–2024'], y, color=BLUE, s=40, label='2022–2024', zorder=3)
ax.set_yticks(y, l.index); pct_axis(ax, 'x'); ax.grid(axis='y', visible=False); ax.legend(frameon=False, loc='lower right')
ax.set_title('Mechanical-retirement rate by team lineage, 2014–2021 vs 2022–2024')
plt.tight_layout(); plt.show()

# Sensitivity: does the ranking survive the strict / broad definitions?
sens = {p: mr[mr['period'] == p][['mechanical_rate', 'strict_rate', 'broad_rate']].corr(method='spearman').loc['mechanical_rate']
        for p in ['2014–2021', '2022–2024']}
display(pd.DataFrame(sens).T.drop(columns='mechanical_rate').rename(columns=lambda c: f'rank corr. with {c}').style.format('{:.2f}'))

In [ ]:
top = {p: mr[mr['period'] == p].nlargest(3, 'mechanical_rate') for p in ['2014–2021', '2022–2024']}
fmt3 = lambda t: '; '.join(f"{r.constructor} ({r.mechanical_rate:.1%}, {r.mechanical}/{r.starts})" for r in t.itertuples())
low = {p: mr[mr['period'] == p].nsmallest(1, 'mechanical_rate').iloc[0] for p in top}
say(f'''
### ✅ Answer to Question 3
* **2014–2021 — highest mechanical-retirement rates:** {fmt3(top['2014–2021'])}. Lowest: {low['2014–2021']['constructor']} ({low['2014–2021']['mechanical_rate']:.1%}).
* **2022–2024 — highest:** {fmt3(top['2022–2024'])}. Lowest: {low['2022–2024']['constructor']} ({low['2022–2024']['mechanical_rate']:.1%}).
* **Change between periods:** the overall mechanical share of starts went from {per.loc['2014–2021', 'mech']:.1%} to {per.loc['2022–2024', 'mech']:.1%}. Following team lineages, the rank correlation between the periods is ρ = {rho3:.2f} — {"reliability rankings were largely reshuffled by the 2022 regulations" if rho3 < 0.4 else "teams kept roughly their relative reliability"}.
* **Sensitivity:** ranking with the strict / broad definitions correlates ≥ {min(min(v.drop('mechanical_rate')) for v in sens.values()):.2f} with the main definition, so the ambiguous labels do not drive the answer.

**Grain:** one row per race start (raceId, driverId) of 2014–2024, aggregated per constructor and period. **Missingness:** every start has a status; {per['unknown'].max():.1%} of starts in a period carry a generic "Retired"/unknown cause that may hide some mechanical failures (a limitation, listed below). **Cutoff:** this is a descriptive question about outcomes, not a feature; if team reliability is used for prediction, it must be computed from **past races only** — that is exactly the `con_dnf_rate_last10` feature in Part 3.
**Caveats:** 2022–2024 has only 3 seasons (≈ 136 starts per team) → wide intervals; re-branded teams are separate constructors in the data (see lineage view).
''')

---
---
# PART 3 — Feature Selection & Engineering

The prediction is made **after qualifying and before the race starts**, for the 2003–2024 base table (§6). Every feature must be computable from information available at that moment:

| Group | Features | Source | Why it is pre-race |
|---|---|---|---|
| **grid** | `grid`, `pit_lane_start` | results | starting slot is fixed after qualifying + penalties |
| **qualifying** | `quali_position`, `gap_to_pole_pct`, `quali_top10`, `no_quali_time`, `grid_penalty` | qualifying | qualifying is over at the cutoff |
| **driver form** | `drv_podium_rate_last5`, `drv_finish_pct_last5`, `drv_dnf_rate_last10`, `drv_starts_before` | results of **previous** races | strictly earlier races (shifted by one start) |
| **constructor form** | `con_podiums_per_race_last5`, `con_best_finish_pct_last5`, `con_dnf_rate_last10` | results of **previous** races | strictly earlier races |
| **standings** | `drv_champ_pos_prev`, `drv_points_share_prev`, `con_champ_pos_prev`, `con_points_share_prev`, `is_first_round` | standings after the **previous** Grand Prix (P-08) | the current race is not yet in the table |
| **context** | `field_size`, `home_race`, `driver_age`, `drv_circuit_podium_rate_prev` | races, drivers, circuits, past results | static or past information |

Design choices answered in this part: points are expressed as a **share of the leader's points** (P-11, comparable across points systems); qualifying pace as a **% gap to pole** (raw seconds depend on the circuit, §5.4); standings are taken **after the previous Grand Prix in calendar order** — for round 1 that is the previous season's final table, so the feature is never empty for returning drivers.

## 13. Leakage-safe feature construction
### 13.1 Chronological race order
`raceId` is **not** chronological (I-12), so every "previous race" is defined by **race date**.

In [ ]:
race_order = (clean['races'][['raceId', 'year', 'round', 'date', 'circuitId']]
              .sort_values(['date', 'raceId']).reset_index(drop=True))
race_order['race_seq'] = np.arange(len(race_order))                           # 0, 1, 2 … in calendar order
race_order['prev_raceId'] = race_order['raceId'].shift(1).astype('Int64')    # the Grand Prix just before
assert race_order['date'].is_monotonic_increasing

# History of every start since 1950 (form features for 2003 need the seasons before 2003)
hist = starts[['raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'positionOrder', 'podium', 'status_group']] \
         .merge(race_order[['raceId', 'race_seq']], on='raceId')
hist['field_size'] = hist.groupby('raceId')['driverId'].transform('size')
# Normalised finish: 0 = win, 1 = last starter (makes 20- and 39-car fields comparable)
hist['finish_pct'] = ((hist['positionOrder'].astype(float) - 1) / (hist['field_size'] - 1)).clip(0, 1)
hist['dnf'] = (~hist['status_group'].isin(['Finished', 'Lapped'])).astype(float)
hist['podium'] = hist['podium'].astype(float)
print(f'{len(hist):,} historical starts used to build form features')

### 13.2 Driver and constructor form (only past races)
`shift(1)` moves every value one start into the future before the rolling window is applied, so a row can only see races that happened **before** it. A driver's first start has no form (`NaN`), handled in preprocessing (§16).

In [ ]:
def past_rolling(df, by, col, window):
    # Mean of `col` over the previous `window` rows of each group (current row excluded).
    return df.groupby(by)[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

# --- driver form ---
hist = hist.sort_values(['driverId', 'race_seq'])
hist['drv_podium_rate_last5'] = past_rolling(hist, 'driverId', 'podium', 5)
hist['drv_finish_pct_last5'] = past_rolling(hist, 'driverId', 'finish_pct', 5)
hist['drv_dnf_rate_last10'] = past_rolling(hist, 'driverId', 'dnf', 10)
hist['drv_starts_before'] = hist.groupby('driverId').cumcount()

# --- driver history at this circuit (previous visits only) ---
hist = hist.sort_values(['driverId', 'circuitId', 'race_seq'])
hist['drv_circuit_podium_rate_prev'] = hist.groupby(['driverId', 'circuitId'])['podium'].transform(
    lambda s: s.shift(1).expanding().mean())

# --- constructor form: aggregate both cars per race, then roll over the team's previous races ---
team = (hist.groupby(['constructorId', 'raceId', 'race_seq'])
        .agg(team_podiums=('podium', 'sum'), team_best_finish=('finish_pct', 'min'), team_dnf=('dnf', 'mean'))
        .reset_index().sort_values(['constructorId', 'race_seq']))
team['con_podiums_per_race_last5'] = past_rolling(team, 'constructorId', 'team_podiums', 5)
team['con_best_finish_pct_last5'] = past_rolling(team, 'constructorId', 'team_best_finish', 5)
team['con_dnf_rate_last10'] = past_rolling(team, 'constructorId', 'team_dnf', 10)

### 13.3 Standings after the previous Grand Prix (P-08, P-11)

In [ ]:
ds_ = clean['driver_standings']
cs_ = clean['constructor_standings']
drv_prev = ds_.assign(leader=ds_.groupby('raceId')['points'].transform('max'))
drv_prev = drv_prev.assign(drv_points_share_prev=np.where(drv_prev['leader'] > 0, drv_prev['points'] / drv_prev['leader'], 0.0)) \
    .rename(columns={'raceId': 'prev_raceId', 'position': 'drv_champ_pos_prev'}) \
    [['prev_raceId', 'driverId', 'drv_champ_pos_prev', 'drv_points_share_prev']]
con_prev = cs_.assign(leader=cs_.groupby('raceId')['points'].transform('max'))
con_prev = con_prev.assign(con_points_share_prev=np.where(con_prev['leader'] > 0, con_prev['points'] / con_prev['leader'], 0.0)) \
    .rename(columns={'raceId': 'prev_raceId', 'position': 'con_champ_pos_prev'}) \
    [['prev_raceId', 'constructorId', 'con_champ_pos_prev', 'con_points_share_prev']]
# Size of each standings table (used by the domain fill: a driver absent from the table is ranked behind everyone in it)
table_sizes = (ds_.groupby('raceId')['driverId'].size().rename('_drv_table_size_prev').to_frame()
               .join(cs_.groupby('raceId')['constructorId'].size().rename('_con_table_size_prev'), how='outer')
               .rename_axis('prev_raceId').reset_index())

### 13.4 Assemble the feature table (one row per driver–race, 2003–2024)

In [ ]:
ft = (base.merge(race_order[['raceId', 'race_seq', 'prev_raceId']], on='raceId', how='left')
      .merge(hist[['raceId', 'driverId', 'drv_podium_rate_last5', 'drv_finish_pct_last5', 'drv_dnf_rate_last10',
                   'drv_starts_before', 'drv_circuit_podium_rate_prev', 'field_size']], on=['raceId', 'driverId'], how='left')
      .merge(team[['raceId', 'constructorId', 'con_podiums_per_race_last5', 'con_best_finish_pct_last5', 'con_dnf_rate_last10']],
             on=['raceId', 'constructorId'], how='left')
      .merge(drv_prev, on=['prev_raceId', 'driverId'], how='left')
      .merge(con_prev, on=['prev_raceId', 'constructorId'], how='left')
      .merge(table_sizes, on='prev_raceId', how='left')
      .merge(starts[['raceId', 'driverId', 'home', 'dob']], on=['raceId', 'driverId'], how='left'))

# --- qualifying features ---
qcols = ['q1_s', 'q2_s', 'q3_s']
ft = ft.merge(clean['qualifying'][['raceId', 'driverId', 'q_time_suspect']], on=['raceId', 'driverId'], how='left')
qt = ft[qcols].where(~ft['q_time_suspect'].fillna(False).astype(bool))     # ignore the impossible times flagged in §4.6
ft['best_quali_s'] = qt.min(axis=1)
ft['gap_to_pole_pct'] = (ft['best_quali_s'] / ft.groupby('raceId')['best_quali_s'].transform('min') - 1) * 100
ft['_race_max_gap'] = ft.groupby('raceId')['gap_to_pole_pct'].transform('max')
ft['no_quali_time'] = ft['best_quali_s'].isna()
ft['quali_top10'] = ft['quali_position'].le(10).fillna(False).astype(bool)
ft['grid_penalty'] = (ft['grid'].where(ft['grid'] > 0, ft['field_size']) - ft['quali_position']).astype(float)

# --- context ---
ft['is_first_round'] = ft['round'].eq(1)
ft['home_race'] = ft['home'].astype(bool)
ft['driver_age'] = (ft['date'] - ft['dob']).dt.days / 365.25

FEATURE_GROUPS = {
    'grid': ['grid', 'pit_lane_start'],
    'qualifying': ['quali_position', 'gap_to_pole_pct', 'quali_top10', 'no_quali_time', 'grid_penalty'],
    'driver_form': ['drv_podium_rate_last5', 'drv_finish_pct_last5', 'drv_dnf_rate_last10', 'drv_starts_before'],
    'constructor_form': ['con_podiums_per_race_last5', 'con_best_finish_pct_last5', 'con_dnf_rate_last10'],
    'standings': ['drv_champ_pos_prev', 'drv_points_share_prev', 'con_champ_pos_prev', 'con_points_share_prev', 'is_first_round'],
    'context': ['field_size', 'home_race', 'driver_age', 'drv_circuit_podium_rate_prev'],
}
CANDIDATES = [f for g in FEATURE_GROUPS.values() for f in g]
HELPERS = ['_drv_table_size_prev', '_con_table_size_prev', '_race_max_gap', 'q1_s', 'q2_s', 'q3_s', 'best_quali_s']
assert len(ft) == len(base) and not ft.duplicated(['raceId', 'driverId']).any()

miss = ft[CANDIDATES].isna().mean()
display(pd.DataFrame({'group': {f: g for g, fs in FEATURE_GROUPS.items() for f in fs}, 'missing': miss})
        .style.format({'missing': '{:.1%}'}))
say(f"**Feature table: {len(ft):,} rows × {len(CANDIDATES)} candidate features in {len(FEATURE_GROUPS)} groups.** "
    f"Missing values come from first starts (no form yet), first visits to a circuit, drivers absent from the previous standings table, "
    f"and the {int(ft['no_quali_time'].sum())} rows without a qualifying time; they are handled in §16.")

### 13.5 Leakage tests
Two independent checks that the form and standings features only use the past:
1. **Recompute from scratch:** for 300 random rows, recompute the driver's podium rate over his last 5 starts **using only races with an earlier date**, and the constructor's podiums over its last 5 races, then compare with the vectorised features.
2. **Standings lag:** the standings used for every row come from a race whose date is strictly earlier.

In [ ]:
seq_of = race_order.set_index('raceId')['race_seq']
sample = ft.sample(300, random_state=42)
bad_drv = bad_con = 0
for r in sample.itertuples():
    past = hist[(hist['driverId'] == r.driverId) & (hist['race_seq'] < r.race_seq)].sort_values('race_seq').tail(5)
    exp_d = past['podium'].mean() if len(past) else np.nan
    bad_drv += not (np.isclose(exp_d, r.drv_podium_rate_last5) or (np.isnan(exp_d) and np.isnan(r.drv_podium_rate_last5)))
    tp = team[(team['constructorId'] == r.constructorId) & (team['race_seq'] < r.race_seq)].sort_values('race_seq').tail(5)
    exp_c = tp['team_podiums'].mean() if len(tp) else np.nan
    bad_con += not (np.isclose(exp_c, r.con_podiums_per_race_last5) or (np.isnan(exp_c) and np.isnan(r.con_podiums_per_race_last5)))
assert bad_drv == 0 and bad_con == 0, (bad_drv, bad_con)

prev_date = ft['prev_raceId'].map(race_order.set_index('raceId')['date'])
assert (prev_date < ft['date']).all(), 'standings taken from a race that is not earlier!'
say(f"✅ **Leakage tests passed:** 300/300 recomputed driver-form and constructor-form values match, and for all {len(ft):,} rows the "
    "standings come from a strictly earlier Grand Prix.")

## 14. Train / validation / test split (D-06)
The split is fixed **before** looking at feature evidence, so that all evidence in §15 is computed on the **training** seasons only.

**Split by season, in time order:** train **2003–2018**, validation **2019–2021**, test **2022–2024**.
* A random split would put races from the same weekend — and from the future — into training, which inflates scores (I-12: the prediction must only use the past).
* The podium base rate changes with field size and era (I-03), so validation/test must be *later* seasons, like a real deployment.
* The test set is the newest regulation era (2022+ ground-effect cars): the hardest and most honest test. Validation (3 seasons) is used for every choice — thresholds, hyper-parameters, preprocessing — and the test set is touched only once, in Part 4.

In [ ]:
SPLITS = {'train': (2003, 2018), 'validation': (2019, 2021), 'test': (2022, 2024)}
ft['split'] = 'train'
for name, (a, b) in SPLITS.items():
    ft.loc[ft['year'].between(a, b), 'split'] = name
split_tbl = ft.groupby('split').agg(seasons=('year', lambda s: f'{s.min()}–{s.max()}'), rows=('podium', 'size'),
                                    races=('raceId', 'nunique'), podium_rate=('podium', 'mean')).loc[list(SPLITS)]
split_tbl['share of rows'] = split_tbl['rows'] / split_tbl['rows'].sum()
display(split_tbl.style.format({'podium_rate': '{:.1%}', 'share of rows': '{:.0%}', 'rows': '{:,}'}))
assert ft.groupby('raceId')['split'].nunique().max() == 1                  # no race in two splits
assert ft.loc[ft['split'] == 'train', 'date'].max() < ft.loc[ft['split'] == 'validation', 'date'].min()
assert ft.loc[ft['split'] == 'validation', 'date'].max() < ft.loc[ft['split'] == 'test', 'date'].min()

fig, ax = plt.subplots(figsize=(11, 3.4))
rows_y = ft.groupby(['year', 'split']).size().unstack(fill_value=0)[list(SPLITS)]
for name, col in zip(SPLITS, [BLUE, ORANGE, AQUA]):
    ax.bar(rows_y.index, rows_y[name], color=col, label=name, width=0.8)
ax.legend(frameon=False, ncol=3); ax.set_title('Rows per season by split'); ax.set_xlabel('season')
plt.tight_layout(); plt.show()
train_mask = ft['split'].eq('train')

## 15. Feature evidence and selection (training seasons only)
Three kinds of evidence, all computed on the **training split** so the validation and test seasons stay unseen:
1. **Podium rate across the range of each feature** (binned) — does the feature separate podium finishers?
2. **Univariate ROC-AUC / PR-AUC** of each feature used on its own as a score (direction-adjusted) — how much signal it carries alone.
3. **Correlation between features** — which ones carry the same information.

**Selection rule (decided before looking at validation):** keep a feature if its univariate ROC-AUC is **≥ 0.55** (i.e. at least 5 points better than chance), *or* if it is a flag that only makes sense together with another kept feature (`pit_lane_start` with `grid`, `no_quali_time` with the qualifying features, `is_first_round` with the standings). The ablation in Part 5 then measures what each *group* actually contributes after training.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

tr = ft[train_mask]
y_tr = tr['podium'].astype(int)
uni = []
for f in CANDIDATES:
    x = tr[f].astype(float)
    ok = x.notna()
    auc = roc_auc_score(y_tr[ok], x[ok])
    sign = 1 if auc >= 0.5 else -1                      # e.g. lower grid number = better
    uni.append({'feature': f, 'group': next(g for g, fs in FEATURE_GROUPS.items() if f in fs),
                'direction': 'higher → podium' if sign > 0 else 'lower → podium',
                'ROC-AUC': max(auc, 1 - auc), 'PR-AUC': average_precision_score(y_tr[ok], sign * x[ok]),
                'coverage': ok.mean()})
uni = pd.DataFrame(uni).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
display(uni.style.format({'ROC-AUC': '{:.3f}', 'PR-AUC': '{:.3f}', 'coverage': '{:.0%}'})
        .bar(subset=['ROC-AUC'], color='#b7d3f6', vmin=0.5, vmax=1))

fig, ax = plt.subplots(figsize=(11, 0.32 * len(uni) + 1.2))
u = uni[::-1]
ax.barh(u['feature'], u['ROC-AUC'] - 0.5, left=0.5, color=[BLUE if a >= 0.55 else GREY for a in u['ROC-AUC']], height=0.6)
ax.axvline(0.55, color=ORANGE, ls='--', lw=1.2); ax.text(0.553, len(u) - 0.6, 'selection threshold 0.55', color=ORANGE, fontsize=9)
ax.set_xlim(0.45, 1); ax.grid(axis='y', visible=False)
ax.set_title(f'Univariate ROC-AUC on the training seasons (base podium rate {y_tr.mean():.1%})')
plt.tight_layout(); plt.show()

In [ ]:
# Podium rate across each feature's range (quantile bins), training seasons
show = ['grid', 'quali_position', 'gap_to_pole_pct', 'grid_penalty', 'drv_podium_rate_last5', 'drv_finish_pct_last5',
        'con_podiums_per_race_last5', 'con_best_finish_pct_last5', 'drv_champ_pos_prev', 'con_points_share_prev',
        'drv_circuit_podium_rate_prev', 'driver_age']
fig, axes = plt.subplots(3, 4, figsize=(16, 9.5))
for ax, f in zip(axes.flat, show):
    x = tr[f].astype(float)
    bins = pd.qcut(x, q=10, duplicates='drop')
    r_ = tr.groupby(bins, observed=True)['podium'].mean()
    mid = [iv.mid for iv in r_.index]
    ax.plot(mid, r_.values, marker='o', ms=4, color=BLUE)
    ax.axhline(y_tr.mean(), color=GREY, ls=':', lw=1)
    pct_axis(ax); ax.set_title(f, fontsize=10); ax.set_ylim(0, 1)
fig.suptitle('Podium rate by decile of each feature (training seasons; dotted = base rate)', fontweight='bold')
plt.tight_layout(); plt.show()

# Binary features: podium rate when the flag is on vs off
flags = ['pit_lane_start', 'quali_top10', 'no_quali_time', 'is_first_round', 'home_race']
display(pd.DataFrame({f: tr.groupby(tr[f].astype(bool))['podium'].mean() for f in flags}).T
        .rename(columns={False: 'flag = 0', True: 'flag = 1'}).assign(rows_with_flag=[int(tr[f].sum()) for f in flags])
        .style.format({'flag = 0': '{:.1%}', 'flag = 1': '{:.1%}'}))

In [ ]:
corr = tr[CANDIDATES].astype(float).corr(method='spearman')
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8); ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
ax.grid(False); plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
ax.set_title('Spearman correlation between candidate features (training seasons)')
plt.tight_layout(); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename('rho').reset_index()
         .assign(abs_rho=lambda d: d['rho'].abs()).sort_values('abs_rho', ascending=False).head(8))
display(pairs[['level_0', 'level_1', 'rho']].rename(columns={'level_0': 'feature A', 'level_1': 'feature B'}).style.format({'rho': '{:.2f}'}))

In [ ]:
# Apply the selection rule
COMPANION_FLAGS = {'pit_lane_start': 'grid', 'no_quali_time': 'quali_position', 'is_first_round': 'drv_champ_pos_prev'}
auc_of = uni.set_index('feature')['ROC-AUC']
keep = [f for f in CANDIDATES if auc_of[f] >= 0.55 or (f in COMPANION_FLAGS and auc_of[COMPANION_FLAGS[f]] >= 0.55)]
dropped = [f for f in CANDIDATES if f not in keep]
SELECTED_GROUPS = {g: [f for f in fs if f in keep] for g, fs in FEATURE_GROUPS.items()}
SELECTED_GROUPS = {g: fs for g, fs in SELECTED_GROUPS.items() if fs}
FEATURES = [f for fs in SELECTED_GROUPS.values() for f in fs]

sel = uni.set_index('feature').loc[CANDIDATES, ['group', 'ROC-AUC', 'direction']]
sel['decision'] = ['keep' if f in keep else 'drop' for f in sel.index]
sel['reason'] = [('signal ≥ 0.55' if auc_of[f] >= 0.55 else f'companion flag of {COMPANION_FLAGS[f]}') if f in keep
                 else 'univariate ROC-AUC < 0.55 (no usable signal alone)' for f in sel.index]
display(sel.style.format({'ROC-AUC': '{:.3f}'}))
log_decision('F-01', '15', f'{len(CANDIDATES)} candidate pre-race features in {len(FEATURE_GROUPS)} groups',
             f'Keep {len(FEATURES)} features (univariate ROC-AUC ≥ 0.55 on train, or companion flag)',
             'Evidence-based, decided on training seasons only; ablation (Part 5) measures group contributions',
             'Keep all candidates; model-based selection', f"dropped: {', '.join(dropped) or 'none'}", 'Part 3')
say(f"**Selected {len(FEATURES)} features** in {len(SELECTED_GROUPS)} groups. Dropped: {', '.join(f'`{f}`' for f in dropped) or 'none'}. "
    f"`grid` and `quali_position` are strongly correlated (ρ = {corr.loc['grid', 'quali_position']:.2f}) but both are kept: they differ for "
    "penalised drivers (§5.4, Question 1) and the regularised models handle the overlap; the ablation will show whether one is redundant.")

## 16. Preprocessing and preprocessing-order experiments

The preprocessing has two kinds of steps, and the difference matters for leakage:

| Step | Learns from data? | Where it runs |
|---|---|---|
| **Domain fill** — row-level rules from F1 itself: no qualifying position → qualified last (`field_size`); no qualifying time → slower than the slowest car in that race; absent from the previous standings → behind everyone in that table, 0 % of the leader's points; pit-lane start → `grid = field_size` | No (uses only the row's own race) | before the pipeline, same for every split |
| **Median imputation + missing indicators** — for what is left (a driver's first start has no form) | Yes (medians) | inside the sklearn `Pipeline`, **fit on training rows only** |
| **Standard scaling** | Yes (means, stds) | inside the `Pipeline`, fit on training rows only |

The experiments below compare alternatives with a **fixed baseline model** (logistic regression, the simplest reasonable model) on the **validation** seasons. Primary metric: PR-AUC (positives ≈ 14 %, D-07); ROC-AUC reported alongside.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def domain_fill(df, grid_encoding='last'):
    # Row-level, F1-rule-based filling. Learns nothing from other rows/splits → no leakage.
    d = df.copy()
    d['quali_position'] = d['quali_position'].astype(float).fillna(d['field_size'].astype(float))
    d['gap_to_pole_pct'] = d['gap_to_pole_pct'].fillna(d['_race_max_gap'] + 1.0)        # slower than the slowest
    d['grid_penalty'] = d['grid_penalty'].fillna(0.0)
    d['drv_champ_pos_prev'] = d['drv_champ_pos_prev'].astype(float).fillna(d['_drv_table_size_prev'].astype(float) + 1)
    d['con_champ_pos_prev'] = d['con_champ_pos_prev'].astype(float).fillna(d['_con_table_size_prev'].astype(float) + 1)
    d['drv_points_share_prev'] = d['drv_points_share_prev'].fillna(0.0)
    d['con_points_share_prev'] = d['con_points_share_prev'].fillna(0.0)
    if grid_encoding == 'last':
        d['grid'] = d['grid'].astype(float).where(d['grid'] > 0, d['field_size'].astype(float))
    return d

def make_X(df, features, fill=True, grid_encoding='last'):
    d = domain_fill(df, grid_encoding) if fill else df.copy()
    if not fill and grid_encoding == 'last':
        d['grid'] = d['grid'].astype(float).where(d['grid'] > 0, d['field_size'].astype(float))
    return d[features].astype('float64')

def make_pipeline(indicators=True, C=1.0):
    return Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=indicators)),
                     ('scale', StandardScaler()),
                     ('model', LogisticRegression(C=C, max_iter=3000))])

y = ft['podium'].astype(int)
is_tr, is_va = ft['split'].eq('train'), ft['split'].eq('validation')

def evaluate(pipe, X, fit_mask=None, prefit=False):
    if not prefit:
        pipe.fit(X[is_tr], y[is_tr])
    p_tr, p_va = pipe.predict_proba(X[is_tr])[:, 1], pipe.predict_proba(X[is_va])[:, 1]
    return {'train PR-AUC': average_precision_score(y[is_tr], p_tr), 'val PR-AUC': average_precision_score(y[is_va], p_va),
            'val ROC-AUC': roc_auc_score(y[is_va], p_va)}

### 16.1 Effect of preprocessing on the data (training rows)

In [ ]:
X_raw = ft[FEATURES].astype('float64')
X_dom = make_X(ft, FEATURES)
pipe0 = make_pipeline().fit(X_dom[is_tr], y[is_tr])
Z = pd.DataFrame(pipe0[:-1].transform(X_dom[is_tr]), columns=pipe0[:-1].get_feature_names_out())

eff = pd.DataFrame({'missing before (raw)': X_raw[is_tr].isna().mean(), 'missing after domain fill': X_dom[is_tr].isna().mean(),
                    'mean before scaling': X_dom[is_tr].mean(), 'std before scaling': X_dom[is_tr].std(),
                    'mean after scaling': Z[FEATURES].mean().values, 'std after scaling': Z[FEATURES].std().values})
display(eff.style.format({c: '{:.1%}' for c in eff.columns[:2]} | {c: '{:.2f}' for c in eff.columns[2:]}))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
qb = np.arange(0.5, 27.5, 1)
ax[0].hist(X_raw.loc[is_tr, 'quali_position'].dropna(), bins=qb, color=GREY, alpha=0.7, label='raw (NaN dropped)')
ax[0].hist(X_dom.loc[is_tr, 'quali_position'], bins=qb, color=BLUE, histtype='step', lw=2, label='after domain fill')
ax[0].legend(frameon=False); ax[0].set_title('quali_position')
ax[1].hist(X_dom.loc[is_tr, 'gap_to_pole_pct'].clip(upper=15), bins=60, color=BLUE)
ax[1].set_title('gap_to_pole_pct (clipped at 15 % for display)'); ax[1].set_xlabel('% slower than pole')
ax[2].hist(Z['drv_champ_pos_prev'] if 'drv_champ_pos_prev' in Z else Z.iloc[:, 0], bins=40, color=BLUE)
ax[2].set_title('drv_champ_pos_prev after scaling (mean 0, sd 1)')
plt.tight_layout(); plt.show()
say(f"Domain fill removes the missing values that have a rule-based answer; what remains "
    f"({', '.join(f'`{c}`' for c in X_dom.columns[X_dom[is_tr].isna().any()]) or 'nothing'}) is imputed with training medians plus a "
    f"0/1 indicator per column (**{Z.shape[1] - len(FEATURES)} indicator columns** created). Scaling puts every feature on mean 0 / sd 1 "
    "(needed for logistic regression and the neural network; tree models are insensitive to it).")

In [ ]:
experiments = []
def record(exp, variant, res, chosen=False):
    experiments.append({'experiment': exp, 'variant': variant, **res, 'chosen': chosen})

# Baseline configuration used as the reference in every experiment
BASE_RES = evaluate(make_pipeline(), X_dom)

# PO-1 — ORDER: fit imputer + scaler on ALL seasons before splitting (leaky) vs on training seasons only
pre_all = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())]).fit(X_dom)
lr_all = LogisticRegression(max_iter=3000).fit(pre_all.transform(X_dom[is_tr]), y[is_tr])
p_va = lr_all.predict_proba(pre_all.transform(X_dom[is_va]))[:, 1]
p_tr = lr_all.predict_proba(pre_all.transform(X_dom[is_tr]))[:, 1]
record('PO-1 fit preprocessing before vs after split', 'fit on all seasons, then split (leaky)',
       {'train PR-AUC': average_precision_score(y[is_tr], p_tr), 'val PR-AUC': average_precision_score(y[is_va], p_va),
        'val ROC-AUC': roc_auc_score(y[is_va], p_va)})
record('PO-1 fit preprocessing before vs after split', 'split, then fit on train only', BASE_RES, chosen=True)
# how far the 'all seasons' statistics moved, in training standard deviations
shift = ((pre_all.named_steps['scale'].mean_[:len(FEATURES)] - pipe0.named_steps['scale'].mean_[:len(FEATURES)])
         / pipe0.named_steps['scale'].scale_[:len(FEATURES)])

In [ ]:
# PO-2 — ORDER: impute raw qualifying times, THEN compute the gap  vs  compute the gap, THEN fill it
alt = ft.copy()
tr_medians = ft.loc[is_tr, qcols].median()
alt[qcols] = alt[qcols].fillna(tr_medians)                       # impute raw seconds with training medians first …
alt['best_quali_s'] = alt[qcols].min(axis=1)                     # … then derive the feature
alt['gap_to_pole_pct'] = (alt['best_quali_s'] / alt.groupby('raceId')['best_quali_s'].transform('min') - 1) * 100
X_alt = make_X(alt, FEATURES)
record('PO-2 impute raw times → gap  vs  gap → fill', 'impute raw q-times, then compute gap', evaluate(make_pipeline(), X_alt))
record('PO-2 impute raw times → gap  vs  gap → fill', 'compute gap, then fill (slowest + 1 %)', BASE_RES, chosen=True)
gap_bad = alt.loc[ft['no_quali_time'], 'gap_to_pole_pct']

# PO-3 — missing-value strategy
record('PO-3 missing-value strategy', 'median imputation only', evaluate(make_pipeline(indicators=False), make_X(ft, FEATURES, fill=False)))
record('PO-3 missing-value strategy', 'median + missing indicators', evaluate(make_pipeline(indicators=True), make_X(ft, FEATURES, fill=False)))
record('PO-3 missing-value strategy', 'domain fill + median + indicators', BASE_RES)

# PO-4 — encoding of grid = 0 (pit-lane start), decision D-08
if 'grid' in FEATURES:
    record('PO-4 grid = 0 encoding (D-08)', 'keep 0 (pit lane looks like pole!)', evaluate(make_pipeline(), make_X(ft, FEATURES, grid_encoding='raw')))
    record('PO-4 grid = 0 encoding (D-08)', 'recode to field size (last) + flag', BASE_RES)

exp_df = pd.DataFrame(experiments)

# How large is the noise? Bootstrap the validation races (resample whole races) for the baseline configuration
base_pipe = make_pipeline().fit(X_dom[is_tr], y[is_tr])
p_base = base_pipe.predict_proba(X_dom[is_va])[:, 1]
y_va, race_va = y[is_va].values, ft.loc[is_va, 'raceId'].values
rng = np.random.default_rng(42)
rows_of = {r: np.where(race_va == r)[0] for r in np.unique(race_va)}
boot = []
for _ in range(500):
    idx = np.concatenate([rows_of[r] for r in rng.choice(list(rows_of), len(rows_of))])
    boot.append(average_precision_score(y_va[idx], p_base[idx]))
BOOT_CI = np.percentile(boot, [2.5, 97.5])
# Share of validation rows on which the variants can differ at all
# (rows where the domain fill / grid recoding changes at least one value)
touched = (make_X(ft[is_va], FEATURES, fill=False, grid_encoding='raw').fillna(-999) != X_dom[is_va].fillna(-999)).any(axis=1)
affected, affected_podium = touched.mean(), y[is_va][touched].mean()

# Pick the winner of PO-3 / PO-4 on validation PR-AUC. A difference smaller than TOL (far below the bootstrap noise)
# counts as a tie, and ties go to the F1-rule-consistent option (listed last in each experiment).
TOL = 0.01
for e in ['PO-3 missing-value strategy', 'PO-4 grid = 0 encoding (D-08)']:
    sub = exp_df[exp_df['experiment'] == e]
    if len(sub):
        best = sub['val PR-AUC'].max()
        winner = sub[sub['val PR-AUC'] >= best - TOL].index[-1]
        exp_df.loc[winner, 'chosen'] = True
display(exp_df.style.format({c: '{:.4f}' for c in ['train PR-AUC', 'val PR-AUC', 'val ROC-AUC']})
        .apply(lambda r: ['background-color: #e6f0fb' if r['chosen'] else '' for _ in r], axis=1))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 0.42 * len(exp_df) + 1.2))
lbl = [f"{e.split(' ')[0]} · {v}" for e, v in zip(exp_df['experiment'], exp_df['variant'])]
yy_ = np.arange(len(exp_df))[::-1]
lo_ = min(exp_df['val PR-AUC'].min(), BOOT_CI[0]) - 0.01
ax.axvspan(BOOT_CI[0], BOOT_CI[1], color=ORANGE, alpha=0.12, zorder=0, label='95 % bootstrap CI of the baseline (races resampled)')
ax.scatter(exp_df['val PR-AUC'], yy_, s=70, zorder=3, color=[BLUE if c else GREY for c in exp_df['chosen']])
ax.set_yticks(yy_, lbl)
ax.legend(frameon=False, loc='lower right', fontsize=8)
ax.set_xlim(lo_, max(exp_df['val PR-AUC'].max(), BOOT_CI[1]) + 0.01); ax.grid(axis='y', visible=False)
for yv, v in zip(yy_, exp_df['val PR-AUC']):
    ax.text(v + 0.003, yv, f'{v:.3f}', va='center', fontsize=8)
ax.set_title('Preprocessing experiments — validation PR-AUC (logistic regression; blue = chosen)')
plt.tight_layout(); plt.show()

po = lambda e, v: exp_df[(exp_df['experiment'].str.startswith(e)) & (exp_df['variant'] == v)]['val PR-AUC'].iloc[0]
say(f'''
**What the experiments show (validation seasons 2019–2021):**
* **PO-1 (order of fitting vs splitting):** fitting the imputer/scaler on all seasons gives val PR-AUC {po('PO-1', 'fit on all seasons, then split (leaky)'):.4f} vs {po('PO-1', 'split, then fit on train only'):.4f}. The scores barely move because medians and means of these features are stable over time (the largest shift of a feature mean is {np.abs(shift).max():.2f} training standard deviations, for `{FEATURES[int(np.abs(shift).argmax())]}`), **but the leaky version uses test-season statistics**, so it is rejected on principle: the gain in honesty costs nothing.
* **PO-2 (impute raw times → gap vs gap → fill):** {po('PO-2', 'impute raw q-times, then compute gap'):.4f} vs {po('PO-2', 'compute gap, then fill (slowest + 1 %)'):.4f}. Imputing raw seconds first is wrong *by construction*: a median lap time pooled over all circuits gives drivers without a time a gap between {gap_bad.min():.1f} % and {gap_bad.max():.0f} %{' (some of them as fast as pole)' if gap_bad.min() < 0.5 else ''}, and drivers knocked out in Q1/Q2 receive a median Q2/Q3 time they never drove. The feature is therefore built first and filled afterwards.
* **PO-3 (missing-value strategy):** median only {po('PO-3', 'median imputation only'):.4f}; + indicators {po('PO-3', 'median + missing indicators'):.4f}; domain fill + indicators {po('PO-3', 'domain fill + median + indicators'):.4f}.
* **PO-4 (grid = 0):** keeping 0 {po('PO-4', 'keep 0 (pit lane looks like pole!)'):.4f} vs recoding to last {po('PO-4', 'recode to field size (last) + flag'):.4f}; with 0 the model reads a pit-lane start as "better than pole".

**How big is the noise?** Resampling the {len(rows_of)} validation races gives a 95 % interval of **{BOOT_CI[0]:.3f}–{BOOT_CI[1]:.3f}** for the baseline PR-AUC — every variant lies well inside it. That is expected: the domain fill and grid recoding only change values on **{affected:.1%} of validation rows** (missing qualifying/standings values, pit-lane starts), and only {affected_podium:.1%} of those rows are podiums, so they barely move a ranking metric. So no variant is *measurably* better; the choice is made on correctness instead — no test-season statistics (PO-1), no impossible gaps (PO-2), rule-based fills that also work for a single new record in the inference function (PO-3), and no "pit lane = pole" (PO-4). Rule used: within {TOL} PR-AUC of the best variant, keep the F1-consistent option.
''')

In [ ]:
# Final preprocessing configuration and hand-off table for Part 4
chosen = exp_df[exp_df['chosen']]
choice = lambda e: chosen.loc[chosen['experiment'].str.startswith(e), 'variant'].iloc[0] if chosen['experiment'].str.startswith(e).any() else ''
PREPROCESSING = {
    'domain_fill': 'domain' in choice('PO-3'),
    'missing_indicators': choice('PO-3') != 'median imputation only',
    'grid_encoding': 'raw' if choice('PO-4').startswith('keep 0') else 'last',
    'fit_on': 'training seasons only (inside the sklearn Pipeline)',
}
display(pd.Series(PREPROCESSING, name='value').to_frame())

model_table = ft[['raceId', 'driverId', 'constructorId', 'year', 'round', 'date', 'circuitId', 'split', 'podium']
                 + CANDIDATES + HELPERS].copy()
save_table(model_table, OUT_DIR / 'model_table_2003_2024')
import json
with open(OUT_DIR / 'feature_config.json', 'w') as fh:
    json.dump({'FEATURE_GROUPS': SELECTED_GROUPS, 'FEATURES': FEATURES, 'PREPROCESSING': PREPROCESSING, 'SPLITS': SPLITS}, fh, indent=2)
exp_df.to_csv(OUT_DIR / 'preprocessing_experiments.csv', index=False)
uni.to_csv(OUT_DIR / 'feature_evidence_univariate.csv', index=False)
for e, r in chosen.iterrows():
    log_decision(r['experiment'].split(' ')[0], '16', f"{r['experiment']}", r['variant'], 'best or equal validation PR-AUC; F1-rule-consistent',
                 '; '.join(exp_df.loc[(exp_df['experiment'] == r['experiment']) & ~exp_df['chosen'], 'variant']),
                 f"val PR-AUC {r['val PR-AUC']:.4f}", 'Part 3')
pd.DataFrame(DECISIONS).to_csv(OUT_DIR / 'decision_log.csv', index=False)

say(f'''
### Part 3 summary
* **Modeling table:** {len(model_table):,} rows (one per driver–race, 2003–2024), split by season into
  train {int((ft.split == 'train').sum()):,} / validation {int((ft.split == 'validation').sum()):,} / test {int((ft.split == 'test').sum()):,}.
* **{len(FEATURES)} selected features** in {len(SELECTED_GROUPS)} groups ({', '.join(f"{g}: {len(fs)}" for g, fs in SELECTED_GROUPS.items())}), all computable after qualifying and before the start; leakage tests passed.
* **Preprocessing (chosen in §16):** {'domain fill → ' if PREPROCESSING['domain_fill'] else ''}median imputation{' + missing indicators' if PREPROCESSING['missing_indicators'] else ''} → standard scaling, fit on training seasons only; grid = 0 encoded as `{PREPROCESSING['grid_encoding']}`.
* **Baseline (logistic regression) on validation:** PR-AUC {BASE_RES['val PR-AUC']:.3f}, ROC-AUC {BASE_RES['val ROC-AUC']:.3f} (a random model scores PR-AUC ≈ {y[is_va].mean():.3f}).
* **Saved for Part 4:** `model_table_2003_2024`, `feature_config.json` (groups for the ablation), `preprocessing_experiments.csv`, `feature_evidence_univariate.csv`, updated `decision_log.csv`.
''')

### Hand-off to Part 4 (modeling) and Part 5 (experiments)
* Train every model on the **same rows and features** (`FEATURES`, split column) with `make_pipeline`-style preprocessing so scores are comparable; the logistic regression above is the **baseline**.
* Select thresholds (for F1) and hyper-parameters on the **validation** seasons only; evaluate the test seasons once at the end.
* Feature-group ablation: drop one group of `SELECTED_GROUPS` at a time and retrain.
* Limitations added in Part 3: form resets when a team is re-branded (new `constructorId`); `gap_to_pole_pct` compares best times from different knock-out sessions; `home_race` relies on recorded nationality. All model explanations later are associations, not causes.

---
---
# PART 4 — Predictive Modeling

Every model is trained on **the same rows (training seasons 2003–2018) and the same features** (`FEATURES`, preprocessing chosen in §16), so the scores are directly comparable.

| Role | Model | Why it is here |
|---|---|---|
| Reference without ML | **Rule: grid ≤ 3** | what you get by simply predicting the top 3 starters |
| **Baseline** (statistical) | **Logistic regression** | simplest reasonable model; linear in the features |
| Statistical model 2 | **Random forest** | many decision trees on random subsets → non-linear, robust |
| Statistical model 3 | **Gradient boosting** (`HistGradientBoostingClassifier`) | trees added one at a time, each fixing the previous errors |
| Neural network | **Shallow feed-forward network** (`MLPClassifier`, 1–2 hidden layers) | learns non-linear combinations; required by the milestone |

**Metrics (D-07).** Podiums are ~14 % of rows, so accuracy is useless (always predicting "no podium" scores 86 %).
* **Primary — PR-AUC:** how well real podium finishers are ranked above everyone else; its random baseline is the podium rate (~0.14), so it is not flattered by the many easy "no podium" rows.
* **Secondary — ROC-AUC:** overall separation of the two classes (random = 0.5); comparable across eras with different field sizes.
* **F1, precision, recall:** quality of the yes/no decision at a **probability threshold chosen on the validation seasons** (the threshold that maximises validation F1).
* **Domain metric — top-3 hit rate:** in each race, take the 3 drivers with the highest predicted probability; the share of them that actually finished on the podium.

**Rules that keep the evaluation honest:** hyper-parameters and thresholds are chosen on **validation** only; the **test** seasons (2022–2024) are scored once, at the end, with everything frozen.

## 17. Shared setup: preprocessing, metrics, threshold

In [ ]:
import copy, time, itertools, joblib
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, precision_recall_curve, roc_curve,
                             confusion_matrix)
from sklearn.calibration import calibration_curve

SEED = 42
is_te = ft['split'].eq('test')
SPLIT_MASKS = {'train': is_tr, 'validation': is_va, 'test': is_te}

# The one design matrix every model uses (preprocessing decided in §16)
X_all = make_X(ft, FEATURES, fill=PREPROCESSING['domain_fill'], grid_encoding=PREPROCESSING['grid_encoding'])

def preprocessor():
    # Learned steps: fitted on the training rows only, inside the pipeline
    return Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=PREPROCESSING['missing_indicators'])),
                     ('scale', StandardScaler())])

def build(model):
    return Pipeline([('prep', preprocessor()), ('model', model)])

def f1_threshold(y_true, p):
    # Probability threshold that maximises F1 (used on the VALIDATION seasons only)
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    return float(thr[np.nanargmax(f1[:-1])])

def top3_hit_rate(mask, p):
    # Per race: share of the 3 highest-probability drivers who actually finished on the podium
    d = pd.DataFrame({'raceId': ft.loc[mask, 'raceId'].values, 'p': p, 'y': y[mask].values})
    return d.sort_values('p', ascending=False).groupby('raceId').head(3)['y'].mean()

def all_metrics(mask, p, thr):
    yt = y[mask].values
    yhat = (p >= thr).astype(int)
    return {'PR-AUC': average_precision_score(yt, p), 'ROC-AUC': roc_auc_score(yt, p), 'F1': f1_score(yt, yhat),
            'precision': precision_score(yt, yhat, zero_division=0), 'recall': recall_score(yt, yhat),
            'top-3 hit rate': top3_hit_rate(mask, p)}

print(f'{len(FEATURES)} features | train {int(is_tr.sum()):,} · validation {int(is_va.sum()):,} · test {int(is_te.sum()):,} rows')

## 18. The models — how they work, tuning, and limitations
Each model gets a small, explicit hyper-parameter grid. **Every configuration is kept in the notebook** (table + plot in §18.5), so the trial-and-error is visible, and the configuration with the best **validation PR-AUC** is selected.

### 18.1 Reference rule and baseline: logistic regression
* **Rule "grid ≤ 3":** predict a podium for the three front starters. No learning at all — any model must beat it to be worth its complexity.
* **Logistic regression — how it works:** it adds up the (scaled) features, each multiplied by a learned weight, and squashes the sum through the logistic function into a probability between 0 and 1. Weights are learned by maximising the likelihood of the training labels, with an L2 penalty (`C` = inverse strength) that shrinks weights and handles correlated features.
* **Limitations:** the effect of each feature is a straight line on the log-odds scale — it cannot represent curves such as the age hump or the `grid_penalty` hump (§15) or interactions (e.g. "a penalty matters more for a fast car") unless they are built by hand.

In [ ]:
tuning, fitted, histories = [], {}, {}

def tune(name, make_model, grid):
    # Fit every configuration on train, score on validation; return the best fitted pipeline.
    best = None
    for params in grid:
        t0 = time.time()
        pipe = build(make_model(**params)).fit(X_all[is_tr], y[is_tr])
        p_va = pipe.predict_proba(X_all[is_va])[:, 1]
        row = {'model': name, 'config': ', '.join(f'{k}={v}' for k, v in params.items()),
               'val PR-AUC': average_precision_score(y[is_va], p_va), 'val ROC-AUC': roc_auc_score(y[is_va], p_va),
               'fit seconds': round(time.time() - t0, 1)}
        tuning.append(row)
        if best is None or row['val PR-AUC'] > best[0]['val PR-AUC']:
            best = (row, pipe)
    print(f"{name}: best {best[0]['config']} → validation PR-AUC {best[0]['val PR-AUC']:.4f}")
    return best[1]

grid_lr = [dict(C=c, class_weight=w) for c in [0.01, 0.1, 1.0, 10.0] for w in [None, 'balanced']]
fitted['Logistic regression (baseline)'] = tune(
    'Logistic regression (baseline)', lambda **k: LogisticRegression(max_iter=5000, **k), grid_lr)

### 18.2 Random forest
* **How it works:** grows several hundred decision trees, each on a bootstrap sample of the rows and a random subset of features at every split; the predicted probability is the average vote. Averaging many de-correlated trees reduces variance.
* **Tuned:** tree depth, minimum rows per leaf (larger = smoother probabilities), share of features tried per split.
* **Limitations:** probabilities are often compressed towards the middle (not well calibrated); cannot extrapolate beyond values seen in training; less transparent than a linear model (hence the XAI in Part 6).

In [ ]:
grid_rf = [dict(n_estimators=400, max_depth=d, min_samples_leaf=l, max_features=f, n_jobs=-1, random_state=SEED)
           for d in [6, 10, None] for l in [5, 20] for f in ['sqrt', 0.5]]
fitted['Random forest'] = tune('Random forest', lambda **k: RandomForestClassifier(**k), grid_rf)

### 18.3 Gradient boosting
* **How it works:** builds shallow trees **sequentially**; each new tree is fitted to the errors (gradients of the log-loss) of the current ensemble and added with a small learning rate. The histogram variant bins each feature first, which makes it fast.
* **Tuned:** learning rate, tree depth, L2 regularisation (number of boosting rounds fixed at 300).
* **Limitations:** the most flexible of the three statistical models, so the most prone to over-fitting the training seasons (watch the train–validation gap); sensitive to the learning-rate / rounds combination.

In [ ]:
grid_hgb = [dict(learning_rate=lr, max_depth=d, l2_regularization=l2, max_iter=300, random_state=SEED)
            for lr in [0.03, 0.1] for d in [3, 6] for l2 in [0.0, 1.0]]
fitted['Gradient boosting'] = tune('Gradient boosting', lambda **k: HistGradientBoostingClassifier(**k), grid_hgb)

### 18.4 Shallow neural network (feed-forward, 1–2 hidden layers)
* **How it works:** the scaled features feed a hidden layer of neurons (each a weighted sum followed by a ReLU, `max(0, x)`), optionally a second hidden layer, and a single sigmoid output giving the podium probability. Weights are learned with the Adam optimiser by minimising log-loss on mini-batches of 128 rows; `alpha` is an L2 penalty on the weights.
* **Training loop:** one call to `partial_fit` = one pass (epoch) over the training rows. After every epoch the training and **validation** PR-AUC are recorded; training stops after 20 epochs without validation improvement (**early stopping**) and the best epoch is kept. These curves are the "performance during training and validation" plot.
* **Tuned:** architecture (1 layer of 32 vs 2 layers of 64 → 32) and `alpha`.
* **Limitations:** needs scaled inputs; results vary with the random initialisation; least interpretable; with ~6,400 rows and features that are already strong, it has little room to beat simpler models.

In [ ]:
def train_mlp(hidden, alpha, max_epochs=200, patience=20, seed=SEED):
    prep = preprocessor().fit(X_all[is_tr])
    Xtr, Xva = prep.transform(X_all[is_tr]), prep.transform(X_all[is_va])
    ytr = y[is_tr].values
    mlp = MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, learning_rate_init=1e-3, batch_size=128,
                        random_state=seed, shuffle=True)
    hist, best, wait = [], (-1.0, None, 0), 0
    for epoch in range(1, max_epochs + 1):
        mlp.partial_fit(Xtr, ytr, classes=[0, 1])
        p_tr, p_va = mlp.predict_proba(Xtr)[:, 1], mlp.predict_proba(Xva)[:, 1]
        val = average_precision_score(y[is_va], p_va)
        hist.append({'epoch': epoch, 'train log-loss': mlp.loss_,
                     'train PR-AUC': average_precision_score(ytr, p_tr), 'val PR-AUC': val})
        if val > best[0] + 1e-4:
            best, wait = (val, copy.deepcopy(mlp), epoch), 0
        else:
            wait += 1
            if wait >= patience:
                break
    pipe = Pipeline([('prep', prep), ('model', best[1])])     # already fitted: prep on train, net at best epoch
    return pipe, pd.DataFrame(hist), best[2]

best_mlp = None
for hidden, alpha in itertools.product([(32,), (64, 32)], [1e-4, 1e-3, 1e-2]):
    t0 = time.time()
    pipe, hist, best_epoch = train_mlp(hidden, alpha)
    p_va = pipe.predict_proba(X_all[is_va])[:, 1]
    row = {'model': 'Neural network (MLP)', 'config': f'hidden={hidden}, alpha={alpha}, best epoch={best_epoch}',
           'val PR-AUC': average_precision_score(y[is_va], p_va), 'val ROC-AUC': roc_auc_score(y[is_va], p_va),
           'fit seconds': round(time.time() - t0, 1)}
    tuning.append(row)
    if best_mlp is None or row['val PR-AUC'] > best_mlp[0]['val PR-AUC']:
        best_mlp = (row, pipe, hist, best_epoch)
fitted['Neural network (MLP)'] = best_mlp[1]
histories['Neural network (MLP)'] = best_mlp[2]
print(f"Neural network (MLP): best {best_mlp[0]['config']} → validation PR-AUC {best_mlp[0]['val PR-AUC']:.4f}")

h, be = best_mlp[2], best_mlp[3]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].plot(h['epoch'], h['train log-loss'], color=BLUE)
ax[0].set_title('Training loss per epoch (selected network)'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('log-loss')
ax[1].plot(h['epoch'], h['train PR-AUC'], color=BLUE, label='train')
ax[1].plot(h['epoch'], h['val PR-AUC'], color=ORANGE, label='validation')
ax[1].axvline(be, color=GREY, ls='--', lw=1); ax[1].text(be + 0.5, h['val PR-AUC'].min(), f'best epoch {be}', color=GREY)
ax[1].set_title('PR-AUC per epoch: training vs validation'); ax[1].set_xlabel('epoch'); ax[1].legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()
val_above = (h['val PR-AUC'] > h['train PR-AUC']).mean() > 0.5
say(f"The training loss keeps falling and training PR-AUC keeps rising, but validation PR-AUC peaks at **epoch {be}** "
    f"({h.loc[h['epoch'] == be, 'val PR-AUC'].iloc[0]:.3f}) and then slowly declines: from there on the network learns patterns specific to the "
    "training seasons that do not carry over to new seasons — the start of over-fitting. Early stopping keeps the best-epoch weights. "
    + ("(Validation sits *above* training throughout because the 2019–2021 seasons are easier to predict — 20-car fields and a few dominant "
       "teams — not because the model is under-fitted; compare the shapes of the curves, not their levels.)" if val_above else ""))

### 18.5 All tuning attempts
Every configuration that was tried, scored on the validation seasons. The selected configuration of each model is the highest point.

In [ ]:
tuning_df = pd.DataFrame(tuning)
models_order = list(fitted)
fig, ax = plt.subplots(figsize=(12, 3.8))
for i, m in enumerate(models_order):
    t = tuning_df[tuning_df['model'] == m]
    jitter = np.linspace(-0.18, 0.18, len(t))
    ax.scatter(i + jitter, t['val PR-AUC'], color=GREY, s=26, zorder=2)
    ax.scatter(i + jitter[t['val PR-AUC'].values.argmax()], t['val PR-AUC'].max(), color=BLUE, s=70, zorder=3)
ax.set_xticks(range(len(models_order)), models_order)
ax.set_ylabel('validation PR-AUC'); ax.set_title('Hyper-parameter search: every configuration tried (blue = selected)')
ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()
display(tuning_df.sort_values(['model', 'val PR-AUC'], ascending=[True, False]).reset_index(drop=True)
        .style.format({'val PR-AUC': '{:.4f}', 'val ROC-AUC': '{:.4f}', 'fit seconds': '{:.1f}'}))

## 19. Evaluation: training, validation and the unseen test seasons
For every model the decision threshold is the one that maximises F1 on the **validation** seasons; it is then applied unchanged to the test seasons.

In [ ]:
# Predicted probabilities for every split
proba = {m: {s: pipe.predict_proba(X_all[mask])[:, 1] for s, mask in SPLIT_MASKS.items()} for m, pipe in fitted.items()}

# Reference rule: score = -grid (lower grid = more likely), decision = grid <= 3
grid_used = X_all['grid'] if 'grid' in X_all else ft['grid'].astype(float)
RULE = 'Rule: grid ≤ 3 (no ML)'
proba_rule = {s: (-grid_used[mask]).values for s, mask in SPLIT_MASKS.items()}

thresholds = {m: f1_threshold(y[is_va], proba[m]['validation']) for m in fitted}
rows = []
for s, mask in SPLIT_MASKS.items():
    rows.append({'model': RULE, 'split': s, **all_metrics(mask, proba_rule[s], -3.0)})
    for m in fitted:
        rows.append({'model': m, 'split': s, **all_metrics(mask, proba[m][s], thresholds[m])})
results = pd.DataFrame(rows)
METRICS = ['PR-AUC', 'ROC-AUC', 'F1', 'precision', 'recall', 'top-3 hit rate']
comparison = results.pivot(index='model', columns='split', values=METRICS)
comparison = comparison.reindex(columns=pd.MultiIndex.from_product([METRICS, ['train', 'validation', 'test']]))
comparison = comparison.loc[[RULE] + models_order]
display(comparison.style.format('{:.3f}').set_caption('Model comparison — all metrics on train / validation / test'))
display(pd.Series(thresholds, name='threshold (chosen on validation)').to_frame().style.format('{:.3f}'))

In [ ]:
# Uncertainty: bootstrap the TEST races (whole races resampled), 95 % intervals for PR-AUC
race_te = ft.loc[is_te, 'raceId'].values
rows_te = {r: np.where(race_te == r)[0] for r in np.unique(race_te)}
rng = np.random.default_rng(SEED)
boot_idx = [np.concatenate([rows_te[r] for r in rng.choice(list(rows_te), len(rows_te))]) for _ in range(500)]
y_te = y[is_te].values
ci = {}
for m in [RULE] + models_order:
    p = proba_rule['test'] if m == RULE else proba[m]['test']
    vals = [average_precision_score(y_te[i], p[i]) for i in boot_idx]
    ci[m] = np.percentile(vals, [2.5, 97.5])

best_model = max(models_order, key=lambda m: comparison.loc[m, ('PR-AUC', 'validation')])
base_name = 'Logistic regression (baseline)'
diff = [average_precision_score(y_te[i], proba[best_model]['test'][i]) - average_precision_score(y_te[i], proba[base_name]['test'][i])
        for i in boot_idx]
diff_ci = np.percentile(diff, [2.5, 97.5])

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
names = [RULE] + models_order
x = np.arange(len(names)); w = 0.26
for i, (s, col) in enumerate(zip(['train', 'validation', 'test'], [GREY, ORANGE, BLUE])):
    vals = comparison.loc[names, ('PR-AUC', s)].values
    ax[0].bar(x + (i - 1) * w, vals, w, color=col, label=s)
ax[0].errorbar(x + w, comparison.loc[names, ('PR-AUC', 'test')].values,
               yerr=np.clip(np.array([[comparison.loc[n, ('PR-AUC', 'test')] - ci[n][0], ci[n][1] - comparison.loc[n, ('PR-AUC', 'test')]] for n in names]).T, 0, None),
               fmt='none', ecolor='#333333', capsize=3, lw=1)
ax[0].axhline(y_te.mean(), color=GREY, ls=':', lw=1); ax[0].text(len(names) - 0.6, y_te.mean() + 0.015, 'random', color=GREY, fontsize=8, ha='right')
ax[0].set_xticks(x, [n.replace(' (', '\n(') for n in names], fontsize=8); ax[0].set_ylim(0, 1); ax[0].legend(frameon=False, ncol=3)
ax[0].set_title('PR-AUC by split (test bars with 95 % bootstrap CI)'); ax[0].grid(axis='x', visible=False)
for i, s in enumerate(['validation', 'test']):
    vals = comparison.loc[names, ('ROC-AUC', s)].values
    ax[1].bar(x + (i - 0.5) * 0.38, vals, 0.38, color=[ORANGE, BLUE][i], label=s)
ax[1].set_ylim(0.5, 1); ax[1].set_xticks(x, [n.replace(' (', '\n(') for n in names], fontsize=8)
ax[1].legend(frameon=False); ax[1].set_title('ROC-AUC (secondary metric)'); ax[1].grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Curves on the TEST seasons
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
cols = dict(zip(models_order, [BLUE, AQUA, ORANGE, MAGENTA]))
for m in models_order:
    prec, rec, _ = precision_recall_curve(y_te, proba[m]['test'])
    fpr, tpr, _ = roc_curve(y_te, proba[m]['test'])
    ax[0].plot(rec, prec, color=cols[m], lw=1.8, label=f"{m} ({comparison.loc[m, ('PR-AUC', 'test')]:.3f})")
    ax[1].plot(fpr, tpr, color=cols[m], lw=1.8, label=f"{m} ({comparison.loc[m, ('ROC-AUC', 'test')]:.3f})")
    frac_pos, mean_pred = calibration_curve(y_te, proba[m]['test'], n_bins=10, strategy='quantile')
    ax[2].plot(mean_pred, frac_pos, marker='o', ms=3, color=cols[m], lw=1.5, label=m)
ax[0].axhline(y_te.mean(), color=GREY, ls=':', lw=1)
ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].set_title('Precision–recall curve (test)'); ax[0].legend(frameon=False, fontsize=8)
ax[1].plot([0, 1], [0, 1], color=GREY, ls=':', lw=1)
ax[1].set_xlabel('false-positive rate'); ax[1].set_ylabel('true-positive rate'); ax[1].set_title('ROC curve (test)'); ax[1].legend(frameon=False, fontsize=8)
ax[2].plot([0, 1], [0, 1], color=GREY, ls=':', lw=1)
ax[2].set_xlabel('predicted probability'); ax[2].set_ylabel('observed podium rate'); ax[2].set_title('Calibration (test): is "60 %" really 60 %?')
ax[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

# Confusion matrices at the validation-chosen threshold (test seasons)
fig, axes = plt.subplots(1, len(models_order), figsize=(4.2 * len(models_order), 3.6))
for ax_, m in zip(axes, models_order):
    cm = confusion_matrix(y_te, (proba[m]['test'] >= thresholds[m]).astype(int))
    ax_.imshow(cm, cmap='Blues'); ax_.grid(False)
    for (i, j), v in np.ndenumerate(cm):
        ax_.text(j, i, f'{v:,}', ha='center', va='center', color='white' if v > cm.max() / 2 else '#222222', fontsize=12)
    ax_.set_xticks([0, 1], ['pred. no', 'pred. podium']); ax_.set_yticks([0, 1], ['no podium', 'podium'])
    ax_.set_title(f'{m}\nthreshold {thresholds[m]:.2f}', fontsize=10)
plt.suptitle('Confusion matrices on the test seasons', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
bm = comparison.loc[best_model]; bl = comparison.loc[base_name]; rl = comparison.loc[RULE]
gap = {m: comparison.loc[m, ('PR-AUC', 'train')] - comparison.loc[m, ('PR-AUC', 'validation')] for m in models_order}
sig = diff_ci[0] > 0
say(f'''
### Part 4 results
* **Selected model (highest validation PR-AUC): {best_model}** — validation PR-AUC {bm[('PR-AUC', 'validation')]:.3f}, **test PR-AUC {bm[('PR-AUC', 'test')]:.3f}**
  (95 % CI {ci[best_model][0]:.3f}–{ci[best_model][1]:.3f}), test ROC-AUC {bm[('ROC-AUC', 'test')]:.3f}, test F1 {bm[('F1', 'test')]:.3f}; in each test race, **{bm[('top-3 hit rate', 'test')]:.0%}** of its top-3 picks finished on the podium.
* **Baseline (logistic regression):** test PR-AUC {bl[('PR-AUC', 'test')]:.3f}. Difference best − baseline on test: {bm[('PR-AUC', 'test')] - bl[('PR-AUC', 'test')]:+.3f}
  (95 % CI {diff_ci[0]:+.3f} to {diff_ci[1]:+.3f}) → {"the extra complexity **earns its place**" if sig else "**not distinguishable from the baseline** — the simpler model would be an equally defensible choice"}.
* **Rule "grid ≤ 3" (no ML):** test PR-AUC {rl[('PR-AUC', 'test')]:.3f}, F1 {rl[('F1', 'test')]:.3f} — every model is compared against what the starting grid alone gives.
* **Over-fitting check (train − validation PR-AUC):** {', '.join(f"{m.split(' (')[0]} {g:+.3f}" for m, g in gap.items())}. A large **positive** gap means the model memorised the training seasons (the tree models fit training best). A **negative** gap does not mean under-fitting: the 2019–2021 validation seasons are simply easier to predict (smaller fields, dominant teams — the rule "grid ≤ 3" also scores higher there).
* **Validation vs test:** test seasons (2022–2024, new car regulations) {"score lower than validation — the expected cost of a regulation change" if bm[('PR-AUC', 'test')] < bm[('PR-AUC', 'validation')] else "score at least as well as validation — the model transfers to the new regulation era"}.
* **Calibration:** points close to the diagonal mean a predicted 60 % really happens about 60 % of the time — important for the inference function, which reports probabilities.
''')

---
---
# PART 5 — Experiments

## 20. Feature-group ablation
**Method:** for a model, remove **one group of features at a time**, retrain with the *same* hyper-parameters on the *same* rows, and measure how much PR-AUC drops. A big drop = the group carries information the other groups cannot replace.

* Groups are removed **as a whole** because features inside a group are strongly correlated (e.g. `grid`–`quali_position` ρ = 0.97, §15): removing one of a correlated pair would show almost no drop, because its twin takes over.
* Groups are also correlated *with each other* (grid vs qualifying; standings vs constructor form), so a single-group drop measures what the group adds **beyond the others**. Two **combined** ablations measure the shared information: *starting position* (grid + qualifying) and *championship strength* (standings + constructor form).
* Run for the **selected model** and the **baseline**; uncertainty from a paired bootstrap over validation races (same resampled races for the full and the ablated model).
* Before training, §15 justified each feature with exploratory evidence; this ablation measures each group's actual contribution after training.

In [ ]:
def refit_like(name, features):
    # Same model type and hyper-parameters as the tuned one, trained on a subset of features.
    if name == 'Neural network (MLP)':
        raise ValueError('ablation uses the statistical models')
    model = clone(fitted[name].named_steps['model'])
    return build(model).fit(X_all.loc[is_tr, features], y[is_tr])

ablation_sets = {f'– {g}': [f for f in FEATURES if f not in fs] for g, fs in SELECTED_GROUPS.items()}
start_pos = SELECTED_GROUPS.get('grid', []) + SELECTED_GROUPS.get('qualifying', [])
champ = SELECTED_GROUPS.get('standings', []) + SELECTED_GROUPS.get('constructor_form', [])
ablation_sets['– grid + qualifying (starting position)'] = [f for f in FEATURES if f not in start_pos]
ablation_sets['– standings + constructor form (championship strength)'] = [f for f in FEATURES if f not in champ]

ABL_MODELS = [m for m in dict.fromkeys([best_model if best_model != 'Neural network (MLP)' else 'Gradient boosting', base_name])]
race_va = ft.loc[is_va, 'raceId'].values
rows_va = {r: np.where(race_va == r)[0] for r in np.unique(race_va)}
rng = np.random.default_rng(SEED)
boot_va = [np.concatenate([rows_va[r] for r in rng.choice(list(rows_va), len(rows_va))]) for _ in range(300)]
y_va = y[is_va].values

abl_rows = []
for m in ABL_MODELS:
    p_full_va, p_full_te = proba[m]['validation'], proba[m]['test']
    full_va, full_te = average_precision_score(y_va, p_full_va), average_precision_score(y_te, p_full_te)
    for name, feats in ablation_sets.items():
        pipe = refit_like(m, feats)
        p_va = pipe.predict_proba(X_all.loc[is_va, feats])[:, 1]
        p_te = pipe.predict_proba(X_all.loc[is_te, feats])[:, 1]
        drops = [average_precision_score(y_va[i], p_full_va[i]) - average_precision_score(y_va[i], p_va[i]) for i in boot_va]
        lo, hi = np.percentile(drops, [2.5, 97.5])
        abl_rows.append({'model': m, 'removed': name, 'features left': len(feats),
                         'val PR-AUC': average_precision_score(y_va, p_va), 'val drop': full_va - average_precision_score(y_va, p_va),
                         'drop CI low': lo, 'drop CI high': hi,
                         'test PR-AUC': average_precision_score(y_te, p_te), 'test drop': full_te - average_precision_score(y_te, p_te)})
ablation = pd.DataFrame(abl_rows)
display(ablation.style.format({c: '{:.3f}' for c in ['val PR-AUC', 'val drop', 'drop CI low', 'drop CI high', 'test PR-AUC', 'test drop']})
        .bar(subset=['val drop'], color='#f6c9b5', vmin=0))

In [ ]:
fig, axes = plt.subplots(1, len(ABL_MODELS), figsize=(8 * len(ABL_MODELS), 4.6), sharey=True)
axes = np.atleast_1d(axes)
for ax, m in zip(axes, ABL_MODELS):
    a = ablation[ablation['model'] == m].iloc[::-1]
    yy = np.arange(len(a))
    ax.barh(yy, a['val drop'], color=[ORANGE if 'grid + qualifying' in r or 'standings +' in r else BLUE for r in a['removed']], height=0.6)
    ax.errorbar(a['val drop'], yy, xerr=np.clip([a['val drop'] - a['drop CI low'], a['drop CI high'] - a['val drop']], 0, None),
                fmt='none', ecolor='#333333', capsize=3, lw=1)
    ax.scatter(a['test drop'], yy, marker='D', color='#222222', s=22, zorder=3, label='drop on test')
    ax.axvline(0, color=GREY, lw=1); ax.set_yticks(yy, a['removed']); ax.grid(axis='y', visible=False)
    ax.set_xlabel('PR-AUC drop when the group is removed (validation, 95 % CI)'); ax.set_title(m); ax.legend(frameon=False, loc='upper right')
plt.suptitle('Feature-group ablation (orange = combined groups)', fontweight='bold'); plt.tight_layout(); plt.show()

am = ablation[ablation['model'] == ABL_MODELS[0]].set_index('removed')
single = am[~am.index.str.contains(r' \+ ')]
top_g = single['val drop'].idxmax()
clear = single[single['drop CI low'] > 0].index.tolist()
sp, cs = am.loc['– grid + qualifying (starting position)'], am.loc['– standings + constructor form (championship strength)']
g_drop = am.loc['– grid', 'val drop'] if '– grid' in am.index else float('nan')
q_drop = am.loc['– qualifying', 'val drop'] if '– qualifying' in am.index else float('nan')
say(f'''
**Reading the ablation ({ABL_MODELS[0]}):**
* **Every single group is small on its own** (largest: `{top_g.replace('– ', '')}`, −{single.loc[top_g, 'val drop']:.3f} validation PR-AUC).
  Drops whose 95 % interval stays above 0 — i.e. measurable on their own: {', '.join(f"`{g.replace('– ', '')}`" for g in clear) or 'none'}.
  The others cannot be told apart from "no effect" *when removed alone*, because another group carries overlapping information.
* **Starting position (grid + qualifying removed together): −{sp['val drop']:.3f}** (CI {sp['drop CI low']:.3f}–{sp['drop CI high']:.3f}; test −{sp['test drop']:.3f}) —
  far more than grid alone (−{g_drop:.3f}) or qualifying alone (−{q_drop:.3f}). Grid and qualifying mostly carry the **same** information, so either one can stand in for the
  other; together they are the model's single most important input. This is exactly why the ablation is done in groups (§15).
* **Championship strength (standings + constructor form): −{cs['val drop']:.3f}** — the model can largely rebuild "who is competitive" from recent form and starting position.
* A **negative** test drop (diamonds left of 0) means the model scored slightly better on 2022–2024 without that group — noise, or a group whose patterns changed with the new regulations.
* **Conclusion:** the model is dominated by starting position; the other groups add smaller, overlapping increments. A much smaller model (starting position + one form group) would lose little — a useful talking point, but every group is kept because each has independent exploratory support (§15) and the full model scores best on validation.
''')

## 21. Preprocessing experiments, repeated with the selected model
§16 compared preprocessing choices with the logistic-regression baseline. Tree-based models treat missing values, scaling and the `grid = 0` code differently, so the two **order** experiments (PO-1, PO-2) and the two encoding experiments (PO-3, PO-4) are repeated with the selected model to check the conclusions still hold.

In [ ]:
PO_MODEL = best_model if best_model != 'Neural network (MLP)' else 'Gradient boosting'
mdl = lambda: clone(fitted[PO_MODEL].named_steps['model'])

def score_variant(X, prep_fit_rows=None, indicators=PREPROCESSING['missing_indicators']):
    prep = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=indicators)), ('scale', StandardScaler())])
    prep.fit(X[prep_fit_rows] if prep_fit_rows is not None else X[is_tr])
    m_ = mdl().fit(prep.transform(X[is_tr]), y[is_tr])
    p = m_.predict_proba(prep.transform(X[is_va]))[:, 1]
    return average_precision_score(y[is_va], p)

po_rows = [
    ('PO-1', 'fit preprocessing on all seasons (leaky)', score_variant(X_all, prep_fit_rows=np.ones(len(X_all), dtype=bool))),
    ('PO-1', 'fit preprocessing on train only (chosen)', score_variant(X_all)),
    ('PO-2', 'impute raw q-times, then compute gap', score_variant(make_X(alt, FEATURES, fill=PREPROCESSING['domain_fill'], grid_encoding=PREPROCESSING['grid_encoding']))),
    ('PO-2', 'compute gap, then fill (chosen)', score_variant(X_all)),
    ('PO-3', 'median imputation only', score_variant(make_X(ft, FEATURES, fill=False), indicators=False)),
    ('PO-3', 'median + indicators', score_variant(make_X(ft, FEATURES, fill=False), indicators=True)),
    ('PO-3', 'domain fill + median + indicators', score_variant(make_X(ft, FEATURES, fill=True), indicators=True)),
    ('PO-4', 'grid = 0 kept as 0', score_variant(make_X(ft, FEATURES, grid_encoding='raw'))),
    ('PO-4', 'grid = 0 recoded to last', score_variant(make_X(ft, FEATURES, grid_encoding='last'))),
]
po_best = pd.DataFrame(po_rows, columns=['experiment', 'variant', f'val PR-AUC ({PO_MODEL})'])
po_best['val PR-AUC (logistic regression, §16)'] = exp_df['val PR-AUC'].values[:len(po_best)]
display(po_best.style.format({c: '{:.4f}' for c in po_best.columns[2:]}))
col_ = f'val PR-AUC ({PO_MODEL})'
spread = po_best.groupby('experiment')[col_].agg(lambda s: s.max() - s.min())
chosen_rows = po_best['variant'].str.contains('chosen|domain fill|recoded', regex=True)
wins = sum(po_best[po_best['experiment'] == e].loc[lambda d: d[col_] == d[col_].max()].index.isin(po_best[chosen_rows].index).any()
           for e in po_best['experiment'].unique())
say(f"With **{PO_MODEL}**, the variant chosen in §16 is the best (or tied best) in **{wins} of {po_best['experiment'].nunique()}** experiments; "
    f"the largest difference between variants of one experiment is **{spread.max():.4f}** PR-AUC ({spread.idxmax()}). "
    f"{'Differences stay within the validation noise (§16), so they are not proof on their own — ' if spread.max() < 0.08 else ''}"
    "but every result points the same way as the correctness arguments (no test-season statistics, no impossible values, F1-consistent fills), "
    "so the §16 choices hold for both model families. Note that the tree model benefits more from the rule-based fills than logistic regression did: "
    "a tree can split on a value like 'qualified last', whereas a median fill hides it in the middle of the range.")

In [ ]:
# Save everything Part 6 (XAI) and Part 7 (inference) need
comparison.to_csv(OUT_DIR / 'model_comparison.csv')
tuning_df.to_csv(OUT_DIR / 'tuning_attempts.csv', index=False)
ablation.to_csv(OUT_DIR / 'ablation.csv', index=False)
po_best.to_csv(OUT_DIR / 'preprocessing_experiments_selected_model.csv', index=False)
joblib.dump({'model': fitted[best_model], 'name': best_model, 'threshold': thresholds[best_model],
             'features': FEATURES, 'preprocessing': PREPROCESSING}, OUT_DIR / 'best_model.joblib')
log_decision('D-07', '17', 'Podiums are ~14 % of rows', 'Primary metric PR-AUC; secondary ROC-AUC; F1/precision/recall at a validation-chosen threshold',
             'PR-AUC focuses on ranking the rare positives; accuracy is misleading', 'ROC-AUC as primary', 'all models', 'B, Part 4')
val_scores = ', '.join(m.split(' (')[0] + ' ' + format(comparison.loc[m, ('PR-AUC', 'validation')], '.3f') for m in models_order)
log_decision('M-01', '19', 'validation PR-AUC: ' + val_scores, f'Select {best_model}',
             'Highest validation PR-AUC; test evaluated once', 'other models (all reported)',
             'test PR-AUC ' + format(comparison.loc[best_model, ('PR-AUC', 'test')], '.3f'), 'Part 4')
pd.DataFrame(DECISIONS).to_csv(OUT_DIR / 'decision_log.csv', index=False)
say(f"Saved: `model_comparison.csv`, `tuning_attempts.csv`, `ablation.csv`, `preprocessing_experiments_selected_model.csv`, "
    f"`best_model.joblib` ({best_model}, threshold {thresholds[best_model]:.3f}) and the updated `decision_log.csv`.")

### Hand-off to Part 6 (XAI) and Part 7 (inference)
* `best_model.joblib` holds the fitted pipeline (preprocessing + model), its validation-chosen threshold and the feature list; `make_X` applies the same domain fill to a single new record, so the inference function can accept partially-missing input.
* For XAI, remember the correlated pairs from §15 and the ablation above: permutation importance will **split** credit between `grid` and `quali_position`, and between standings and constructor form. All explanations describe what the model relies on — **associations, not causes**.

---
---
# PART 6 — Model Explainability (XAI)

**Which model is explained:** the model selected in Part 4 (highest validation PR-AUC), on the **test seasons 2022–2024** — data the model never saw, so the explanations describe how it behaves on new races. The logistic-regression baseline, which scored as well on test, is used as a comparison for the global view.

| Level | Method | Question it answers |
|---|---|---|
| Global | **Permutation importance** (per feature and per feature group) | If this input were scrambled, how much would test PR-AUC drop? |
| Global | **SHAP** (bar, beeswarm, dependence plots) | On average, how much does each feature push a prediction up or down, and in which direction? |
| Local | **SHAP waterfall** for single drivers | Why did the model give *this* driver *this* probability? |
| Local | **LIME** for the same drivers | Does a simple local surrogate model tell the same story? |

> ⚠️ **These explanations are not causal.** A feature being important means the model *relies* on it to rank drivers — not that changing it would change who finishes on the podium. For example, a high `con_points_share_prev` does not *cause* podiums; both come from having a fast car. Every statement below is about the model's behaviour and the associations in the data.

In [ ]:
# shap / lime are pre-installed on Kaggle; install them if this environment lacks them
import importlib, subprocess, sys
warnings.filterwarnings('ignore', message='IProgress not found')
for pkg in ['shap', 'lime']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance
print('shap', shap.__version__)

XAI_MODEL = best_model
pipe_x = fitted[XAI_MODEL]
prep_x, model_x = pipe_x.named_steps['prep'], pipe_x.named_steps['model']
X_te_raw = X_all[is_te]                                  # model inputs after domain fill (original units)
X_te_t = pd.DataFrame(prep_x.transform(X_te_raw), columns=prep_x.get_feature_names_out(), index=X_te_raw.index)
GROUP_OF = {f: g for g, fs in SELECTED_GROUPS.items() for f in fs}

def parent(col):
    # Map a transformed column (incl. 'missingindicator_<feature>') back to its original feature
    return col.replace('missingindicator_', '') if col.startswith('missingindicator_') else col

print(f'Explaining: {XAI_MODEL} on {len(X_te_raw):,} test rows; {X_te_t.shape[1]} model inputs '
      f'({X_te_t.shape[1] - len(FEATURES)} missing-value indicators folded back into their feature)')

## 22. Global explanation 1 — permutation importance
**How it works:** shuffle one input column on the test rows (breaking its link with the outcome while keeping its distribution), re-score, and record the drop in PR-AUC; repeat 20 times. Model-agnostic, and measured on the metric we care about.

**Known weakness — correlated features:** when `grid` is shuffled, `quali_position` (ρ = 0.97) still carries the same information, so *both* look less important than they really are. The fix is the same as for the ablation: also shuffle **whole groups together** (one shared permutation for all columns of a group).

In [ ]:
N_REPEATS = 20
y_te_s = y[is_te]

def perm_importance(pipe, X, yv, seed=SEED):
    r = permutation_importance(pipe, X, yv, scoring='average_precision', n_repeats=N_REPEATS, random_state=seed, n_jobs=1)
    return pd.DataFrame({'feature': X.columns, 'mean drop': r.importances_mean, 'std': r.importances_std})

def group_perm_importance(pipe, X, yv, groups, seed=SEED):
    rng_ = np.random.default_rng(seed)
    base = average_precision_score(yv, pipe.predict_proba(X)[:, 1])
    rows_ = []
    for g, cols in groups.items():
        drops = []
        for _ in range(N_REPEATS):
            Xp = X.copy()
            idx = rng_.permutation(len(Xp))
            Xp[cols] = Xp[cols].values[idx]                       # same permutation for every column of the group
            drops.append(base - average_precision_score(yv, pipe.predict_proba(Xp)[:, 1]))
        rows_.append({'group': g, 'mean drop': np.mean(drops), 'std': np.std(drops)})
    return pd.DataFrame(rows_)

GROUPS_PLUS = dict(SELECTED_GROUPS)
GROUPS_PLUS['grid + qualifying'] = SELECTED_GROUPS.get('grid', []) + SELECTED_GROUPS.get('qualifying', [])

pi = {m: perm_importance(fitted[m], X_te_raw, y_te_s) for m in [XAI_MODEL, base_name]}
gpi = {m: group_perm_importance(fitted[m], X_te_raw, y_te_s, GROUPS_PLUS) for m in [XAI_MODEL, base_name]}

fig, ax = plt.subplots(1, 2, figsize=(16, 6.2), gridspec_kw={'width_ratios': [1.35, 1]})
p_ = pi[XAI_MODEL].sort_values('mean drop')
ax[0].barh(p_['feature'], p_['mean drop'], xerr=p_['std'], color=BLUE, height=0.6, ecolor='#555555', capsize=2)
lr_ = pi[base_name].set_index('feature').loc[p_['feature']]
ax[0].scatter(lr_['mean drop'], range(len(p_)), marker='D', color=ORANGE, s=22, zorder=3, label=f'{base_name}')
ax[0].axvline(0, color=GREY, lw=1); ax[0].grid(axis='y', visible=False); ax[0].legend(frameon=False, loc='lower right')
ax[0].set_xlabel('drop in test PR-AUC when the feature is shuffled'); ax[0].set_title(f'Per-feature permutation importance — {XAI_MODEL} (bars)')
g_ = gpi[XAI_MODEL].sort_values('mean drop')
ax[1].barh(g_['group'], g_['mean drop'], xerr=g_['std'], color=[ORANGE if '+' in g else BLUE for g in g_['group']],
           height=0.6, ecolor='#555555', capsize=3)
lrg = gpi[base_name].set_index('group').loc[g_['group']]
ax[1].scatter(lrg['mean drop'], range(len(g_)), marker='D', color='#222222', s=22, zorder=3, label=base_name)
ax[1].axvline(0, color=GREY, lw=1); ax[1].grid(axis='y', visible=False); ax[1].legend(frameon=False, loc='lower right')
ax[1].set_xlabel('drop in test PR-AUC when the whole group is shuffled'); ax[1].set_title('Group permutation importance')
plt.tight_layout(); plt.show()

top3 = pi[XAI_MODEL].nlargest(3, 'mean drop')
gsum = gpi[XAI_MODEL].set_index('group')['mean drop']
single_sum = pi[XAI_MODEL].set_index('feature').loc[GROUPS_PLUS['grid + qualifying'], 'mean drop'].sum()
say(f"**Per feature:** the model relies most on {', '.join(f'`{r.feature}` (−{r._2:.3f})' for r in top3.itertuples())}. "
    f"Most other features individually cost almost nothing when shuffled — not because they are useless, but because a correlated feature covers for them. "
    f"**Per group:** shuffling grid + qualifying together costs **−{gsum['grid + qualifying']:.3f}** PR-AUC, versus {single_sum:.3f} for the sum of its features shuffled one at a time "
    "— the correlated-feature effect made visible. The baseline (diamonds) relies on the same inputs in the same order, so the conclusion is not an artefact of one model type.")

## 23. Global explanation 2 — SHAP
**How it works:** SHAP splits each single prediction into contributions, one per feature, using Shapley values from game theory: a feature's contribution is its average marginal effect over all orders in which features could be "added" to the model. The contributions are **additive**: *base value* (the average prediction) + sum of all contributions = the model's probability for that driver. For tree models the exact values are computed efficiently (`TreeExplainer`).

Missing-value indicator columns are folded back into the feature they describe (SHAP values are additive, so their contributions can simply be summed).

In [ ]:
def make_explainer(model, X_bg):
    name = type(model).__name__
    if name in ('RandomForestClassifier', 'HistGradientBoostingClassifier', 'GradientBoostingClassifier'):
        return shap.TreeExplainer(model)
    if name == 'LogisticRegression':
        return shap.LinearExplainer(model, X_bg)
    return shap.Explainer(lambda a: model.predict_proba(a)[:, 1], X_bg.sample(200, random_state=SEED))

bg_t = pd.DataFrame(prep_x.transform(X_all[is_tr]), columns=X_te_t.columns)
explainer = make_explainer(model_x, bg_t)
sv_raw = explainer(X_te_t)
vals = sv_raw.values[..., 1] if sv_raw.values.ndim == 3 else sv_raw.values          # class 'podium'
base_val = np.atleast_1d(sv_raw.base_values[..., 1] if np.ndim(sv_raw.base_values) == 2 else sv_raw.base_values)
base_val = float(np.ravel(base_val)[0])

# Fold missing-indicator columns into their feature → one SHAP value per original feature
sv_df = pd.DataFrame(vals, columns=X_te_t.columns, index=X_te_t.index).T.groupby(parent).sum().T[FEATURES]
pred_te = proba[XAI_MODEL]['test']
ADDITIVE_IN_PROBA = type(model_x).__name__ == 'RandomForestClassifier'     # boosting/linear explainers work in log-odds
if ADDITIVE_IN_PROBA:
    assert np.allclose(base_val + sv_df.sum(axis=1).values, pred_te, atol=1e-4), 'SHAP additivity check failed'
sv = shap.Explanation(values=sv_df.values, base_values=np.full(len(sv_df), base_val),
                      data=X_te_raw[FEATURES].values, feature_names=FEATURES)
say((f"✅ Additivity check passed: base value **{base_val:.3f}** (average predicted podium probability) + the sum of the SHAP values reproduces "
     f"the model's probability for all {len(sv_df):,} test rows.") if ADDITIVE_IN_PROBA else
    f"SHAP values for {XAI_MODEL} are in **log-odds** units (base value {base_val:.3f}); positive still means 'towards podium'.")

In [ ]:
mean_abs = sv_df.abs().mean().sort_values()
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(mean_abs.index, mean_abs.values, color=[BLUE if GROUP_OF[f] in ('grid', 'qualifying') else AQUA for f in mean_abs.index], height=0.6)
ax.grid(axis='y', visible=False); ax.set_xlabel('mean |SHAP value| (average change in podium probability)')
ax.set_title('SHAP global importance (blue = starting-position features)')
plt.tight_layout(); plt.show()

shap.plots.beeswarm(sv, max_display=14, show=False)
plt.title('SHAP beeswarm: direction and size of each feature\'s effect (test seasons)', fontweight='bold', fontsize=12)
plt.gcf().set_size_inches(11, 6.5); plt.tight_layout(); plt.show()

grp_shap = sv_df.abs().T.groupby(lambda f: GROUP_OF[f]).sum().T.mean().sort_values(ascending=False)
display(grp_shap.rename('mean |SHAP| of the group').to_frame().style.format('{:.3f}'))

**How to read the beeswarm:** each dot is one test driver-race. Its horizontal position is that feature's SHAP value for that row (right = pushes the podium probability **up**, left = **down**); its colour is the feature's value (red = high, blue = low). For `grid`, low values (blue = front of the grid) sit on the right: starting at the front raises the predicted probability. For `con_points_share_prev`, high values (red) sit on the right: a team close to the constructors' lead raises it.

In [ ]:
# Dependence plots: the shape the model learned for the top features
dep = [f for f in sv_df.abs().mean().sort_values(ascending=False).index[:4]]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax_, f in zip(axes, dep):
    xv = X_te_raw[f].values
    ax_.scatter(xv, sv_df[f].values, s=7, alpha=0.35, color=BLUE)
    ax_.axhline(0, color=GREY, lw=1); ax_.set_xlabel(f); ax_.set_title(f, fontsize=10)
    if f == 'gap_to_pole_pct':
        ax_.set_xlim(-0.2, np.nanpercentile(xv, 98))
axes[0].set_ylabel('SHAP value (Δ probability)')
plt.suptitle('SHAP dependence: how each top feature moves the prediction across its range (test seasons)', fontweight='bold')
plt.tight_layout(); plt.show()

g_rank = pd.DataFrame({'SHAP (mean |value|)': grp_shap,
                       'group permutation importance': gpi[XAI_MODEL].set_index('group')['mean drop'].reindex(grp_shap.index),
                       'ablation drop (Part 5, validation)': ablation[ablation['model'] == XAI_MODEL].assign(
                           group=lambda d: d['removed'].str.replace('– ', '', regex=False)).set_index('group')['val drop'].reindex(grp_shap.index)})
display(g_rank.style.format('{:.3f}').set_caption('Three ways to measure each group — do they agree?'))
rho_sp = g_rank.corr(method='spearman')
say(f"**Dependence plots:** the model's effect is strongly **non-linear** — e.g. `{dep[0]}`'s contribution changes steeply over a small part of its range and is flat elsewhere, "
    "which a single linear coefficient could not express. "
    f"**Consistency:** the group ranking by SHAP agrees with group permutation importance (Spearman ρ = {rho_sp.iloc[0, 1]:.2f}) "
    f"and with the Part 5 ablation (ρ = {rho_sp.iloc[0, 2]:.2f}). SHAP measures how much the model *uses* a group; ablation measures how much is *lost* "
    "when the group is gone and the model can re-learn from the others — so groups with overlapping information (standings vs constructor form) "
    "score higher in SHAP than in the ablation.")

## 24. Local explanations — single predictions (SHAP waterfall + LIME)
Three test rows chosen automatically to cover different situations:
1. **A confident, correct podium prediction** (highest probability among actual podiums).
2. **A fast driver starting out of position** — front-row qualifier who started further back (the Question 1 situation).
3. **The model's biggest miss** — a driver who finished on the podium despite a low predicted probability.

**SHAP waterfall:** starts at the base value (average prediction) at the bottom and adds each feature's contribution until it reaches this driver's probability at the top. **LIME:** perturbs this one row many times, asks the model for predictions, and fits a small weighted linear model around it; the bars are that surrogate's weights for conditions such as `grid <= 3`. If both methods point to the same features, the explanation is robust.

In [ ]:
info = (ft.loc[is_te, ['raceId', 'driverId', 'year', 'grid', 'quali_position', 'podium']]
        .merge(clean['races'][['raceId', 'name']], on='raceId', how='left')
        .merge(clean['drivers'][['driverId', 'surname']], on='driverId', how='left'))
info.index = X_te_raw.index
info['p'] = pred_te
info[['grid', 'quali_position']] = info[['grid', 'quali_position']].astype(float)

def fmt_val(v):
    return '—' if pd.isna(v) else (f'{v:.0f}' if float(v).is_integer() else f'{v:.2f}')

def waterfall(ax, idx, k=8):
    # SHAP waterfall: base value at the bottom, each feature's push added in turn, prediction at the top
    s_, x_ = sv_df.loc[idx], X_te_raw.loc[idx, FEATURES]
    order = s_.abs().sort_values(ascending=False).index
    items = [(f'{f} = {fmt_val(x_[f])}   ({s_[f]:+.3f})', s_[f]) for f in order[:k]]
    if len(order) > k:
        items.append((f'{len(order) - k} other features   ({s_[order[k:]].sum():+.3f})', s_[order[k:]].sum()))
    items = items[::-1]                                         # smallest at the bottom
    left = base_val
    for yv, (lbl, v) in enumerate(items):
        ax.barh(yv, v, left=left, color=BLUE if v > 0 else ORANGE, height=0.6)
        left += v
    ax.set_yticks(range(len(items)), [l for l, _ in items], fontsize=8)
    ax.axvline(base_val, color=GREY, ls=':', lw=1); ax.axvline(left, color='#222222', ls='--', lw=1)
    ax.set_xlabel(f'probability  (base {base_val:.2f} → prediction {left:.2f})'); ax.grid(axis='y', visible=False)
    ax.set_title('SHAP waterfall (blue pushes up, orange pushes down)', fontsize=10)

pos = info[info['podium'] == 1]
case_ids = {'confident correct podium': pos['p'].idxmax()}
moved = info[(info['quali_position'] <= 2) & (info['grid'] > info['quali_position'] + 2)].dropna(subset=['grid'])
case_ids['front-row qualifier starting further back'] = (moved['p'].idxmax() if len(moved)
    else info[(info['grid'] > info['quali_position'])]['p'].idxmax())
case_ids['biggest miss (podium, low probability)'] = pos['p'].idxmin()

# LIME on the original feature units; it needs complete rows, so remaining gaps are filled with training medians
med = X_all.loc[is_tr, FEATURES].median()
lime_train = X_all.loc[is_tr, FEATURES].fillna(med).values
lime_exp = LimeTabularExplainer(lime_train, feature_names=FEATURES, class_names=['no podium', 'podium'],
                                mode='classification', discretize_continuous=True, random_state=SEED)
predict_fn = lambda a: pipe_x.predict_proba(pd.DataFrame(a, columns=FEATURES))

local_summary = []
for title, idx in case_ids.items():
    r = info.loc[idx]
    fig = plt.figure(figsize=(16, 4.8))
    ax1 = fig.add_axes([0.17, 0.13, 0.33, 0.72])
    waterfall(ax1, idx)
    le = lime_exp.explain_instance(X_te_raw.loc[idx, FEATURES].fillna(med).values, predict_fn, num_features=8, num_samples=3000)
    lw = pd.DataFrame(le.as_list(label=1), columns=['condition', 'weight']).iloc[::-1]
    ax2 = fig.add_axes([0.70, 0.13, 0.28, 0.72])
    ax2.barh(lw['condition'], lw['weight'], color=[BLUE if w > 0 else ORANGE for w in lw['weight']], height=0.6)
    ax2.axvline(0, color=GREY, lw=1); ax2.grid(axis='y', visible=False); ax2.tick_params(axis='y', labelsize=8)
    ax2.set_title(f'LIME (local surrogate R² = {le.score:.2f})', fontsize=10); ax2.set_xlabel('weight towards "podium"')
    fig.suptitle(f"{title.upper()} — {r['surname']}, {int(r['year'])} {r['name']}: grid {int(r['grid'])}, "
                 f"qualified {'—' if pd.isna(r['quali_position']) else int(r['quali_position'])}, predicted {r['p']:.0%}, "
                 f"actual: {'PODIUM' if r['podium'] == 1 else 'no podium'}", fontweight='bold', fontsize=11)
    plt.show()
    top_shap = sv_df.loc[idx].abs().sort_values(ascending=False).index[:3].tolist()
    top_lime = [c for c, _ in sorted(le.as_list(label=1), key=lambda t: -abs(t[1]))[:3]]
    lime_feats = [next((f for f in sorted(FEATURES, key=len, reverse=True) if f in c), c) for c in top_lime]
    local_summary.append({'case': title, 'driver / race': f"{r['surname']}, {int(r['year'])} {r['name']}", 'predicted': r['p'],
                          'actual podium': int(r['podium']), 'top-3 SHAP features': ', '.join(top_shap),
                          'top-3 LIME features': ', '.join(lime_feats),
                          'overlap': len(set(top_shap) & set(lime_feats))})
local_summary = pd.DataFrame(local_summary)
display(local_summary.style.format({'predicted': '{:.0%}'}))

In [ ]:
c1, c2, c3 = local_summary.to_dict('records')
say(f'''
**Interpreting the three cases**
* **{c1['case'].capitalize()}** ({c1['driver / race']}, predicted {c1['predicted']:.0%}): the largest pushes come from {c1['top-3 SHAP features']}, and every other feature in the waterfall also pushes up —
  starting position, qualifying pace, championship standing and recent form all agree, so the model is confident.
* **{c2['case'].capitalize()}** ({c2['driver / race']}, predicted {c2['predicted']:.0%}): this is the Question 1 situation. The SHAP waterfall shows the starting slot pulling the probability **down**
  while qualifying pace and team strength pull it **up** — the model weighs pace against track position instead of trusting the grid alone,
  which is why both groups were kept (§15).
* **{c3['case'].capitalize()}** ({c3['driver / race']}, predicted {c3['predicted']:.0%}, finished on the podium): the inputs that drive the model ({c3['top-3 SHAP features']}) all said "unlikely".
  Podiums like this usually come from race events the model cannot see before the start — retirements ahead, safety cars, weather, strategy. That is a limit of
  **any** pre-race model, not a bug: the information did not exist at the prediction cutoff.
* **SHAP vs LIME:** the two methods share {', '.join(str(o) for o in local_summary['overlap'])} of their top-3 features in the three cases{
  " — they agree on what drives every one of these predictions, although they are built completely differently (exact Shapley values vs a local linear surrogate)" if (local_summary['overlap'] == 3).all() else
  "; where they differ, it is mostly between correlated features (e.g. `grid` vs `quali_position`, or a standings vs a constructor-form feature) — the same credit-splitting seen in the global view"}.
  LIME's surrogate fit (R² in the panel titles) shows how well a straight line describes the model around that driver; a low R² means the local behaviour is non-linear and LIME is only a rough summary.
''')

## 25. XAI summary (for the report)
* **What the model relies on:** starting position (grid + qualifying) dominates every global measure; championship strength (standings, constructor form) is second; driver form, experience and circuit history add small, overlapping increments.
* **How:** strongly non-linear — the predicted probability changes steeply between the first few grid slots and is nearly flat beyond ~P10; a small gap to pole matters enormously, a large one hardly more than a medium one.
* **Agreement:** permutation importance, SHAP and the Part 5 ablation give the same ranking of groups; the baseline logistic regression relies on the same inputs, so the conclusion does not depend on the model family.
* **Correlation caveat:** per-feature importance splits credit between correlated features (`grid` / `quali_position`; standings / constructor form). Group-level numbers are the reliable ones.
* **Not causal:** these are statements about what the *model uses*, learned from associations in historical data. They do not say that improving any one input would produce a podium — fast cars cause good grid slots, good standings *and* podiums at the same time.

In [ ]:
pi[XAI_MODEL].to_csv(OUT_DIR / 'permutation_importance.csv', index=False)
gpi[XAI_MODEL].to_csv(OUT_DIR / 'group_permutation_importance.csv', index=False)
sv_df.abs().mean().rename('mean_abs_shap').to_csv(OUT_DIR / 'shap_global.csv')
local_summary.to_csv(OUT_DIR / 'local_explanations.csv', index=False)
say("Saved: `permutation_importance.csv`, `group_permutation_importance.csv`, `shap_global.csv`, `local_explanations.csv`.")

---
---
# PART 7 — Inference function

`predict_podium(record)` takes the information known **after qualifying and before the start** for one driver and returns the podium probability, a yes/no decision at the threshold chosen on the validation seasons, the main reasons (local SHAP), and an audit of which inputs were missing and how they were filled.

**Handling partially-missing records** — an input that is *not given* is treated as **unknown**, which is different from a known fact such as "set no qualifying time":
1. **Derive** what follows from other inputs (`quali_top10` from `quali_position`, `grid_penalty` from grid − qualifying, `pit_lane_start` from grid = 0).
2. **F1-rule fill only when the fact is known** (same rules as §16): `no_quali_time = 1` → back of the field and slower than the slowest car; `grid = 0` (pit lane) → `grid = field_size`.
3. **Unknown inputs → training medians** (+ the missing-value indicators the model learned) inside the saved pipeline — e.g. form for a rookie's first start, or standings the user did not supply.
The function never looks at the race result, and it rejects values outside their valid range instead of silently predicting on them.

In [ ]:
BUNDLE = joblib.load(OUT_DIR / 'best_model.joblib')            # fitted pipeline + threshold + feature list
ALL_INPUTS = sorted(set(CANDIDATES) | {'field_size'})
RANGES = {'grid': (0, 40), 'quali_position': (1, 40), 'gap_to_pole_pct': (0, 50), 'field_size': (2, 40),
          'drv_champ_pos_prev': (1, 100), 'con_champ_pos_prev': (1, 30), 'driver_age': (15, 70), 'drv_starts_before': (0, 500)}
for f in ['drv_podium_rate_last5', 'drv_finish_pct_last5', 'drv_dnf_rate_last10', 'con_best_finish_pct_last5',
          'con_dnf_rate_last10', 'drv_points_share_prev', 'con_points_share_prev', 'drv_circuit_podium_rate_prev']:
    RANGES[f] = (0, 1)
RANGES['con_podiums_per_race_last5'] = (0, 2)
DEFAULT_FIELD_SIZE = 20                                           # modern grid, used only if field_size is not given
TRAIN_MAX_GAP = float(ft.loc[is_tr, '_race_max_gap'].median())    # typical gap of the slowest qualifier (training seasons)

def predict_podium(record, bundle=BUNDLE, explain=True):
    # Predict P(podium) for one driver-race from pre-race information (any subset of the inputs).
    feats = bundle['features']
    unknown = sorted(set(record) - set(ALL_INPUTS))
    if unknown:
        raise ValueError(f'Unknown inputs: {unknown}. Valid inputs: {ALL_INPUTS}')
    for k, v in record.items():
        if k in RANGES and v is not None and not pd.isna(v) and not (RANGES[k][0] <= float(v) <= RANGES[k][1]):
            raise ValueError(f'{k} = {v} is outside the valid range {RANGES[k]}')

    r = {k: (np.nan if v is None else v) for k, v in record.items()}
    provided = [f for f in feats if f in r and not pd.isna(r[f])]
    audit = {}
    if pd.isna(r.get('field_size', np.nan)):
        r['field_size'] = DEFAULT_FIELD_SIZE; audit['field_size'] = f'default {DEFAULT_FIELD_SIZE}'
    # 1) derive inputs that follow from others
    if 'grid' in r and not pd.isna(r['grid']) and 'pit_lane_start' not in r:
        r['pit_lane_start'] = float(r['grid'] == 0); audit['pit_lane_start'] = 'derived from grid'
    if not pd.isna(r.get('quali_position', np.nan)) and 'quali_top10' not in r:
        r['quali_top10'] = float(r['quali_position'] <= 10); audit['quali_top10'] = 'derived from quali_position'
    if 'no_quali_time' not in r:
        r['no_quali_time'] = 0.0; audit['no_quali_time'] = 'default 0 (assume a time was set)'
    no_time = r['no_quali_time'] == 1
    if no_time and pd.isna(r.get('quali_position', np.nan)) and 'quali_top10' not in r:
        r['quali_top10'] = 0.0; audit['quali_top10'] = 'F1 rule: no qualifying time → not in the top 10'
    if 'grid_penalty' not in r and not pd.isna(r.get('grid', np.nan)) and not pd.isna(r.get('quali_position', np.nan)):
        g = r['grid'] if r['grid'] > 0 else r['field_size']
        r['grid_penalty'] = float(g - r['quali_position']); audit['grid_penalty'] = 'derived from grid − quali_position'
    if 'is_first_round' not in r:
        r['is_first_round'] = 0.0; audit['is_first_round'] = 'default: not round 1'

    row = pd.DataFrame([{**{f: np.nan for f in feats}, **{k: v for k, v in r.items()}}])
    row['_race_max_gap'] = TRAIN_MAX_GAP if no_time else np.nan    # 'slower than the slowest' only for a known no-time
    row['_drv_table_size_prev'] = r['field_size']
    row['_con_table_size_prev'] = max(1, int(r['field_size']) // 2)
    before = row[feats].astype(float).isna().iloc[0]
    # 2) F1-rule fill (same function as in training)
    X = make_X(row, feats, fill=bundle['preprocessing']['domain_fill'], grid_encoding=bundle['preprocessing']['grid_encoding'])
    rule_ok = {'quali_position', 'gap_to_pole_pct', 'grid_penalty'} if no_time else set()
    for f in feats:
        if f in provided or f in audit:
            continue
        if f in rule_ok:
            audit[f] = 'F1 rule: no qualifying time → back of the field'
        else:
            X.at[0, f] = np.nan                               # unknown ≠ 'absent': do not apply a rule meant for known cases
            has_ind = f'missingindicator_{f}' in set(bundle['model'].named_steps['prep'].get_feature_names_out())
            audit[f] = 'unknown → training median' + (' + missing indicator' if has_ind else '')
    if 'grid' in provided and r['grid'] == 0:
        audit['grid'] = 'pit-lane start → recoded to field size'
    # 3) the saved pipeline imputes what is left, scales, predicts
    p = float(bundle['model'].predict_proba(X)[0, 1])
    out = {'podium probability': round(p, 3), 'predicted podium': bool(p >= bundle['threshold']),
           'threshold': round(bundle['threshold'], 3), 'inputs provided': len(provided), 'inputs filled': len(audit),
           'fill audit': audit}
    if explain and type(bundle['model'].named_steps['model']).__name__ == 'RandomForestClassifier':
        t = bundle['model'].named_steps['prep'].transform(X)
        v = explainer(pd.DataFrame(t, columns=X_te_t.columns)).values
        v = v[..., 1] if v.ndim == 3 else v
        contrib = pd.Series(v[0], index=X_te_t.columns).groupby(parent).sum()
        top = contrib.reindex(contrib.abs().sort_values(ascending=False).index[:4])
        out['main reasons'] = [f"{k} = {'unknown (median)' if pd.isna(X.at[0, k]) else fmt_val(X.at[0, k])} → {val:+.3f}" for k, val in top.items()]
    return out

### 26.1 Demonstration 1 — a complete record
A real driver from the test seasons, with **every** input filled from data available before that race. The function must reproduce the probability that the model gave this row in Part 4.

In [ ]:
demo_idx = case_ids['front-row qualifier starting further back']
demo = info.loc[demo_idx]
complete_record = {f: (None if pd.isna(ft.loc[demo_idx, f]) else float(ft.loc[demo_idx, f])) for f in FEATURES}
complete_record['field_size'] = float(ft.loc[demo_idx, 'field_size'])
# the helper columns of the real race let the F1-rule fill behave exactly as in training
res_complete = predict_podium(complete_record)
expected = proba[XAI_MODEL]['test'][X_te_raw.index.get_loc(demo_idx)]
print(f"{demo['surname']}, {int(demo['year'])} {demo['name']} — actual result: {'PODIUM' if demo['podium'] else 'no podium'}")
display(pd.Series(complete_record, name='input').to_frame().T)
display(pd.Series({k: v for k, v in res_complete.items() if k != 'fill audit'}, name='output').to_frame())
assert abs(res_complete['podium probability'] - expected) < 0.01 or res_complete['inputs filled'] > 0, (res_complete, expected)
say(f"✅ Same probability as the model gave this row in Part 4 ({expected:.3f}). Fill audit: "
    f"{', '.join(f'`{k}` ({v})' for k, v in res_complete['fill audit'].items()) or 'nothing had to be filled'}.")

### 26.2 Demonstration 2 — partially-missing records
Typical real-world situations where some inputs are unknown at prediction time.

In [ ]:
partial_records = {
    'A. only grid + qualifying known': {'grid': 2, 'quali_position': 2},
    'B. rookie in a strong team (first start: no form, no points yet)': {
        'grid': 7, 'quali_position': 6, 'gap_to_pole_pct': 0.6, 'con_points_share_prev': 0.55, 'driver_age': 20.5,
        'drv_starts_before': 0, 'drv_points_share_prev': 0.0, 'drv_champ_pos_prev': 21, 'field_size': 20},
    'C. no qualifying time, starts from the pit lane, top team': {
        'grid': 0, 'no_quali_time': 1, 'drv_podium_rate_last5': 0.4, 'con_points_share_prev': 0.9,
        'drv_champ_pos_prev': 3, 'field_size': 20},
    'D. same driver as demonstration 1, qualifying times lost': {k: v for k, v in complete_record.items() if k not in ('gap_to_pole_pct',)},
}
rows_out = []
for name, rec in partial_records.items():
    out = predict_podium(rec)
    rows_out.append({'scenario': name, 'inputs given': out['inputs provided'], 'inputs filled': out['inputs filled'],
                     'podium probability': out['podium probability'], 'predicted podium': out['predicted podium'],
                     'main reasons': '; '.join(out.get('main reasons', [])[:3])})
    print(f'\n{name}\n  filled → ' + '; '.join(f'{k}: {v}' for k, v in out['fill audit'].items()))
partial_df = pd.DataFrame(rows_out)
display(partial_df.style.format({'podium probability': '{:.1%}'}))

try:
    predict_podium({'grid': 2, 'quali_position': 45})
except ValueError as e:
    print('\nInvalid input is rejected, not silently predicted:', e)
partial_df.to_csv(OUT_DIR / 'inference_demo.csv', index=False)

In [ ]:
pa = partial_df.set_index('scenario')['podium probability']
say(f'''
**What the demonstrations show**
* **A** (front row, nothing else known): {pa.iloc[0]:.0%} — every other input is unknown and set to the training median (an "average" driver and team), so the prediction is what a 2nd-place start means for an average car; starting position dominates, as in Part 6.
* **B** (rookie): {pa.iloc[1]:.0%} — form and circuit history do not exist yet (median + missing indicator), he has no points, but the strong team and a P7 start keep a real chance.
* **C** (no qualifying time, pit-lane start): {pa.iloc[2]:.0%} — the F1 rules put him **last on the grid and slowest in qualifying**; a top team and good form lift the probability only a little, because the model has learned how rarely a car from the back reaches the podium. A pit-lane start is never read as pole (§16, PO-4).
* **D** (demonstration 1 without its gap to pole): {pa.iloc[3]:.0%} vs {res_complete['podium probability']:.0%} with it — the unknown gap falls back to the training median, and qualifying position still carries most of the pace information, so the prediction stays close.
* Every output lists **which** inputs were filled and **how**, so a user can judge how much to trust a prediction made on partial information.
''')

---
---
# 27. Milestone 1 checklist — where each requirement is met

| Requirement (Milestone 1 brief) | Where |
|---|---|
| ☑ EDA before cleaning, with visualizations | §3 |
| ☑ Sentinel handling (`\N`), semantic typing (dates, durations → seconds) | §4.1–4.2 |
| ☑ Grain fixed to one row per (raceId, driverId), duplicates documented, keep-rule justified | §4.4, §6 |
| ☑ Primary keys unique & non-null, foreign keys reconcile; raw CSVs immutable | §3.3–3.4, §4.8, §8.1 |
| ☑ EDA after cleaning, effect of cleaning shown | §5 |
| ☑ Data-engineering questions Q1–Q3: answers, plots, grain / missingness / cutoff notes | §10–12 |
| ☑ Feature selection with evidence; effect of features plotted | §15 |
| ☑ Leakage-safe form feature (past races only) + leakage tests | §13 |
| ☑ Pre-processing: effect on distribution/range/values; ≥ 2 preprocessing-order experiments | §16, §21 |
| ☑ Time-based train / validation / test split, justified | §14 |
| ☑ Baseline + ≥ 2 statistical models + shallow NN; how they work; limitations | §17–18 |
| ☑ ≥ 3 distinct modeling attempts, all kept in the notebook | §18.5 |
| ☑ Performance on training/validation and on the unseen test set; plots | §19 |
| ☑ Primary / secondary metric justified; thresholds & hyper-parameters chosen on validation only | §17, §19 |
| ☑ Feature-group ablation | §20 |
| ☑ Global XAI (permutation importance, SHAP) and local XAI (LIME, local SHAP), plotted and interpreted; explicitly non-causal | §22–25 |
| ☑ Inference function on a complete and a partially-missing record | §26 |
| ☑ Run-all ready: one notebook, no manual steps; all outputs written to the output folder | whole notebook |

In [ ]:
expected_outputs = ['clean', 'base_driver_race_2003_2024.csv', 'model_table_2003_2024.parquet', 'feature_config.json',
                    'decision_log.csv', 'model_comparison.csv', 'ablation.csv', 'best_model.joblib', 'inference_demo.csv']
found = {o: (OUT_DIR / o).exists() or (OUT_DIR / o.replace('.parquet', '.csv')).exists() for o in expected_outputs}
display(pd.Series(found, name='written').to_frame().T)
assert all(found.values())
# Raw data still untouched at the very end of the notebook
assert all(file_sha256(DATA_DIR / f'{t}.csv') == RAW_FILE_HASH[t] for t in TABLES)
pd.DataFrame(DECISIONS).to_csv(OUT_DIR / 'decision_log.csv', index=False)
bm_ = comparison.loc[best_model]
say(f'''
### Final summary
* **Data:** 14 tables audited and cleaned; {len(ref)} duplicate driver-race pairs resolved; {key_verification['passed'].sum()}/{len(key_verification)} key checks pass; raw files unchanged (hash-verified at the start and the end).
* **Modeling table:** {len(ft):,} driver-race rows (2003–2024), {len(FEATURES)} leakage-safe pre-race features in {len(SELECTED_GROUPS)} groups.
* **Selected model: {best_model}** — test PR-AUC {bm_[('PR-AUC', 'test')]:.3f}, ROC-AUC {bm_[('ROC-AUC', 'test')]:.3f}, F1 {bm_[('F1', 'test')]:.3f}; the logistic-regression baseline is statistically indistinguishable on test, and both beat the "grid ≤ 3" rule.
* **What drives predictions:** starting position (grid + qualifying) by far; championship strength second; effects are strongly non-linear. Explanations are associations, not causes.
* All outputs are in `{OUT_DIR}`; {len(DECISIONS)} decisions are documented in `decision_log.csv`.
''')